# StorageLLM JUJU physical shard materializer

This notebook takes source GGUF files from Hugging Face and creates StorageLLM
JUJU runtime artifacts. GGUF is input only. The generated model artifacts are:

1. `<original_shard_stem>.juju` - immutable weight container with tensor payloads
   physically rewritten into 4096-byte aligned JUJU sections.
2. `<original_shard_stem>.juju.idx` - runtime tensor/section index for load,
   prefetch, telemetry, and tier updates.
3. `<original_shard_stem>.juju.verify.json` - verification and upload summary.

The notebook preserves the original shard stem and only changes the extension.
It does not create `model1`, `model2`, or patched GGUF final artifacts. Run the
setup cells, then run the final `CHANGE HERE / RUN HERE` cell.

Configured source in this notebook: interactive source selected in CHANGE HERE / RUN HERE


In [ ]:
import gc
import hashlib
import io
import json
import math
import os
import re
import shutil
import struct
import subprocess
import sys
import time
import urllib.parse
import urllib.request
from getpass import getpass
from pathlib import Path


NOTEBOOK_BUILD_ID = "juju-physical-shard-materializer-20260501"

# ========================= CHANGE HERE =========================
# Set SOURCE_SUBDIR for a GGUF folder, or SOURCE_FILE_NAME for one GGUF file.
# SOURCE_FILE_PREFIX and SOURCE_FILE_COUNT can stay blank; the notebook will
# discover shard names from Hugging Face and derive both values automatically.
SOURCE_HF_REPO_ID = os.environ.get("SOURCE_HF_REPO_ID", "").strip()
SOURCE_SUBDIR = os.environ.get("SOURCE_SUBDIR", "").strip("/")
SOURCE_FILE_NAME = os.environ.get("SOURCE_FILE_NAME", "").strip().strip("/")
SOURCE_FILE_PREFIX = os.environ.get("SOURCE_FILE_PREFIX", "").strip()
SOURCE_FILE_COUNT_TEXT = os.environ.get("SOURCE_FILE_COUNT", "").strip()
SOURCE_FILE_EXT = os.environ.get("SOURCE_FILE_EXT", ".gguf")
if not SOURCE_FILE_EXT.startswith("."):
    SOURCE_FILE_EXT = "." + SOURCE_FILE_EXT
TARGET_SUBDIR = os.environ.get("TARGET_SUBDIR", SOURCE_SUBDIR).strip("/")
TARGET_HF_OWNER = os.environ.get("TARGET_HF_OWNER", os.environ.get("HF_USERNAME", "")).strip()
TARGET_HF_REPO_ID = os.environ.get("TARGET_HF_REPO_ID", "").strip()
TARGET_REPO_SUFFIX = os.environ.get("TARGET_REPO_SUFFIX", "").strip()
# ===============================================================

BASE_DOWNLOAD_URL = f"https://huggingface.co/{SOURCE_HF_REPO_ID}/resolve/main"

MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", "/content/GGUF_OFFLOAD_MODEL"))
WORK_DIR = Path(os.environ.get("OFFLOAD_PATCH_WORK_DIR", "/content/gguf_offload_work"))
IN_DIR = WORK_DIR / "input"
OUT_DIR = WORK_DIR / "output"
VERIFY_DIR = MODEL_ROOT / "verify"
EXPORT_DIR = MODEL_ROOT / "offload_exports"
GGUF_EXPORT_DIR = EXPORT_DIR / "gguf"
SAFETENSORS_EXPORT_DIR = EXPORT_DIR / "safetensors"
SIDECAR_EXPORT_DIR = EXPORT_DIR / "sidecar"
CLEAN_LOCAL_WORK_ON_START = os.environ.get("CLEAN_LOCAL_WORK_ON_START", "1") != "0"
if CLEAN_LOCAL_WORK_ON_START:
    for path in (IN_DIR, OUT_DIR, VERIFY_DIR, EXPORT_DIR):
        if path.exists():
            shutil.rmtree(path)
for path in (IN_DIR, OUT_DIR, VERIFY_DIR, GGUF_EXPORT_DIR, SAFETENSORS_EXPORT_DIR, SIDECAR_EXPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

def repo_path_join(folder, name):
    return f"{folder}/{name}" if folder else name


def hf_tree_api_url(repo_id, folder):
    repo = urllib.parse.quote(repo_id, safe="/")
    if folder:
        return f"https://huggingface.co/api/models/{repo}/tree/main/{urllib.parse.quote(folder, safe='/')}"
    return f"https://huggingface.co/api/models/{repo}/tree/main"


def env_or_colab_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value or ""


def list_hf_gguf_names(repo_id, folder, ext):
    request = urllib.request.Request(hf_tree_api_url(repo_id, folder))
    token = env_or_colab_secret("SOURCE_HF_TOKEN") or env_or_colab_secret("HF_TOKEN")
    if token:
        request.add_header("Authorization", f"Bearer {token}")
    with urllib.request.urlopen(request, timeout=120) as response:
        entries = json.loads(response.read().decode("utf-8"))
    names = []
    prefix = f"{folder}/" if folder else ""
    for entry in entries:
        path = str(entry.get("path") or "")
        kind = str(entry.get("type") or "")
        if kind and kind != "file":
            continue
        if not path.lower().endswith(ext.lower()):
            continue
        name = path[len(prefix):] if prefix and path.startswith(prefix) else Path(path).name
        if "/" not in name:
            names.append(name)
    names.sort()
    return names


def derive_shard_prefix_and_count(names):
    pattern = re.compile(r"^(?P<prefix>.+)-(?P<idx>\d{5})-of-(?P<count>\d{5})" + re.escape(SOURCE_FILE_EXT) + r"$")
    parsed = []
    for name in names:
        match = pattern.match(name)
        if not match:
            return "", 0
        parsed.append((match.group("prefix"), int(match.group("idx")), int(match.group("count"))))
    if not parsed:
        return "", 0
    prefixes = {x[0] for x in parsed}
    counts = {x[2] for x in parsed}
    if len(prefixes) != 1 or len(counts) != 1:
        return "", 0
    count = next(iter(counts))
    indexes = sorted(x[1] for x in parsed)
    if indexes != list(range(1, count + 1)):
        return "", 0
    return next(iter(prefixes)), count


def build_source_file_specs():
    global SOURCE_FILE_PREFIX
    if not SOURCE_HF_REPO_ID:
        raise RuntimeError("Set SOURCE_HF_REPO_ID before running this notebook")
    explicit_source_file = SOURCE_FILE_NAME.strip("/")
    source_paths = []
    if explicit_source_file:
        source_path = explicit_source_file
        if SOURCE_SUBDIR and "/" not in source_path:
            source_path = repo_path_join(SOURCE_SUBDIR, source_path)
        names = [Path(source_path).name]
        source_paths = [source_path]
        SOURCE_FILE_PREFIX = SOURCE_FILE_PREFIX or Path(names[0]).stem
    elif SOURCE_FILE_PREFIX and SOURCE_FILE_COUNT_TEXT:
        count = int(SOURCE_FILE_COUNT_TEXT)
        names = [
            f"{SOURCE_FILE_PREFIX}-{idx:05d}-of-{count:05d}{SOURCE_FILE_EXT}"
            for idx in range(1, count + 1)
        ]
    else:
        names = list_hf_gguf_names(SOURCE_HF_REPO_ID, SOURCE_SUBDIR, SOURCE_FILE_EXT)
        if not names:
            location = f"{SOURCE_HF_REPO_ID}/{SOURCE_SUBDIR}" if SOURCE_SUBDIR else SOURCE_HF_REPO_ID
            raise RuntimeError(f"no {SOURCE_FILE_EXT} files found in {location}")
        detected_prefix, detected_count = derive_shard_prefix_and_count(names)
        if detected_prefix and detected_count:
            SOURCE_FILE_PREFIX = SOURCE_FILE_PREFIX or detected_prefix
            names = [
                f"{SOURCE_FILE_PREFIX}-{idx:05d}-of-{detected_count:05d}{SOURCE_FILE_EXT}"
                for idx in range(1, detected_count + 1)
            ]
        else:
            SOURCE_FILE_PREFIX = SOURCE_FILE_PREFIX or Path(names[0]).stem
    specs = []
    count = len(names)
    for idx, name in enumerate(names, start=1):
        specs.append({
            "index": idx,
            "count": count,
            "name": name,
            "source_path": source_paths[idx - 1] if source_paths else repo_path_join(SOURCE_SUBDIR, name),
        })
    return specs


# Source discovery is intentionally deferred until the final RUN cell.
# That keeps setup cells cheap and lets the last cell own the HF URL/start range.
GGUF_FILE_SPECS = []
SOURCE_FILE_COUNT = 0
START_FILE_INDEX = int(os.environ.get("START_FILE_INDEX", os.environ.get("START_PART", "1")))
END_FILE_INDEX = int(os.environ.get("END_FILE_INDEX", os.environ.get("END_PART", "0")))
FILES_TO_RUN = []

def parse_hf_source_url(source):
    source = str(source or "").strip()
    if not source:
        return "", "", ""
    source = source.split("?", 1)[0].split("#", 1)[0].strip()
    marker = "huggingface.co/"
    if marker in source:
        source = source.split(marker, 1)[1]
    parts = [p for p in source.strip("/").split("/") if p]
    if len(parts) < 2:
        raise RuntimeError(f"Could not parse Hugging Face source URL or repo id: {source!r}")
    repo_id = "/".join(parts[:2])
    rest = parts[2:]
    path = ""
    if len(rest) >= 2 and rest[0] in {"tree", "blob", "resolve"}:
        path = "/".join(rest[2:])
    elif rest:
        path = "/".join(rest)
    if path.lower().endswith(SOURCE_FILE_EXT.lower()):
        return repo_id, "", path
    return repo_id, path.strip("/"), ""

def refresh_model_profile_for_current_source():
    global SOURCE_WEIGHT_QUANT, MODEL_PROFILE
    SOURCE_WEIGHT_QUANT = infer_source_weight_quantization(SOURCE_SUBDIR, SOURCE_FILE_PREFIX, SOURCE_FILE_NAME)
    if "MODEL_PROFILE" not in globals() or not isinstance(MODEL_PROFILE, dict):
        return
    MODEL_PROFILE["model_id"] = os.environ.get("MODEL_ID", SOURCE_FILE_PREFIX.lower().replace("/", "_").replace("-", "_"))
    MODEL_PROFILE["artifact_stem"] = SOURCE_FILE_PREFIX
    MODEL_PROFILE["source_repo"] = SOURCE_HF_REPO_ID
    MODEL_PROFILE["source_subdir"] = SOURCE_SUBDIR
    MODEL_PROFILE["source_file_prefix"] = SOURCE_FILE_PREFIX
    MODEL_PROFILE["source_file_count"] = SOURCE_FILE_COUNT
    MODEL_PROFILE["target_repo"] = TARGET_HF_REPO_ID
    MODEL_PROFILE["target_subdir"] = TARGET_SUBDIR
    weight_schema = MODEL_PROFILE.setdefault("weight_quant_schema", {})
    weight_schema.update({
        "source_family": SOURCE_WEIGHT_QUANT["family"],
        "source_bits": SOURCE_WEIGHT_QUANT["bits"],
        "source_kernel_family": SOURCE_WEIGHT_QUANT["kernel_family"],
        "source_block_size": SOURCE_WEIGHT_QUANT["block_size"],
        "dispatch_key": SOURCE_WEIGHT_QUANT["dispatch_key"],
        "runtime_dispatch_rule": SOURCE_WEIGHT_QUANT["runtime_rule"],
    })

def configure_juju_notebook_job(
    source_url=None,
    target_repo_id=None,
    target_owner=None,
    target_subdir=None,
    start_file_index=None,
    end_file_index=None,
    source_file_prefix=None,
    source_file_count=None,
    upload_to_hf=True,
    upload_foreign_metadata=True,
    delete_local_after_upload=True,
):
    global SOURCE_HF_REPO_ID, SOURCE_SUBDIR, SOURCE_FILE_NAME, SOURCE_FILE_PREFIX, SOURCE_FILE_COUNT_TEXT
    global TARGET_SUBDIR, TARGET_HF_OWNER, TARGET_HF_REPO_ID, TARGET_REPO_SUFFIX, BASE_DOWNLOAD_URL
    global GGUF_FILE_SPECS, SOURCE_FILE_COUNT, START_FILE_INDEX, END_FILE_INDEX, FILES_TO_RUN
    global UPLOAD_TO_HF, UPLOAD_FOREIGN_METADATA, DELETE_LOCAL_AFTER_UPLOAD
    if not str(source_url or SOURCE_HF_REPO_ID or "").strip():
        source_url = input("SOURCE HF URL / repo / folder / GGUF file: ").strip()
    repo_id, parsed_subdir, parsed_file = parse_hf_source_url(source_url or SOURCE_HF_REPO_ID)
    if repo_id:
        SOURCE_HF_REPO_ID = repo_id
    if parsed_file:
        SOURCE_FILE_NAME = parsed_file
        SOURCE_SUBDIR = ""
    elif parsed_subdir:
        SOURCE_SUBDIR = parsed_subdir.strip("/")
        SOURCE_FILE_NAME = ""
    if source_file_prefix is not None:
        SOURCE_FILE_PREFIX = str(source_file_prefix).strip()
    if source_file_count is not None:
        SOURCE_FILE_COUNT_TEXT = "" if source_file_count in (None, "") else str(source_file_count)
    if target_subdir is not None:
        TARGET_SUBDIR = str(target_subdir).strip("/")
    elif not TARGET_SUBDIR:
        TARGET_SUBDIR = SOURCE_SUBDIR
    if target_owner is not None:
        TARGET_HF_OWNER = str(target_owner).strip()
    if target_repo_id is not None:
        TARGET_HF_REPO_ID = str(target_repo_id).strip()
    BASE_DOWNLOAD_URL = f"https://huggingface.co/{SOURCE_HF_REPO_ID}/resolve/main"
    GGUF_FILE_SPECS = build_source_file_specs()
    SOURCE_FILE_COUNT = len(GGUF_FILE_SPECS)
    if not TARGET_REPO_SUFFIX:
        safe_prefix = re.sub(r"[^A-Za-z0-9._-]+", "-", SOURCE_FILE_PREFIX).replace("_", "-").strip("-").lower()
        TARGET_REPO_SUFFIX = f"{safe_prefix}-juju" if safe_prefix else "juju-offload-model"
    if not TARGET_HF_REPO_ID:
        if not TARGET_HF_OWNER:
            target_value = input("TARGET HF repo id owner/name, or owner only: ").strip()
            if "/" in target_value:
                TARGET_HF_REPO_ID = target_value
            else:
                TARGET_HF_OWNER = target_value
        if not TARGET_HF_REPO_ID:
            if not TARGET_HF_OWNER:
                raise RuntimeError("Target HF repo or owner is required when upload is enabled")
            TARGET_HF_REPO_ID = f"{TARGET_HF_OWNER}/{TARGET_REPO_SUFFIX}"
    START_FILE_INDEX = int(start_file_index if start_file_index is not None else os.environ.get("START_FILE_INDEX", os.environ.get("START_PART", "1")))
    END_FILE_INDEX = int(end_file_index if end_file_index is not None else os.environ.get("END_FILE_INDEX", os.environ.get("END_PART", str(SOURCE_FILE_COUNT))))
    FILES_TO_RUN = [
        spec for spec in GGUF_FILE_SPECS
        if START_FILE_INDEX <= int(spec["index"]) <= END_FILE_INDEX
    ]
    UPLOAD_TO_HF = bool(upload_to_hf)
    UPLOAD_FOREIGN_METADATA = bool(upload_foreign_metadata)
    DELETE_LOCAL_AFTER_UPLOAD = bool(delete_local_after_upload)
    refresh_model_profile_for_current_source()
    print("configured source:", SOURCE_HF_REPO_ID, SOURCE_FILE_NAME or SOURCE_SUBDIR or "repo root")
    print("target repo:", TARGET_HF_REPO_ID)
    print("files:", [spec["index"] for spec in FILES_TO_RUN], "of", SOURCE_FILE_COUNT)
    return FILES_TO_RUN

UPLOAD_TO_HF = True
EXPORT_FOREIGN_METADATA = True
UPLOAD_FOREIGN_METADATA = True
DELETE_LOCAL_AFTER_UPLOAD = True
KEEP_FAILED_LOCAL_FILES = True
CONTINUE_ON_PART_ERROR = os.environ.get("CONTINUE_ON_PART_ERROR", "1") != "0"
PROGRESS_INTERVAL_BYTES = int(os.environ.get("PROGRESS_INTERVAL_BYTES", str(256 * 1024 * 1024)))
HTTP_CHUNK_BYTES = int(os.environ.get("HTTP_CHUNK_BYTES", str(16 * 1024 * 1024)))
GGUF_HEADER_RANGE_BYTES = int(os.environ.get("GGUF_HEADER_RANGE_BYTES", str(256 * 1024 * 1024)))
SMALL_FILE_DIRECT_GET_LIMIT_BYTES = int(os.environ.get("SMALL_FILE_DIRECT_GET_LIMIT_BYTES", str(512 * 1024 * 1024)))
GGUF_ARCH_META_PREFIXES = tuple(
    x.strip()
    for x in os.environ.get(
        "GGUF_ARCH_META_PREFIXES",
        "general.,split.,llama.,glm.,gemma.,qwen.,qwen2.,qwen3.,qwen2moe.,deepseek2.,mpt.,gptneox.",
    ).split(",")
    if x.strip()
)
BASE_MODEL_REPO = os.environ.get("BASE_MODEL_REPO", "")
DEFAULT_REQUIRED_RUNTIME_FLAGS = os.environ.get("DEFAULT_REQUIRED_RUNTIME_FLAGS", "")
REQUIRED_RUNTIME_FLAGS = [
    x.strip()
    for x in os.environ.get("REQUIRED_RUNTIME_FLAGS", DEFAULT_REQUIRED_RUNTIME_FLAGS).split(",")
    if x.strip()
]
SHARED_EXPERTS_FUSION_ALLOWED = os.environ.get("SHARED_EXPERTS_FUSION_ALLOWED", "0" if REQUIRED_RUNTIME_FLAGS else "1") == "1"
QKV_K_BITS = int(os.environ.get("QKV_K_BITS", "3"))
QKV_V_BITS = int(os.environ.get("QKV_V_BITS", "2"))
QKV_GROUP_SIZE = int(os.environ.get("QKV_GROUP_SIZE", "64"))
QKV_PAGE_SIZE_TOKENS = int(os.environ.get("QKV_PAGE_SIZE_TOKENS", "16"))
QKV_OUTLIER_CHANNELS = int(os.environ.get("QKV_OUTLIER_CHANNELS", "32"))
QKV_OUTLIER_BITS = int(os.environ.get("QKV_OUTLIER_BITS", "3"))
FORMAT_MAXIMIZATION_TARGET_COUNT = int(os.environ.get("FORMAT_MAXIMIZATION_TARGET_COUNT", "5000"))

def infer_source_weight_quantization(*values):
    text = " ".join(str(v or "") for v in values).lower().replace("-", "_")
    checks = [
        ("ud_iq2", "gguf_iq2_m", 2, "gguf_iq2"),
        ("iq2", "gguf_iq2", 2, "gguf_iq2"),
        ("iq3", "gguf_iq3", 3, "gguf_iq3"),
        ("iq4", "gguf_iq4", 4, "gguf_iq4"),
        ("mxfp4", "gguf_mxfp4", 4, "gguf_mxfp4"),
        ("nvfp4", "gguf_nvfp4", 4, "gguf_nvfp4"),
        ("q8", "gguf_q8", 8, "gguf_q8"),
        ("q6", "gguf_q6", 6, "gguf_q6"),
        ("q5", "gguf_q5", 5, "gguf_q5"),
        ("q4", "gguf_q4", 4, "gguf_q4"),
        ("q3", "gguf_q3", 3, "gguf_q3"),
        ("q2", "gguf_q2", 2, "gguf_q2"),
    ]
    for needle, family, bits, kernel in checks:
        if needle in text:
            return {
                "family": family,
                "bits": bits,
                "kernel_family": kernel,
                "block_size": 256 if "k" in text or "iq" in text else 0,
                "dispatch_key": f"{kernel}:{bits}",
                "runtime_rule": "select_backend_kernel_by_quant_family_not_by_bits_only",
            }
    return {
        "family": "runtime_read_from_gguf_tensor_type",
        "bits": 0,
        "kernel_family": "runtime_read_from_gguf_tensor_type",
        "block_size": 0,
        "dispatch_key": "runtime_detect",
        "runtime_rule": "parse_each_gguf_tensor_type_before_kernel_selection",
    }


def build_format_maximization_catalog(target_count=5000):
    target_count = int(target_count)
    families = [
        ("gguf_header", "GGUF magic, version, tensor count, metadata count, and alignment"),
        ("gguf_metadata", "namespaced GGUF key/value metadata and source architecture fields"),
        ("tensor_directory", "tensor name, shape, type, file offset, and shard lookup table"),
        ("weight_residency", "weight placement across device memory, host memory, and disk"),
        ("expert_residency", "MoE expert triplet placement, route hotness, and anchor layers"),
        ("qkv_cache", "packed QKV cache quantization, paging, sink tokens, and outliers"),
        ("prefetch_pipeline", "transfer scheduling, speculative fetch, and compute overlap"),
        ("eviction_policy", "pressure triggers, rollback safety, and cold span release"),
        ("io_streaming", "HTTP range, mmap-style spans, sequential reads, and local cleanup"),
        ("runtime_adaptation", "startup probe, profiling feedback, fallback, and telemetry"),
    ]
    scopes = [
        ("discovery", "discover exact facts before allocation"),
        ("validation", "reject impossible or unsafe runtime plans"),
        ("indexing", "build stable ids and lookup tables"),
        ("placement", "choose hot and cold residency locations"),
        ("prefetch", "move bytes before the dependent kernel needs them"),
        ("eviction", "free cold data before pressure stalls decode"),
        ("batching", "shape microbatch and continuous batch decisions"),
        ("scheduling", "order work to overlap transfer and compute"),
        ("telemetry", "measure latency, bandwidth, stalls, and miss rates"),
        ("recovery", "fall back without corrupting generation state"),
    ]
    actions = [
        ("record", "record a metadata value the engine can consume directly"),
        ("normalize", "normalize names, units, and quant families before planning"),
        ("bound", "clamp runtime choices to safe memory and quality limits"),
        ("rank", "rank candidates by expected latency and memory pressure"),
        ("pin", "pin anchors that should remain resident under pressure"),
        ("stream", "stream cold spans instead of materializing the whole file"),
        ("verify", "verify offsets, counts, and checksums before use"),
        ("fallback", "select a conservative path when probing is incomplete"),
        ("compact", "compact or pack metadata-driven runtime state"),
        ("expose", "expose runtime hooks for engine policy overrides"),
    ]
    policies = [
        ("required", "engine must consume this point or emit an explicit unsupported reason"),
        ("preferred", "engine should use this point when the probed hardware supports it"),
        ("adaptive", "engine may update the choice after runtime telemetry"),
        ("conservative", "engine should favor correctness and bounded memory"),
        ("verify_only", "engine should use this as a validation and diagnostics point"),
    ]
    research_profiles = [
        ("juju_final_model_structure", "final JUJU model structure splits the runtime model into shared core, routing brain, expert groups, expert segments, QKV subsystem, code locality cache, and tier scheduler", "runtime_adaptation.indexing.juju_final_model_structure"),
        ("semantic_trajectory_prefetch_core", "semantic hint and expert trajectory are both first-class routing predictors rather than side heuristics", "expert_residency.prefetch_pipeline.semantic_trajectory_core"),
        ("seqtopk_primary_router", "SeqTopK sequence-budget routing is part of the runtime routing layer so hard tokens can spend more expert budget without changing the immutable weight file", "runtime_adaptation.routing.seqtopk_primary"),
        ("expert_pattern_signal", "expert selection patterns are telemetry signals stored and replayed through JUJU index tables", "expert_residency.telemetry.expert_pattern_signal"),
        ("comoe_resource_scheduler", "CoMoE-style edge scheduling is modeled through tier capabilities, resource state, cache policy, and prefetch worker decisions", "runtime_adaptation.scheduling.comoe_resource_scheduler"),
        ("code_locality_candidate_cache", "coding best-of-N and beam candidates share prefix KV, expert hotsets, candidate groups, and compile-equivalent pruning state", "runtime_adaptation.batching.code_locality_candidate_cache"),
        ("structure_locked_custom_extension", "the top-level JUJU structure is locked; future changes go into CUSTOM sections, policy tables, or mutable index records", "tensor_directory.validation.structure_locked_extension"),
        ("code_moe_shared_prefix_locality", "code-generation best-of-N and beam candidates share MoE experts strongly under a common prefix; same-token Jaccard 0.649, different-token Jaccard 0.175, middle-layer crossing around L14-L20, and 67 percent of compiled candidates concentrate in the top three assembly-equivalent groups", "expert_residency.prefetch_pipeline.code_shared_prefix_locality"),
        ("code_moe_middle_layer_pin", "coding forks reuse middle-layer experts even when generated tokens differ, so middle layers should get higher GPU-residency and prefetch priority", "expert_residency.placement.middle_layer_gpu_pin"),
        ("code_best_of_n_candidate_cache", "best-of-N, top-P, and beam-search branches should share expert cache, prefix KV cache, compile-equivalent pruning state, and fork-local telemetry", "runtime_adaptation.batching.shared_prefix_candidate_cache"),
        ("language_framework_expert_specialization", "coding requests should expose language and framework hints so Python, Rust, JavaScript, build-system, and repository-pattern expert hot sets can be profiled separately", "expert_residency.telemetry.language_framework_hotset"),
        ("powerinfer_hot_cold_split", "activation locality follows a power-law: hot neurons or experts stay resident on device while cold units use host or disk paths with predictors", "weight_residency.placement.hot_cold_split"),
        ("fmoe_iteration_prefetch", "semantic embedding plus expert trajectory prediction for iteration-level MoE expert prefetch", "expert_residency.prefetch_pipeline.semantic_and_trajectory"),
        ("seqtopk_sequence_budget", "sequence-level TopK shifts the expert budget from per-token K to a sequence budget T*K, giving hard tokens more experts and easy tokens fewer while preserving total work", "runtime_adaptation.routing.sequence_budget"),
        ("local_routing_consistency_srp_sch", "measure whether a model suits expert offloading using segment routing best performance and segment cache best hit rate, with cache size near twice active experts as a useful policy point", "expert_residency.telemetry.srp_sch_cache_sizing"),
        ("oracle_moe_semantic_locality", "route tokens in a compact semantic or oracle space to reduce inter-token expert variation and swapping demand", "expert_residency.prefetch_pipeline.semantic_locality_space"),
        ("comoe_edge_scheduler", "combine expert prediction, prefetching, caching, scheduling, and multi-tier storage with resource-aware edge scheduling", "runtime_adaptation.scheduling.edge_multitier"),
        ("specexec_offload_speculation", "bulk draft-token verification to use transfer bandwidth under offloading", "prefetch_pipeline.speculative_decode.bulk_verify"),
        ("swift_self_speculation", "uniform layer-skip draft path without a separate draft model", "runtime_adaptation.speculative_decode.layer_skip"),
        ("flexinfer_async_preservation", "async prefetch, balanced memory locking, and flexible tensor preservation under a user budget", "weight_residency.io_streaming.flexible_preservation"),
        ("kvswap_mobile_storage", "KV cache offload policy for NVMe, UFS, eMMC, and SD-class bandwidth limits", "qkv_cache.io_streaming.mobile_storage"),
        ("ssd_moe_energy", "SSD expert loading energy and active expert fraction aware placement", "expert_residency.placement.energy_aware"),
        ("mlp_offload_multilevel_multipath", "GPU to CPU RAM to disk multi-level and multi-path I/O hiding", "io_streaming.scheduling.multilevel_multipath"),
        ("io_cache_execution_frequency", "profile tensor execution order, access frequency, tensor size distribution, and queue depth to build prefetch tables that remove I/O from the critical path", "io_streaming.prefetch.execution_frequency_table"),
        ("async_double_buffer_moe", "while layer N computes, transfer layer N+1 RAM-to-device and layer N+2 disk-to-RAM so compute and I/O overlap", "prefetch_pipeline.scheduling.double_buffer_two_level"),
        ("gpudirect_storage_path", "when hardware supports it, expose a direct NVMe-to-device DMA path that bypasses CPU bounce buffers", "io_streaming.stream.gpudirect_storage_optional"),
        ("expert_two_level_granularity", "expert is the offload decision unit while internal segments are the I/O and overlap unit", "expert_residency.indexing.two_level_granularity"),
        ("moepic_top_bottom_segment", "split large experts into cached top segment and demand-loaded bottom segment so misses fetch less data and partial execution remains possible", "expert_residency.placement.moepic_top_bottom"),
        ("moeshard_row_column_split", "row-wise and column-wise expert sharding can balance FFN matrix loading while respecting activation boundaries", "tensor_directory.placement.row_column_expert_split"),
        ("prescope_fine_chunk_io", "fine-grained expert chunks saturate PCIe and heterogeneous memory bandwidth when batched asynchronously", "io_streaming.scheduling.fine_grained_chunk_io"),
        ("segment_auto_decision", "converter/runtime chooses segment count from expert size, measured NVMe bandwidth, compute time, VRAM pressure, and RAM capacity", "runtime_adaptation.bound.segment_auto_decision"),
        ("io_batching_queue_control", "io_uring requests are batched 16 to 64 at a time to avoid IOPS waste and queue churn", "io_streaming.scheduling.io_batching_queue_control"),
        ("co_used_expert_readahead", "physically adjacent co-used experts allow one large sequential read to carry upcoming experts", "expert_residency.prefetch_pipeline.co_used_readahead"),
        ("juju_primary_format", "JUJU is the primary custom MoE engine format and the only target container; GGUF is input only", "tensor_directory.indexing.juju_primary_format"),
        ("universal_tier_abstraction", "engine logic targets COMPUTE_MEM, FAST_MEM, and SLOW_MEM instead of hardcoding GPU, CPU, RAM, NVMe, or unified memory roles", "runtime_adaptation.indexing.universal_tier_map"),
        ("tier_capability_probe", "each tier records measured capacity, bandwidth, latency, async support, and DMA support before strategy selection", "runtime_adaptation.telemetry.tier_capability_probe"),
        ("pipeline_budget_config", "prefetch depth, resident expert counts, I/O threads, and block size come from compute time versus transfer time", "prefetch_pipeline.bound.pipeline_budget_config"),
        ("execution_path_selector", "runtime chooses full resident, direct storage, three-tier, rotating buffer, CPU cache, or storage streaming path from tier capabilities", "runtime_adaptation.scheduling.execution_path_selector"),
        ("path_specific_io_hints", "section table records per-path sequential and random block sizes, prefetch distance, and mmap friendliness", "io_streaming.indexing.path_specific_io_hints"),
        ("layer_order_index", "format can carry path-specific layer/expert offset order for GPU tiered mode and CPU sequential streaming mode", "tensor_directory.indexing.layer_order_index"),
        ("juju_header_4096", "JUJU header is a fixed 4096-byte aligned block with magic, version, section table pointer, model summary, size hints, and integrity hashes", "tensor_directory.indexing.juju_header_4096"),
        ("juju_section_table", "JUJU section table records section type, offset, size, uncompressed size, compression, checksum, and debug name", "tensor_directory.indexing.juju_section_table"),
        ("juju_expert_chunk_header", "each JUJU expert chunk carries layer id, expert id, data size, task affinity, buddy ids, prefetch hints, tier, read size, and chunk hash", "expert_residency.indexing.juju_expert_chunk_header"),
        ("juju_hw_cache", "optional .juju.hw stores prior hardware probing results and is always validated against fresh startup measurements", "runtime_adaptation.telemetry.juju_hw_cache"),
        ("gguf_to_juju_converter", "converter classifies experts, computes buddy map, trains or imports predictors, writes hot/warm/cold sections, and initializes .juju.idx", "runtime_adaptation.indexing.gguf_to_juju_converter"),
        ("juju_immutable_container", "JUJU is an immutable weight container, not a GGUF metadata-only extension; GGUF is the conversion source", "tensor_directory.indexing.juju_immutable_container"),
        ("juju_idx_mutable_runtime", "JUJU keeps runtime-updated expert statistics and trajectory tables in a separate .juju.idx file while model.juju remains immutable", "runtime_adaptation.telemetry.juju_idx_mutable"),
        ("qkv_required_default", "QKV cache uses the engine's quantized QKV path as mandatory default rather than an optional sidecar", "qkv_cache.validation.required_quantized_qkv"),
        ("rtx3090_overlap_budget", "if MoE compute is about 13 ms and RAM-to-device expert transfer about 6 ms, at most two expert transfers fit under one block without stalling", "prefetch_pipeline.bound.rtx3090_two_expert_window"),
        ("probe_gate_initialized_predictor", "distill the target router into a gate-initialized lookahead predictor so prediction and prefetch stay off the critical path", "expert_residency.prefetch_pipeline.probe_gate_initialized"),
        ("prescope_llapor_presched", "layer-aware predictors plus cross-layer scheduling and async I/O balance prediction accuracy, PCIe competition, and cross-device plan cost", "expert_residency.scheduling.llapor_presched"),
        ("spmoe_cutoff_policy", "speculative decoding aware offloading uses cutoff layers to bound prefetch depth and prevent overfetch cache thrash", "prefetch_pipeline.bound.spmoe_cutoff_layer"),
        ("buddymoe_graceful_miss", "co-activation buddy experts provide immediate fallback when a predicted expert is absent from GPU cache", "expert_residency.fallback.buddy_expert"),
        ("hardware_probe_strategy", "engine must measure PCIe, pinned allocation, NVMe bandwidth, queue depth, GDS, and RAM bandwidth instead of trusting device specs", "runtime_adaptation.telemetry.hardware_probe"),
        ("adaptive_controller_runtime", "runtime metrics such as GPU wait, queue fullness, pinned failures, miss rate, and RAM pressure adjust prefetch depth, queues, and tier balance", "runtime_adaptation.scheduling.adaptive_controller"),
        ("serverlessllm_chunk_pool", "loading-optimized checkpoints should expose sequential chunks, in-memory address mapping, chunk pools, and a multi-stage loading pipeline", "io_streaming.indexing.sequential_chunk_pool"),
        ("llm_in_flash_bundling", "flash-friendly row-column bundling physically groups data used together so sequential reads replace random expert seeks", "tensor_directory.placement.row_column_bundle"),
        ("expert_tiered_physical_layout", "shared weights first, then hot expert block, warm expert block, cold expert block, each with offset, size, tier, and affinity", "expert_residency.indexing.hot_warm_cold_blocks"),
        ("direct_io_alignment", "expert chunks should expose 4KB alignment, sector size, direct-I/O eligibility, and DMA-safe boundaries", "io_streaming.validation.direct_io_alignment"),
        ("mutable_expert_stats_sidecar", "runtime-updated access statistics should live beside the model file so weight bytes remain immutable", "runtime_adaptation.telemetry.mutable_stat_sidecar"),
        ("kv_cache_sidecar_isolation", "KV cache persistence and spill should use a separate file or region so weight streaming and cache I/O do not fight each other", "qkv_cache.placement.sidecar_isolation"),
        ("io_uring_iopoll_pipeline", "Linux engines can submit cold expert disk reads through io_uring IOPOLL and poll completions without blocking compute", "io_streaming.stream.io_uring_iopoll"),
        ("pinned_memory_slab_pool", "NVMe-to-RAM and RAM-to-device transfer should reuse pinned slabs instead of allocating during decode", "prefetch_pipeline.compact.pinned_slab_pool"),
        ("three_stage_async_pipeline", "layer N compute, layer N+1 RAM-to-device transfer, and layer N+2 disk-to-RAM prefetch run concurrently", "prefetch_pipeline.scheduling.three_stage_async"),
        ("temporal_spatial_expert_scheduler", "expert scheduler should combine temporal trajectory, spatial layer locality, semantic hints, and task affinity", "expert_residency.scheduling.temporal_spatial_scheduler"),
        ("slo_select_n", "latency SLO aware selection of tensors to keep in host/device memory", "runtime_adaptation.placement.slo_select_n"),
        ("flexgen_placement_search", "joint placement of weights, KV cache, and activations across device, host, and disk", "weight_residency.placement.search"),
        ("kivi_gear_kv_compression", "asymmetric KV quantization with residual, low-rank, or sparse outlier compensation", "qkv_cache.compact.error_compensation"),
    ]
    items = []
    idx = 1
    for family_key, family_desc in families:
        for scope_key, scope_desc in scopes:
            for action_key, action_desc in actions:
                for policy_key, policy_desc in policies:
                    items.append({
                        "id": f"ofmt-{idx:04d}",
                        "priority": idx,
                        "family": family_key,
                        "scope": scope_key,
                        "action": action_key,
                        "policy": policy_key,
                        "metadata_key": f"offload.optimization_catalog_v1.items.{idx:04d}",
                        "engine_hook": f"{family_key}.{scope_key}.{action_key}.{policy_key}",
                        "runtime_goal": f"{action_desc}; {scope_desc}; {policy_desc}",
                        "engine_action": f"Use {family_key}/{scope_key} metadata to {action_key} runtime policy while preserving GGUF tensor payloads.",
                        "bottleneck_target": family_desc,
                        "fallback": "runtime_probe_then_conservative_path",
                        "research_profile": research_profiles[(idx - 1) % len(research_profiles)][0],
                        "research_idea": research_profiles[(idx - 1) % len(research_profiles)][1],
                        "research_hook": research_profiles[(idx - 1) % len(research_profiles)][2],
                        "basis": [
                            "gguf_extensible_namespaced_metadata",
                            "placement_across_device_host_disk",
                            "kv_cache_asymmetric_quantization",
                            "kv_cache_error_compensation",
                            "gpu_centric_prefetch_and_sync",
                            research_profiles[(idx - 1) % len(research_profiles)][0],
                        ],
                    })
                    idx += 1
    if len(items) != target_count:
        raise RuntimeError(f"format maximization catalog expected {target_count}, built {len(items)}")
    return {
        "schema_version": 1,
        "target_count": target_count,
        "actual_count": len(items),
        "namespace": "offload.optimization_catalog_v1",
        "engine_mutation_contract": {
            "engine_may_override_runtime_knobs": True,
            "engine_must_preserve_tensor_payload": True,
            "engine_must_preserve_generation_correctness": True,
            "adaptive_fields": [
                "placement", "residency", "prefetch", "eviction", "qkv_cache", "batching", "io_schedule", "kernel_dispatch", "telemetry", "fallback",
            ],
        },
        "juju_physical_container_contract": {
            "format_name": "JUJU",
            "target_container_format": "JUJU",
            "gguf_role": "conversion_source_only",
            "weight_file": "model.juju",
            "index_file": "model.juju.idx",
            "hardware_cache_file": "model.juju.hw",
            "weight_file_mutable": False,
            "index_file_mutable": True,
            "header": {
                "size_bytes": 4096,
                "alignment_bytes": 4096,
                "fields": [
                    "magic", "version", "flags", "expert_count", "layer_count",
                    "shared_weights_offset", "hot_block_offset", "warm_block_offset", "cold_block_offset",
                    "predictor_weights_offset", "buddy_map_offset", "tier_table_offset", "index_file_checksum",
                ],
            },
            "sections": [
                {"name": "predictor_weights", "purpose": "PROBE/PreScope lightweight layer predictors", "alignment_bytes": 4096},
                {"name": "buddy_map", "purpose": "BuddyMoE co-activation fallback", "alignment_bytes": 4096},
                {"name": "tier_table", "purpose": "initial hot/warm/cold expert classification", "alignment_bytes": 4096},
                {"name": "shared_weights", "purpose": "embed, norm, attention, shared experts", "alignment_bytes": 4096},
                {"name": "hot_experts", "purpose": "device-preferred contiguous expert chunks", "alignment_bytes": 4096},
                {"name": "warm_experts", "purpose": "RAM-cache contiguous expert chunks", "alignment_bytes": 4096},
                {"name": "cold_experts", "purpose": "NVMe-resident large sequential expert chunks", "alignment_bytes": 4096},
            ],
            "direct_io": {"o_direct_required": True, "io_uring_preferred": True, "iopoll_optional": True, "optimal_block_size_runtime_probed": True},
        },
        "juju_header_schema": {
            "sizeof_bytes": 4096,
            "alignas_bytes": 4096,
            "magic": "JUJU\0\1\0\0",
            "version_major": 1,
            "version_minor": 0,
            "fields": {
                "created_at": "uint64_unix_timestamp",
                "file_size": "uint64",
                "section_count": "uint32",
                "section_table_offset": "uint64",
                "section_table_size": "uint64",
                "model_name": "char[64]",
                "layer_count": "uint32",
                "expert_count_per_layer": "uint32",
                "top_k": "uint32",
                "hidden_dim": "uint32",
                "expert_dim": "uint32",
                "quant_type": "uint8",
                "arch_type": "uint8",
                "recommended_vram_mb": "uint32_hint_only",
                "recommended_ram_mb": "uint32_hint_only",
                "shared_weights_size": "uint64",
                "hot_section_size": "uint64",
                "warm_section_size": "uint64",
                "cold_section_size": "uint64",
                "file_hash": "sha256_header_excluded",
                "header_hash": "sha256_header",
            },
            "strategy_rule": "header hints never replace startup probing",
        },
        "juju_section_table_schema": {
            "entry_fields": ["type", "offset", "size", "uncompressed_size", "compression", "checksum_xxhash128", "name"],
            "section_types": {
                "MODEL_META": "0x0001",
                "PREDICTOR": "0x0002",
                "BUDDY_MAP": "0x0003",
                "TIER_HINT": "0x0004",
                "SHARED_WEIGHTS": "0x0010",
                "HOT_EXPERTS": "0x0011",
                "WARM_EXPERTS": "0x0012",
                "COLD_EXPERTS": "0x0013",
                "CUSTOM": "0xFF00",
            },
            "unknown_section_rule": "skip_if_unknown_and_checksum_valid",
            "alignment_bytes": 4096,
        },
        "juju_expert_chunk_schema": {
            "header_alignment_bytes": 4096,
            "data_alignment_bytes": 4096,
            "level_1_unit": "expert_offload_unit",
            "level_2_unit": "expert_internal_io_segment",
            "fields": [
                "layer_id", "expert_id", "total_size", "data_offset",
                "access_frequency", "task_affinity[8]", "buddy_expert_ids[4]", "prefetch_hints[8]",
                "optimal_read_size", "tier_hint", "chunk_hash_xxhash64",
                "segment_count", "segments[4]", "supports_partial", "partial_accuracy",
            ],
            "segment_fields": ["offset", "size", "priority", "can_partial_exec", "matrix_role", "row_range", "column_range", "checksum"],
            "segment_size_rule": "512KB_to_2MB_for_nvme_to_fast_mem_unless_runtime_probe_finds_a_better_block_size",
            "segment_count_rule": "default_1_for_small_experts_2_to_4_for_large_experts_or_low_compute_overlap",
            "next_chunk_rule": "next ExpertChunkHeader starts at the next 4K boundary",
        },
        "juju_index_contract": {
            "file": "model.juju.idx",
            "mutable": True,
            "checkpoint_policy": "periodic_tokens_or_request_end",
            "idx_header_fields": ["magic=JUJUIDX", "version", "juju_hash", "last_updated", "total_tokens_processed", "record_count"],
            "expert_record_fields": [
                "layer_id", "expert_id", "access_count", "last_access_token", "recent_frequency", "task_frequency[8]",
                "current_tier", "tier_locked", "prefetch_hits", "prefetch_misses", "physical_offset",
            ],
            "trajectory_entry_fields": ["pattern_hash", "predicted_experts[32]", "confidence", "observation_count", "task_type", "language_framework"],
            "predictor_window_experts": 32,
            "buddy_fallback_confidence_threshold": 0.62,
            "language_framework_affinity_tables": {
                "enabled": True,
                "task_types": ["general_chat", "coding", "math", "tool_use", "translation", "summarization", "retrieval", "multimodal"],
                "coding_frameworks": ["python", "rust", "javascript", "typescript", "cpp", "cuda", "android", "shell"],
                "record_fields": ["task_affinity[8]", "language_framework", "recent_frequency", "prefetch_hits", "prefetch_misses"],
            },
            "task_affinity_scoring": {
                "enabled": True,
                "score_formula": "0.42*temporal_frequency + 0.24*spatial_layer_locality + 0.20*semantic_hint + 0.14*task_affinity",
                "auto_reweight": "adjust weights after telemetry windows using prefetch_hit_rate_delta",
                "cold_start_default": "temporal=0.25, spatial=0.35, semantic=0.15, task_affinity=0.25",
            },
            "hardware_cache_file": "model.juju.hw",
            "hardware_cache_fields": ["nvme_seq_bw_gbps", "pcie_bw_gbps", "optimal_queue_depth", "optimal_block_size", "gds_available", "iopoll_available", "measured_at"],
            "promotion_rule": "COLD_to_WARM_to_HOT changes priority and cache placement, not immutable weight bytes",
            "demotion_rule": "HOT_to_WARM_to_COLD when hit rate decays, memory pressure rises, batch budget shrinks, or qkv_spill_pressure dominates",
            "dynamic_reclassification": "recompute tier score when batch_size, token_budget, task_type, or available_ram changes",
        },
        "prediction_contract": {
            "probe": {"enabled": True, "predictor": "gate_initialized_lookahead", "router_distillation": True, "critical_path_overhead_allowed": False},
            "quantized_draft_shard_oracle": {"enabled": True, "source": "small_on_device_quantized_moe_draft", "purpose": "predict_future_expert_sequence_without_training_full_predictor", "vram_residency": "tiny_always_on_if_budget_allows"},
            "shadow_model_predictor": {"enabled": True, "shard_bits": 4, "residency": "vram_always_if_budget_allows", "lookahead_layers": 3, "kv_alignment": "sync_every_N_tokens", "token_alignment": "verify_shadow_token_against_target", "drift_correction": "layer_wise_gate_alignment"},
            "cross_layer_gate": {"enabled": True, "formula": "next_layer_expert_logits = W_cross * current_layer_gate_output + bias", "gpu_overhead": "no_extra_forward_when_gate_output_available"},
            "residual_based_prefetch": {"enabled": True, "input": "inter_layer_residual_delta", "target": "high_workload_experts"},
            "prescope": {"enabled": True, "llapor_layer_aware": True, "presched_cross_layer": True, "asyncio_no_wait_bubbles": True},
            "spmoe": {"enabled": True, "cutoff_layer_policy": True, "batched_io": True, "prefetch_stream": "background_worker_stream"},
            "buddymoe": {"enabled": True, "coactivation_map": True, "fallback_when_miss": "cached_buddy_if_confidence_below_threshold", "confidence_threshold": 0.62},
            "trajectory_table": {"predicted_expert_window": 32, "pattern_hash": "rolling_task_and_layer_aware_hash", "observation_decay": "ema_0_92"},
            "semantic_hint": {"combine_with_task_type": True, "task_affinity_fields": 8},
            "language_framework_affinity_tables": {"enabled": True, "languages": ["python", "rust", "javascript", "typescript", "cpp", "cuda", "android", "shell"]},
            "temporal_spatial_expert_scheduler": {
                "enabled": True,
                "score_formula": "w_t*temporal + w_s*spatial + w_sem*semantic + w_task*task_affinity",
                "initial_weights": {"temporal": 0.42, "spatial": 0.24, "semantic": 0.20, "task_affinity": 0.14},
                "feedback_update": "increase weights that improve prefetch_hit_rate and reduce qkv_spill_pressure",
            },
            "speculative_decode_prefetch_sync": {"enabled": True, "draft_verify_stage": "align_with_three_stage_expert_prefetch", "drop_unused_draft_prefetch": True},
            "safety_margin": {"topk_multiplier_min": 1.5, "topk_multiplier_max": 2.0, "overfetch_guard": "cutoff_layer_cache_pressure_and_qkv_spill_pressure"},
        },
        "hardware_probe_contract": {
            "gpu": ["vram_total", "vram_free", "pcie_gen", "pcie_bw_measured", "gds_available", "peer_access", "compute_capability"],
            "ram": ["ram_total", "ram_free", "max_pinned_alloc", "ram_bw_measured"],
            "nvme": ["nvme_count", "seq_bw", "rand_iops", "optimal_queue_depth", "optimal_block_size", "supports_iopoll"],
            "measurement_required": True,
            "block_size_sweep_bytes": [65536, 131072, 262144, 524288, 1048576, 2097152, 4194304],
            "queue_depth_sweep": [1, 2, 4, 8, 16, 32, 64],
            "pinned_test_size_bytes": 268435456,
        },
        "runtime_design_principles": {
            "probe_first": "run 2_to_3_second_startup_probe_then_runtime_remeasure",
            "no_strategy_hardcoding": True,
            "all_parameters_from": ["HardwareProfile", "ModelProfile", "RuntimeMetrics", "JUJU metadata hints"],
            "fallback_required": True,
            "tier_split_basis": "measured_bandwidth_and_compute_window_not_fixed_ratios",
            "pinned_overallocation_forbidden": True,
            "specs_are_hints_not_truth": True,
        },
        "offload_strategy_contract": {
            "decision_inputs": ["hardware_profile", "model_profile", "qkv_cache_profile", "expert_size", "layer_compute_ms", "runtime_metrics"],
            "nvme_methods_ranked": ["gds", "io_uring_iopoll", "io_uring", "o_direct_pread", "mmap_fallback", "read_fallback"],
            "ram_to_gpu_methods_ranked": ["gds", "pinned_async", "pinned_sync", "pageable_fallback"],
            "fallback_chain": {
                "storage_to_gpu": ["gds", "o_direct_to_pinned_ram_then_async_gpu", "o_direct_to_host_then_sync_gpu", "page_cache_fallback"],
                "nvme_io": ["io_uring_iopoll", "io_uring", "o_direct_pread", "mmap", "read"],
                "predictor": ["probe", "prescope", "trajectory_table", "semantic_hint", "hotset_static", "on_demand"],
            },
            "rtx3090_reference": {"moe_compute_ms": 13.0, "ram_to_gpu_expert_ms": 6.0, "nvme_to_ram_expert_ms_min": 15.0, "nvme_to_ram_expert_ms_max": 30.0, "max_overlapped_expert_transfers": 2},
            "prefetch_depth_formula": "ceil(nvme_transfer_ms / max(layer_compute_ms * effective_batch, epsilon)) + 1, clamped by cache_pressure_and_cutoff_layer",
            "prefill_prefetch_depth_formula": "ceil(prefill_expert_bytes / max(prefill_compute_window_ms * measured_nvme_bw_bytes_per_ms, epsilon)) + pipeline_margin",
            "decode_prefetch_depth_formula": "ceil(nvme_transfer_ms / max(decode_compute_ms * effective_batch, epsilon)) + 1",
            "gds_enable_rule": "only_when_juju_sections_are_4k_aligned_and_o_direct_dma_eligible",
            "pinned_pool_policy": "bounded_by_probe_max_pinned_and_os_pressure; never allocate enough pinned memory to trigger swap",
            "tier_split_policy": "hot_count equals experts transferable inside measured compute window; warm_count equals safe pinned capacity and measured PCIe/NVMe overlap; cold is the remainder",
            "expert_importance_quant_policy": {"hot": "source_or_fp16", "warm": "int8_or_source_quant", "cold": "int4_or_source_quant", "accuracy_guard": "disable_lower_precision_if_eval_delta_exceeds_threshold"},
        },
        "adaptive_controller_contract": {
            "update_period": "every_N_tokens_or_request_end",
            "metrics": ["gpu_wait_ratio", "io_queue_full_ratio", "pinned_alloc_fail", "prefetch_miss_rate", "available_ram", "qkv_spill_pressure"],
            "rules": [
                {"if": "gpu_wait_ratio > 0.1", "then": "increase_prefetch_depth"},
                {"if": "io_queue_full_ratio > 0.5", "then": "increase_nvme_threads_or_block_size"},
                {"if": "pinned_alloc_fail > 0", "then": "rebalance_warm_to_cold"},
                {"if": "qkv_spill_pressure > 0.20", "then": "raise_kv_spill_priority_and_reduce_qkv_warm_pages"},
                {"if": "prefetch_miss_rate > 0.15", "then": "trigger_predictor_retrain_then_enable_buddy_fallback"},
                {"if": "available_ram_below_threshold", "then": "evict_warm_to_cold"},
            ],
        },
        "gguf_to_juju_converter_contract": {
            "tool_name": "gguf2juju",
            "input_role": "GGUF source tensor and metadata reader",
            "output_files": ["<original_shard_stem>.juju", "<original_shard_stem>.juju.idx", "<original_shard_stem>.juju.hw_optional"],
            "physical_transform_required": True,
            "metadata_only_patch_is_not_final_model": True,
            "steps": [
                "read GGUF tensor table and metadata",
                "load optional access statistics",
                "classify experts into hot warm cold using statistics or conservative defaults",
                "compute buddy map from co-activation or weight similarity",
                "train or import predictor weights when sample data exists",
                "compute segment plan per expert from expert size, compute time, bandwidth, and memory pressure",
                "write JUJU header and section table",
                "physically rewrite tensors into 4K aligned JUJU sections instead of leaving GGUF order unchanged",
                "write MODEL_META PREDICTOR BUDDY_MAP TIER_HINT SHARED_WEIGHTS HOT_EXPERTS WARM_EXPERTS COLD_EXPERTS",
                "initialize mutable JUJU index file",
            ],
            "expert_order_rule": "within each tier, keep layer order and place co-used experts adjacent for sequential reads",
            "segment_order_rule": "top or high-priority segment first, then bottom or low-priority segments, all 4K aligned",
            "weight_mutation_rule": "weights may be physically reordered and segmented into JUJU sections but tensor payload bytes remain semantically unchanged",
            "co_used_expert_physical_adjacency_required": True,
            "hot_warm_cold_section_order_required": True,
        },
        "qkv_cache_contract": {
            "required": True,
            "default_path": "quantized_qkv_cache",
            "plain_fp16_bf16_persistent_kv_allowed": False,
            "storage_isolation": "separate_kv_spill_file_or_region",
            "engine_compatibility": "storage_llm_current_packed_qkv_cache",
            "spill_pressure_metric": "qkv_spill_pressure",
            "spill_pressure_response": "raise_kv_spill_priority_and_reduce_qkv_warm_pages_before_expert_cache_starves",
            "direct_storage_optional": True,
            "engine_must_route_attention_through_qkv": True,
        },
        "expert_segmentation_contract": {
            "enabled": True,
            "two_level_model": {"level_1": "expert_offload_prediction_unit", "level_2": "segment_io_overlap_unit"},
            "default_segment_count": "runtime_or_converter_decided",
            "nvme_to_fast_mem_segment_size_range_bytes": [524288, 2097152],
            "large_expert_threshold_bytes": 104857600,
            "small_expert_no_split_threshold_bytes": 20971520,
            "split_if": [
                "expert_bytes_over_100MB",
                "transfer_time_exceeds_compute_time",
                "vram_pressure_requires_top_segment_residency",
                "partial_execution_latency_goal_enabled",
            ],
            "do_not_split_if": [
                "expert_bytes_under_20MB",
                "nvme_bandwidth_prefers_large_sequential_read",
                "ram_cache_can_hold_entire_expert",
                "metadata_or_iops_overhead_exceeds_overlap_gain",
            ],
            "decision_formula": "ratio = compute_time_ms / transfer_time_ms; if ratio >= 1 keep whole expert, else split into segments that fit the compute overlap window",
            "segment_cap": 4,
        },
        "moepic_segment_contract": {
            "enabled": True,
            "top_segment": {"priority": 0, "residency": "tier0_or_tier1_preferred", "purpose": "partial_compute_and_cache_hit_acceleration"},
            "bottom_segment": {"priority": 2, "residency": "demand_load", "purpose": "complete_exact_expert_compute"},
            "supports_partial": True,
            "partial_accuracy_field_required": True,
        },
        "moeshard_matrix_split_contract": {
            "enabled": True,
            "supported_splits": ["row_wise", "column_wise", "top_bottom", "hybrid"],
            "ffn_roles": ["W_gate", "W_up", "W_down"],
            "activation_boundary_rule": "SwiGLU_gate_up_down_split_points_must_align_to_pre_activation_row_boundaries; fused_gate_up_segments_must_not_split_inside_activation",
            "load_balance_goal": "balance segment transfer and fused kernel execution across rows and columns",
        },
        "chunk_io_contract": {
            "compute_io_separate_streams_required": True,
            "transfer_order_rule": "next_needed_first_no_random_order",
            "io_uring_batch_submit_range": [16, 64],
            "pinned_memory_reuse_required": True,
            "co_used_expert_physical_adjacency": True,
            "physical_adjacency_requires_juju_rewrite": True,
            "queue_saturation_guard": True,
            "metadata_overhead_guard": True,
        },
        "final_model_structure_contract": {
            "structure_name": "JUJU_MAX_MOE_STRUCTURE_V1",
            "locked": True,
            "target": "maximum_offload_overlap_for_moe_and_qkv_runtime",
            "gguf_role": "source_import_only",
            "exact_mode_default": True,
            "approximate_accelerators_require_quality_gate": True,
            "components": [
                {"name": "SHARED_CORE", "contains": ["embedding", "attention", "norm", "shared_expert"], "default_residency": "COMPUTE_MEM_required", "shared_expert_residency": "VRAM_mandatory_when_gpu_available"},
                {"name": "ROUTING_BRAIN", "contains": ["router_state", "SeqTopK_budget", "semantic_hint", "trajectory_table", "task_affinity"], "default_residency": "FAST_MEM"},
                {"name": "PREDICTOR_BANK", "contains": ["PROBE_predictor", "PreScope_LLaPor", "SP_MoE_cutoff_policy"], "default_residency": "FAST_MEM_or_COMPUTE_MEM"},
                {"name": "EXPERT_GROUPS", "contains": ["hot_experts", "warm_experts", "cold_experts", "co_used_groups"], "default_residency": "tiered"},
                {"name": "EXPERT_SEGMENTS", "contains": ["MoEpic_top", "MoEpic_bottom", "MoESHARD_rows", "MoESHARD_columns", "fine_io_chunks"], "default_residency": "segment_priority_based"},
                {"name": "BUDDY_FALLBACK", "contains": ["buddy_pairs", "coactivation_score", "quality_gate"], "default_residency": "FAST_MEM"},
                {"name": "QKV_SUBSYSTEM", "contains": ["quantized_qkv_cache", "qkv_aux", "qkv_spill", "qkv_residency"], "required": True},
                {"name": "CODE_LOCALITY_CACHE", "contains": ["shared_prefix_kv", "candidate_expert_hotset", "assembly_equivalent_group", "middle_layer_boost"], "task": "coding"},
                {"name": "TIER_SCHEDULER", "contains": ["HardwareProfile", "TierCapability", "PipelineBudget", "ExecutionPath", "AdaptiveController"], "default_residency": "runtime"},
            ],
            "engine_rule": "engine implementation must follow this component graph; optimizations tune policies and sections, not top-level structure",
        },
        "routing_optimization_contract": {
            "primary_predictors": ["semantic_hint", "trajectory_table", "SeqTopK_sequence_budget", "PROBE_router_distilled", "PreScope_layer_aware", "SP_MoE_cutoff"],
            "fallback_predictors": ["expert_pattern_signal", "code_task_hotset", "static_hot_expert", "on_demand_exact"],
            "fmoe_policy": {"semantic_near_window": True, "trajectory_far_window": True, "hit_rate_feedback": True},
            "seqtopk_policy": {"enabled": True, "budget": "sequence_level_T_times_K", "hard_token_extra_budget": True, "immutable_weight_file": True},
            "expert_pattern_policy": {"selection_pattern_is_signal": True, "store_in_idx": True, "promote_patterns_to_prefetch_graph": True},
            "comoe_policy": {"resource_aware": True, "edge_and_mobile_supported": True, "tier_map_driven": True, "cache_schedule_prefetch_unified": True},
        },
        "code_generation_moe_contract": {
            "enabled": True,
            "evidence": {"same_token_jaccard": 0.649, "different_token_jaccard": 0.175, "middle_layer_peak": "L14_to_L20", "compiled_top3_assembly_equivalent_group_fraction": 0.67},
            "policies": [
                "shared_prefix_candidate_cache",
                "candidate_group_expert_hotset_reuse",
                "middle_layer_priority_boost",
                "compile_equivalent_group_pruning",
                "language_framework_affinity_tables",
                "best_of_n_and_beam_search_batching",
            ],
            "routing_effect": "coding requests get stronger expert cache reuse and deeper prefetch confidence than generic chat requests",
        },
        "structure_finalization_contract": {
            "model_structure_change_required_after_this": False,
            "allowed_future_changes": ["CUSTOM_section", "policy_table", "predictor_weights", "mutable_idx_stats", "hardware_cache", "engine_kernel_implementation"],
            "forbidden_future_changes_without_new_major_version": ["remove_QKV_required_path", "remove_three_tier_abstraction", "remove_expert_segment_level", "remove_JUJU_primary_container", "replace_exact_mode_default"],
            "versioning_rule": "top_level_structure_changes_require_JUJU_major_version_increment",
        },
        "juju_container_contract": {
            "format_name": "JUJU",
            "schema_family": "custom_moe_engine_format",
            "gguf_role": "conversion_source_only",
            "files": {"weights": "model.juju", "runtime_index": "model.juju.idx", "hardware_cache": "model.juju.hw"},
            "immutable_weight_file": True,
            "mutable_index_file": True,
            "dynamic_all_hardware": True,
            "qkv_required_default": True,
        },
        "universal_tier_contract": {
            "tier_names": {"0": "COMPUTE_MEM", "1": "FAST_MEM", "2": "SLOW_MEM"},
            "principle": "engine code talks to tiers, backend mapping binds tiers to hardware at startup",
            "hardware_mappings": [
                {"case": "gpu_ram_nvme", "compute_backend": ["cuda", "rocm", "vulkan"], "tier0": "vram", "tier1": "ram", "tier2": "nvme"},
                {"case": "gpu_ram_only", "compute_backend": ["cuda", "rocm", "vulkan"], "tier0": "vram", "tier1": "ram", "tier2": "ram_swap_or_none"},
                {"case": "low_vram_gpu", "compute_backend": ["cuda", "rocm", "vulkan"], "tier0": "vram", "tier1": "ram", "tier2": "nvme"},
                {"case": "cpu_ram_nvme", "compute_backend": ["cpu"], "tier0": "ram", "tier1": "nvme", "tier2": "nvme"},
                {"case": "cpu_ram_only", "compute_backend": ["cpu"], "tier0": "ram", "tier1": "ram", "tier2": "none"},
                {"case": "low_ram_cpu", "compute_backend": ["cpu"], "tier0": "ram", "tier1": "nvme", "tier2": "none"},
                {"case": "unified_memory", "compute_backend": ["metal", "uma_vulkan", "cpu_uma"], "tier0": "unified_memory", "tier1": "unified_memory", "tier2": "nvme"},
            ],
            "tier_capability_fields": ["capacity", "bandwidth_gbps", "latency_us", "supports_async", "supports_dma", "supports_cpu_compute", "supports_ndp_compute"],
            "cpu_compute_path": {"enabled": True, "target_experts": "warm", "method": "amx_avx512_matmul_when_available_else_avx2", "overlap": "cpu_compute_warm_experts_while_gpu_runs_hot_experts"},
            "ndp_compute_path": {"enabled": "hardware_probe_only", "target_experts": "cold_or_ndp_saturating_warm", "method": "dimm_ndp_if_available"},
            "backend_detection_order": ["cuda", "rocm", "metal", "vulkan", "cpu"],
        },
        "pipeline_budget_contract": {
            "stall_free_condition": "compute_time_current_expert_ms >= transfer_time_next_expert_ms",
            "compute_time_source": "runtime_probe_or_profiled_layer_compute",
            "t1_to_t0_formula_ms": "expert_bytes / tier1.bandwidth_gbps / 1e6",
            "t2_to_t1_formula_ms": "expert_bytes / tier2.bandwidth_gbps / 1e6",
            "prefetch_depth_formula": "max(1, ceil(t2_to_t1_ms / max(compute_ms * effective_batch, epsilon)) + 1)",
            "prefill_prefetch_depth_formula": "max(2, ceil(t2_to_t1_ms / max(prefill_compute_ms * prefill_effective_batch, epsilon)) + 1)",
            "decode_prefetch_depth_formula": "max(1, ceil(t2_to_t1_ms / max(decode_compute_ms * decode_effective_batch, epsilon)) + 1)",
            "stage_separator": "first_token_generated",
            "overlap_prefill_and_decode_when_scheduler_allows": True,
            "tier0_expert_count_formula": "tier0.capacity / expert_bytes",
            "tier1_expert_count_formula": "tier1.capacity / expert_bytes",
            "io_threads_formula": "min(8, ceil(t2_to_t1_ms / compute_ms) * 2)",
            "block_size_rule": "512KB when async storage tier exists, otherwise 4KB conservative fallback",
        },
        "execution_path_contract": {
            "selector_inputs": ["tier_map", "model_profile", "pipeline_budget", "qkv_cache_profile"],
            "paths": [
                {"name": "FULL_COMPUTE_MEM", "condition": "tier0.capacity >= model.total_size", "io_path": "none"},
                {"name": "DIRECT_STORAGE_TO_COMPUTE", "condition": "tier2.supports_dma and direct_storage_available", "io_path": "tier2_to_tier0_dma"},
                {"name": "THREE_TIER_PIPELINE", "condition": "tier1.capacity > expert_bytes * 10", "io_path": "tier2_to_tier1_to_tier0_async"},
                {"name": "ROTATING_BUFFER", "condition": "tier1.capacity > expert_bytes * 2", "io_path": "small_fast_mem_double_buffer"},
                {"name": "CPU_WITH_CACHE", "condition": "compute_backend == cpu and tier0.capacity > model.total_size * 0.3", "io_path": "mmap_or_sequential_cache"},
                {"name": "STORAGE_STREAMING", "condition": "fallback", "io_path": "direct_stream"},
                {"name": "SPECULATIVE_DRAFT_ORACLE", "condition": "quantized_shadow_shard_in_vram", "io_path": "shadow_prediction_multi_layer_lookahead", "accuracy_target": 0.99, "alignment": "kv_sync_with_full_precision_model"},
            ],
            "auto_degrade": True,
        },
        "path_specific_io_hint_schema": {
            "section_entry_extension": {
                "sequential_block_size": "uint64",
                "random_block_size": "uint64",
                "prefetch_distance": "uint8",
                "mmap_friendly": "bool",
            },
            "path_rules": {
                "DIRECT_STORAGE_TO_COMPUTE": "requires 4K aligned expert chunks and DMA eligibility",
                "THREE_TIER_PIPELINE": "uses hot warm cold contiguous section order",
                "ROTATING_BUFFER": "prefers padded uniform expert chunk sizes",
                "CPU_WITH_CACHE": "prefers mmap and sequential advice when available",
                "STORAGE_STREAMING": "prefers layer order inside each tier block",
            },
        },
        "layer_order_index_schema": {
            "purpose": "provide path-specific offsets without changing immutable weight bytes",
            "gpu_offsets": "layer_major_expert_offsets_for_tiered_gpu_paths",
            "cpu_offsets": "layer_major_expert_offsets_for_sequential_cpu_streaming",
            "storage_stream_offsets": "strict_layer_order_offsets_for_minimum_memory_path",
        },
        "format_layout_contract": {
            "mode": "gguf_metadata_driven_or_new_container",
            "payload_mutation_required": False,
            "layout_sections": [
                {"name": "shared_weights_section", "placement": "front", "residency": "device_required_for_shared_expert", "contents": ["attention", "norm", "embedding", "shared_expert"]},
                {"name": "hot_expert_block", "placement": "front_contiguous", "residency": "device_or_ram", "sort_key": "access_frequency_then_task_affinity"},
                {"name": "warm_expert_block", "placement": "middle_contiguous", "residency": "ram_cache", "sort_key": "recent_hit_rate_then_prefetch_probability"},
                {"name": "cold_expert_block", "placement": "tail_contiguous", "residency": "nvme_or_remote", "sort_key": "large_sequential_chunks"},
                {"name": "kv_cache_metadata", "placement": "sidecar_or_dedicated_region", "residency": "runtime_managed"},
            ],
            "chunk_index_fields": ["layer_id", "expert_id", "tensor_names", "offset", "size", "alignment", "tier", "task_affinity", "access_frequency", "prefetch_successor_ids"],
            "alignment_bytes": 4096,
            "direct_io_eligible": True,
            "gds_eligible_only_after_juju_4k_rewrite": True,
            "sequential_read_bias": True,
            "random_seek_penalty_note": "MoE expert random reads can collapse NVMe bandwidth, so hot/warm/cold physical clustering and row-column bundling are exposed as metadata hooks",
        },
        "engine_pipeline_contract": {
            "thread_roles": [
                "gpu_compute_layer_n",
                "ram_to_device_transfer_layer_n_plus_1",
                "disk_to_ram_prefetch_layer_n_plus_2",
                "telemetry_and_tier_update",
            ],
            "async_io_backends": ["io_uring_iopoll", "pread_fallback", "remote_http_range"],
            "transfer_buffers": "pinned_memory_slab_pool",
            "double_buffering": True,
            "pipeline_goal": "remove disk and host transfer from the decode critical path",
            "gpudirect_storage_optional": True,
            "fallback_rule": "if direct storage or pinned memory is unavailable, keep chunk order and prefetch distance but use conservative host buffers",
        },
        "expert_scheduler_contract": {
            "prediction_inputs": ["current_layer", "current_expert_pattern", "semantic_hint", "task_type", "language_hint", "framework_hint", "shared_prefix_candidate_id"],
            "prediction_tables": ["trajectory_table", "semantic_hint_index", "task_affinity_table", "successor_dependency_graph", "hot_warm_cold_tier_map"],
            "coding_task_policy": {
                "shared_prefix_reuse": True,
                "best_of_n_candidate_cache": True,
                "middle_layer_priority_layers": "runtime_detect_or_L14_to_L20_hint",
                "compile_equivalent_group_pruning": True,
            },
            "tier_update_rule": "promote on repeated hits and demote cold random-seek offenders after telemetry windows",
            "per_layer_cache_budget": {"enabled": True, "policy": "allocate more hot quota to layers with higher reuse and sensitivity", "cold_start": "shallow_and_middle_layers_get_priority_until_runtime_stats_override"},
            "layer_position_weight": {"early_layers_boost": 2.0, "middle_layers_boost": 1.25, "late_layers_penalty": 0.5, "basis": "shallow_favoring_hit_rate_evidence"},
            "phase_aware_scheduling": {"prefill": "wide_parallel_prefetch_and_token_reorder", "decode": "low_latency_next_layer_prefetch"},
        },
        "runtime_update_files": {
            "expert_access_statistics": {
                "mutable": True,
                "recommended_path_suffix": ".expert.stat.jsonl",
                "fields": ["layer_id", "expert_id", "access_freq", "task_type_affinity", "language_affinity", "last_hit_step", "prefetch_hit_rate"],
                "reason": "update runtime statistics without rewriting model weights",
            },
            "expert_map": {
                "mutable": True,
                "recommended_path_suffix": ".expert.map.json",
                "fields": ["trajectory_key", "successor_experts", "semantic_bucket", "tier", "confidence"],
            },
            "kv_cache_spill": {
                "mutable": True,
                "recommended_path_suffix": ".kv.spill",
                "reason": "avoid contention between weight streaming and KV spill traffic",
            },
        },
        "research_basis": [
            {
                "id": "gguf_spec",
                "intent": "store additional namespaced metadata inside GGUF without external files",
                "metadata_use": "offload.* keys are GGUF string/number KVs and the tensor payload is stream-copied unchanged",
            },
            {
                "id": "code_moe_shared_prefix_locality_2604_17182",
                "intent": "exploit MoE routing overlap in shared-prefix code generation candidates",
                "metadata_use": "same-token Jaccard 0.649, different-token Jaccard 0.175, L14-L20 crossing, top-three assembly-equivalent group concentration 0.67, and compile-equivalence pruning hooks",
                "source": "arXiv:2604.17182",
            },
            {
                "id": "powerinfer_hot_cold_split",
                "intent": "keep power-law hot neurons or experts resident and process cold units through slower tiers",
                "metadata_use": "hot/cold classification, adaptive predictor hooks, sparse operator dispatch, and device-resident hotset policy",
                "source": "arXiv:2312.12456",
            },
            {
                "id": "fmoe_iteration_prefetch",
                "intent": "predict MoE experts with semantic similarity near the prefetch window and trajectory patterns farther out",
                "metadata_use": "expert hotness, semantic prefetch, trajectory cache, and hit-rate telemetry hooks",
            },
            {
                "id": "seqtopk_sequence_budget",
                "intent": "allocate a fixed sequence-level T*K expert budget instead of forcing K experts per token",
                "metadata_use": "sequence budget, per-token min/max bounds, token difficulty, expert-score history cache, and high-sparsity routing hooks",
                "source": "arXiv:2511.06494",
            },
            {
                "id": "local_routing_consistency_srp_sch",
                "intent": "decide if an MoE model is offload-friendly by measuring segment routing and cache hit consistency",
                "metadata_use": "SRP/SCH telemetry, cache size around two times active experts, and model-specific offload suitability gates",
                "source": "arXiv:2505.16056",
            },
            {
                "id": "oracle_moe_semantic_locality",
                "intent": "reduce expert swapping by routing in a compact semantic locality space",
                "metadata_use": "semantic grouping, oracle-space bucket ids, inter-token expert variation telemetry, and swap reduction hooks",
                "source": "ICML 2025 OpenReview Oracle-MoE",
            },
            {
                "id": "comoe_edge_scheduler",
                "intent": "jointly optimize expert aggregation and offloading under edge resource, network, and input conditions",
                "metadata_use": "expert prediction, prefetching, caching, scheduling, multi-tier storage, and dynamic resource-state hooks",
                "source": "arXiv:2508.09208",
            },
            {
                "id": "specexec_offload_speculation",
                "intent": "combine speculative decoding with offloaded inference by generating and verifying draft tokens in bandwidth-friendly batches",
                "metadata_use": "draft batch sizing, transfer overlap, and verification scheduling hooks",
            },
            {
                "id": "swift_self_speculation",
                "intent": "support self-speculative decoding through uniform layer skipping without a separate draft model",
                "metadata_use": "layer skip masks, verification layers, and fallback hooks",
            },
            {
                "id": "flexinfer_async_preservation",
                "intent": "hide SSD or host-memory I/O with async prefetch, balanced locking, and budget-driven tensor preservation",
                "metadata_use": "memory budget, preserved tensor set, stream distance, and lock balancing hooks",
                "source": "arXiv:2503.03777",
            },
            {
                "id": "kvswap_mobile_storage",
                "intent": "adapt KV cache offloading to mobile and embedded storage bandwidth instead of assuming PCIe-class transfer",
                "metadata_use": "storage bandwidth class, QKV page size, eviction threshold, compact in-memory metadata, hardware-aware read pattern, and slow-device fallback hooks",
                "source": "arXiv:2511.11907",
            },
            {
                "id": "ssd_moe_energy",
                "intent": "treat SSD expert loading energy and sparse expert activation as part of placement policy",
                "metadata_use": "expert activation ratio, energy-aware prefetch, grouped SSD reads, and cold expert demotion hooks",
            },
            {
                "id": "mlp_offload_multilevel_multipath",
                "intent": "hide critical-path I/O through GPU, CPU RAM, and disk multi-level and multi-path scheduling",
                "metadata_use": "path rank, level rank, queue depth, barrier, write/read contention, and fallback hooks",
                "source": "arXiv:2509.02480",
            },
            {
                "id": "gpudirect_storage_path",
                "intent": "allow NVMe or remote storage to DMA directly into device memory when platform support is present",
                "metadata_use": "optional direct-storage path, DMA eligibility, alignment, fallback-to-RAM path, and CPU-bypass telemetry hooks",
                "source": "NVIDIA GPUDirect Storage docs",
            },
            {
                "id": "runtime_design_principles",
                "intent": "make JUJU a measured adaptive runtime format rather than a hardcoded strategy blob",
                "metadata_use": "probe-first rule, fallback chain, HardwareProfile-derived parameters, bandwidth-derived tier split, and pinned-memory safety caps",
            },
            {
                "id": "juju_header_section_chunk_schema",
                "intent": "define exact 4K-aligned header, section table, and expert chunk structures for direct-I/O engines",
                "metadata_use": "JujuHeader fields, SectionEntry fields, ExpertChunkHeader fields, checksum, compression, and unknown-section skip rules",
            },
            {
                "id": "gguf_to_juju_converter",
                "intent": "convert GGUF into immutable JUJU weights plus mutable JUJU index and optional hardware cache",
                "metadata_use": "converter steps, expert classification, buddy map, predictor weights, hot/warm/cold section writing, and initial index generation",
            },
            {
                "id": "juju_immutable_container",
                "intent": "treat JUJU as the primary engine container and GGUF as the import/conversion source",
                "metadata_use": "model.juju immutable weight sections, model.juju.idx mutable runtime state, 4KB aligned direct-I/O sections, and conversion manifest hooks",
            },
            {
                "id": "qkv_required_default",
                "intent": "make quantized QKV cache mandatory and default for attention instead of optional plain KV persistence",
                "metadata_use": "QKV cache schema, QKV spill isolation, attention routing requirement, and direct-storage option hooks",
            },
            {
                "id": "probe_gate_initialized_predictor",
                "intent": "predict upcoming expert hotspots through a router-distilled lookahead predictor while keeping control work off the critical path",
                "metadata_use": "predictor weight offsets, hiding window, replication and assignment planner hooks",
                "source": "arXiv:2602.00509",
            },
            {
                "id": "prescope_llapor_presched",
                "intent": "combine layer-aware expert prediction, cross-layer scheduling, and async I/O for resource-constrained MoE inference",
                "metadata_use": "LLaPor predictor section, PreSched plan table, AsyncIO queue configuration, and PCIe contention guards",
                "source": "arXiv:2509.23638",
            },
            {
                "id": "spmoe_cutoff_policy",
                "intent": "bound speculative decoding expert prefetch with a cutoff layer policy to avoid overfetch and cache thrashing",
                "metadata_use": "cutoff layer, draft-target correspondence, batched I/O, prefetch thread, and per-layer empirical profile hooks",
                "source": "arXiv:2510.10302",
            },
            {
                "id": "buddymoe_graceful_miss",
                "intent": "replace a missed expert with a cached buddy expert when co-activation and quality gates allow it",
                "metadata_use": "buddy map offset, co-activation scores, token activation entropy gate, distribution gate, and fallback policy hooks",
            },
            {
                "id": "serverlessllm_chunk_pool",
                "intent": "make checkpoint loading chunk based with in-memory address mapping and multi-stage loading across storage tiers",
                "metadata_use": "chunk pool id, chunk offset, tensor-to-memory mapping, loader stage, and copy-efficient data path hooks",
                "source": "ServerlessLLM OSDI 2024",
            },
            {
                "id": "llm_in_flash_bundling",
                "intent": "reorder or describe bundles so flash reads follow large sequential row-column groups instead of random expert seeks",
                "metadata_use": "bundle id, bundled rows, bundled columns, contiguous byte range, and sequential prefetch hint",
                "source": "LLM in a Flash",
            },
            {
                "id": "expert_tiered_physical_layout",
                "intent": "expose a future GGUF extension or new container layout that stores shared weights, hot experts, warm experts, and cold experts in contiguous regions",
                "metadata_use": "tier table, section offsets, section sizes, direct-I/O alignment, affinity tags, and dependency graph hooks",
            },
            {
                "id": "three_stage_async_pipeline",
                "intent": "coordinate compute, RAM-to-device transfer, and disk-to-RAM prefetch as separate asynchronous stages",
                "metadata_use": "N/N+1/N+2 prefetch distance, stream id, queue depth, pinned slot id, and fallback hooks",
            },
            {
                "id": "mutable_expert_stats_sidecar",
                "intent": "record runtime expert access statistics without mutating the immutable weight container",
                "metadata_use": "stat sidecar schema, expert map schema, task affinity fields, and promotion/demotion policy hooks",
            },
        ],
        "items": items,
    }


SOURCE_WEIGHT_QUANT = infer_source_weight_quantization(SOURCE_SUBDIR, SOURCE_FILE_PREFIX, SOURCE_FILE_NAME)

MODEL_PROFILE = {
    "model_id": os.environ.get("MODEL_ID", SOURCE_FILE_PREFIX.lower().replace("/", "_").replace("-", "_")),
    "artifact_stem": SOURCE_FILE_PREFIX,
    "source_repo": SOURCE_HF_REPO_ID,
    "source_subdir": SOURCE_SUBDIR,
    "source_file_prefix": SOURCE_FILE_PREFIX,
    "source_file_count": SOURCE_FILE_COUNT,
    "format_maximization_target_count": FORMAT_MAXIMIZATION_TARGET_COUNT,
    "target_repo": TARGET_HF_REPO_ID,
    "target_subdir": TARGET_SUBDIR,
    "format_maximization_5000": build_format_maximization_catalog(FORMAT_MAXIMIZATION_TARGET_COUNT),
    "base_model_repo": BASE_MODEL_REPO,
    "arch_meta": {
        "model_family": "runtime_read_from_gguf_general.architecture",
        "n_layers": None,
        "moe_layers": "runtime_read_or_infer_from_gguf_tensor_names",
        "experts_per_moe_layer": "runtime_read_or_infer_from_gguf_tensor_names",
        "routed_experts_per_token": "runtime_read_or_model_config",
        "hidden_dim": None,
        "expert_intermediate_dim": None,
        "n_heads": None,
        "n_kv_heads": None,
        "head_dim": 128,
        "vocab_size": None,
        "max_seq_len": None,
        "attention_type": "runtime_read_from_gguf",
        "norm_type": "runtime_read_from_gguf_or_kernel_probe",
        "activation": "runtime_read_from_gguf_or_kernel_probe",
        "positional_encoding": "runtime_read_from_gguf",
        "tie_embeddings": "runtime_read_from_gguf",
    },
    "qkv_cache_schema": {
        "standard": "qkv_packed",
        "plain_kv_persistent_storage": False,
        "plain_kv_fallback": "debug_only_outside_hot_path",
        "append_path": "projection_scratch_to_qkv_page_append",
        "attention_path": "packed_qkv_attention",
        "offload_unit": "qkv_page",
        "k_bits": QKV_K_BITS,
        "v_bits": QKV_V_BITS,
        "group_size": QKV_GROUP_SIZE,
        "page_size_tokens": QKV_PAGE_SIZE_TOKENS,
        "append_granularity_tokens": 1,
        "layout": "layer_kvhead_page_token_dim",
        "rotation": {"enabled": True, "type": "hadamard_signs_or_materialized_matrix", "seed": 1234},
        "lloyd_max": {"enabled": True, "distribution": "beta_gaussian_approx", "store_codebook": True},
        "qjl": {"enabled": True, "seed": 5678, "materialize_matrix": False},
        "outlier": {"enabled": True, "channels": QKV_OUTLIER_CHANNELS, "bits": QKV_OUTLIER_BITS, "selection": "profile_or_static_first_n"},
        "residency": {
            "hot": "vram",
            "warm": "pinned_ram",
            "cold": "nvme_mmap",
            "eviction_policy": "sink_recent_h2o_score",
            "sink_tokens": 4,
            "recent_ratio": 0.10,
            "heavy_hitter_ratio": 0.10,
        },
    },
    "weight_quant_schema": {
        "packed_weight_policy": "preserve_source_gguf_quantization",
        "scale_policy": "runtime_read_from_gguf_tensor_types_and_blocks",
        "source_family": SOURCE_WEIGHT_QUANT["family"],
        "source_bits": SOURCE_WEIGHT_QUANT["bits"],
        "source_kernel_family": SOURCE_WEIGHT_QUANT["kernel_family"],
        "source_block_size": SOURCE_WEIGHT_QUANT["block_size"],
        "dispatch_key": SOURCE_WEIGHT_QUANT["dispatch_key"],
        "runtime_dispatch_rule": SOURCE_WEIGHT_QUANT["runtime_rule"],
        "backend_policy": {
            "required": "quant_family_specific_dequant_matmul_or_safe_rejection",
            "bits_are_not_sufficient": True,
            "iq_formats_require_importance_quant_metadata": True,
            "mxfp4_requires_mxfp4_table_not_nvfp4_table": True,
            "fallback": "do_not_route_unknown_quant_to_fp4_kernel",
        },
        "expert_bundle_unit": ["gate_proj", "up_proj", "down_proj"],
        "shared_experts_policy": {
            "dtype": "runtime_read_from_gguf",
            "fuse_with_routed_experts": SHARED_EXPERTS_FUSION_ALLOWED,
            "required_runtime_flags": REQUIRED_RUNTIME_FLAGS,
            "reason": "model_specific_policy_from_env_or_source_notes",
        },
    },
    "residency_policy": {
        "strategy": "active_set_vram",
        "anchors": {"embed": True, "lm_head": True, "final_norm": True, "layers": [0, 1]},
        "floating_layers": "runtime_derive_from_arch_meta_n_layers_excluding_anchors",
        "tier_order": ["vram", "pinned_ram", "nvme_mmap"],
        "valid_split_points": "runtime_derive_from_layer_count_and_memory_probe",
        "reentry_points": "runtime_derive_from_layer_count_and_memory_probe",
        "restore_priority_order": [
            "current_router_selected_expert_triplets",
            "lookahead_expert_triplets",
            "attention_current_layer",
            "anchors",
        ],
        "evict_priority_order": [
            "floating_ffn_down_proj",
            "late_attention_kv_related",
            "floating_ffn_gate_up",
            "early_attention",
            "anchors",
        ],
    },
    "prefetch_plan_hints": {
        "execution_order": "layer_major",
        "moe_unit": "expert_triplet_if_moe_else_dense_layer_block",
        "lookahead_window_layers_default": 2,
        "dedupe_key": ["layer", "expert"],
        "priority_classes": ["router_selected", "lookahead", "topology", "cold"],
    },
    "kernel_hints": {
        "required": ["packed_qkv_attention", "qkv_page_append", "residency_planner"],
        "preferred": ["quant_family_dequant_matmul", "scale_or_block_metadata_fusion", "async_prefetch"],
        "debug_only": ["plain_float_kv_attention"],
        "accumulate_dtype": "fp32_for_attention",
    },
    "execution_hints": {
        "backend_specific_terms_allowed": False,
        "async_transfer": {
            "enabled": True,
            "channel_count": 2,
            "channel_roles": ["compute", "host_device_transfer"],
            "channel_semantics": "independent_progress_preferred",
        },
        "pipeline_barriers": {
            "points": [4, 8, 16, 24],
            "type": "dependency_only",
            "full_device_sync_required": False,
        },
        "static_subgraph": {
            "capturable_ranges": [[0, 7], [16, 23], [32, 39], [48, 55]],
            "condition": {"min_repeat_count": 10, "static_shape_required": True},
            "runtime_translation": "backend_selects_graph_command_buffer_or_function_cache",
        },
        "fast_scratchpad": {
            "required_bytes_per_layer": 49152,
            "scope": "kernel_or_threadgroup_local",
        },
        "matrix_accelerator": {
            "preferred": True,
            "fallback": "simd",
        },
        "simd_width_hint": 32,
    },
    "dynamic_swap_triggers": [
        {
            "level": 1,
            "condition": {"accelerator_memory_pressure_gt": 0.80},
            "action": "old_qkv_pages_to_pinned_ram",
            "hysteresis": 0.05,
        },
        {
            "level": 2,
            "condition": {"accelerator_memory_pressure_gt": 0.88},
            "action": "floating_weight_blocks_to_host_memory",
            "compare_recompute_cost": True,
        },
        {
            "level": 3,
            "condition": {"accelerator_memory_pressure_gt": 0.94},
            "action": "qkv_pages_to_storage_and_floating_weights_to_host",
            "aggressive": True,
        },
        {
            "level": 4,
            "condition": {"accelerator_memory_pressure_gt": 0.98},
            "action": "force_qkv_page_eviction_and_emergency_residency_compaction",
            "qkv_force_evict_ratio": 0.30,
        },
    ],
    "memory_management_hints": {
        "pooling": {
            "accelerator_pool": {
                "preallocate": "runtime_decides",
                "slab_classes_bytes": [67108864, 268435456, 536870912, 1073741824, 2147483648],
                "eviction_policy": "lru_plus_hot_score",
                "defrag_threshold": 0.30,
            },
            "host_pool": {
                "prefer_page_locked": True,
                "prefer_large_pages": True,
                "numa_policy": "runtime_local",
            },
            "storage_pool": {
                "alignment_bytes": 4096,
                "preallocate": "runtime_decides",
            },
        },
        "buffer_reuse": {
            "alias_classes": [
                ["ffn_gate_output", "ffn_intermediate"],
                ["attention_output", "residual_input_when_safe"],
            ],
            "inplace_ops": ["rmsnorm", "rope", "silu"],
            "inter_layer_reuse": "output_input_pointer_handoff_when_shape_equal",
        },
        "transfer_compression": {
            "enabled": "runtime_decides_by_bandwidth_and_entropy",
            "host_to_accelerator": {"preferred_family": "fast_lz", "min_size_bytes": 67108864},
            "storage_to_host": {"preferred_family": "fast_zstd", "min_size_bytes": 67108864},
            "skip_when_already_entropy_dense": True,
        },
    },
    "cpu_execution_hints": {
        "capability_classes_preferred": ["matrix_tile_accelerator", "wide_simd", "numa_local_memory"],
        "threading": {
            "thread_count": "runtime_decides",
            "affinity": "performance_cores_preferred",
            "oversubscribe_io_threads": False,
        },
        "numa_hints": {
            "enabled": True,
            "node_selection": "runtime_local_to_active_accelerator_or_storage_path",
            "remote_access": "avoid_unless_required",
            "page_locked_allocation": "prefer_local_node",
            "worker_affinity": "bind_io_and_compute_workers_to_local_node_when_known",
        },
        "cache_blocking": {
            "l1": True,
            "l2": True,
            "software_prefetch_distance_bytes": 256,
        },
    },
    "kernel_config_hints": {
        "matmul": {
            "tile_m": 128,
            "tile_n": 64,
            "tile_k": 32,
            "target_occupancy": 0.75,
            "max_register_pressure": "medium",
        },
        "attention": {
            "tile_size": 128,
            "online_softmax": True,
            "accumulate_dtype": "fp32",
        },
        "packed_qkv": {
            "page_tokens": 16,
            "decode_directly_from_packed": True,
        },
    },
    "context_length_table": {
        "seq_1k": {"qkv_hot_ratio": 1.0, "weight_residency": "max_active_set", "batch_class": "large"},
        "seq_8k": {"qkv_hot_ratio": 0.70, "weight_residency": "active_set", "batch_class": "medium"},
        "seq_32k": {"qkv_hot_ratio": 0.30, "weight_residency": "reduced_active_set", "batch_class": "small"},
        "seq_128k": {"qkv_hot_ratio": 0.10, "weight_residency": "streaming", "batch_class": "single_or_micro"},
    },
    "io_schedule": {
        "backend_specific_terms_allowed": False,
        "queue_model": {
            "submission_queue_entries": 256,
            "completion_queue_entries": 512,
            "queue_depth_max": 128,
            "polling_thread_preferred": True,
            "polling_thread_affinity": "runtime_decides",
        },
        "backend_candidates": {
            "linux": "io_uring_if_available_else_pread_thread_pool",
            "windows": "overlapped_io_or_thread_pool",
            "macos": "dispatch_io_or_thread_pool",
            "portable_fallback": "bounded_thread_pool",
        },
        "priority_classes": {
            "anchor_tensors": "critical",
            "current_layer": "critical",
            "next_2_layers": "high",
            "lookahead_prefetch": "medium",
            "cold_restore": "low",
        },
        "deadline_policy": {
            "current_layer_deadline_ms": 15,
            "lookahead_deadline_ms": 50,
            "cold_deadline_ms": 250,
        },
        "runtime_translation": "native_async_io_submission_queue_or_threaded_fallback",
    },
    "streaming_attention": {
        "enabled": True,
        "mode": "qkv_paged_streaming",
        "window_size_tokens": 4096,
        "sink_tokens": 4,
        "chunked_attention": {
            "enabled": True,
            "chunk_size_tokens": 512,
            "overlap_tokens": 64,
        },
        "full_context_policy": "stream_qkv_pages_without_materializing_plain_kv",
    },
    "numerical_stability": {
        "global": {
            "attention_score_accumulate_dtype": "fp32",
            "softmax_method": "online",
            "qkv_dequant_accumulate_dtype": "fp32",
            "ffn_accumulate_dtype": "runtime_decides",
        },
        "per_role": {
            "attention": {"overflow_risk": "high_for_long_context", "softmax_method": "online"},
            "moe_expert": {"overflow_risk": "medium", "accumulate_dtype": "runtime_decides"},
            "lm_head": {"overflow_risk": "medium", "accumulate_dtype": "fp32_preferred"},
        },
        "per_layer_policy": {
            "layers_0_7": {"overflow_risk": "high", "attention_accumulate_dtype": "fp32"},
            "layers_8_63": {"overflow_risk": "medium", "attention_accumulate_dtype": "fp32"},
            "layers_64_78": {"overflow_risk": "high_for_long_context", "attention_accumulate_dtype": "fp32"},
        },
    },
    "continuous_batching": {
        "enabled": True,
        "max_concurrent_requests": "runtime_decides",
        "priority_classes": ["realtime", "interactive", "batch"],
        "prefix_caching": {
            "enabled": True,
            "cache_unit": "qkv_page",
            "share_common_prefix_pages": True,
        },
        "preemption": {
            "policy": "swap_qkv_pages_or_recompute",
            "preserve_sink_tokens": True,
        },
        "scheduler": "runtime_decides",
    },
    "batch_schedule": {
        "prefill": {
            "strategy": "accelerator_preferred_chunked",
            "chunked_prefill": True,
            "chunk_size_tokens": 512,
            "max_prefill_tokens_per_chunk": 4096,
            "overlap_with_decode": True,
        },
        "decode": {
            "strategy": "split_compute_and_transfer",
            "token_micro_batch_size": 4,
            "continuous_batching": True,
            "qkv_append_unit": "token_or_microbatch",
        },
        "scheduler": {
            "policy": "runtime_decides",
            "fairness": "priority_with_aging",
            "preemption": "swap_qkv_pages_or_recompute",
        },
    },
    "tensor_layout_schema": {
        "default_alignment_bytes": 256,
        "default_page_alignment_bytes": 4096,
        "payload": "per_tensor_layout_padding_stride_when_known",
        "missing_payload_policy": "runtime_derives_from_footer_blocks",
        "qkv_cache_layout": "layer_kvhead_page_token_dim",
        "fields": ["memory_order", "padding_bytes", "stride", "packed_axis", "tile_shape"],
    },
    "integrity_schema": {
        "file_hash": "sha256",
        "footer_hash": "sha256",
        "block_hash": "preserve_existing_sha1_when_present",
        "layer_hash": "derived_from_block_hashes_when_possible",
        "verify_on_load": True,
        "verify_on_transfer": "runtime_decides",
        "corruption_action": "reload_from_storage_source",
    },
    "activation_stats_schema": {
        "supported": True,
        "payload": "optional_per_layer_histogram_and_outlier_ratios",
        "missing_payload_policy": "runtime_uses_conservative_quant_defaults",
    },
    "sparsity_schema": {
        "supported": True,
        "payload": "optional_per_tensor_sparsity_and_format",
        "runtime_rule": "ignore_when_backend_has_no_profitable_sparse_kernel",
    },
    "runtime_monitoring_hints": {
        "sample_interval_ms": 10,
        "metrics": [
            "accelerator_memory_used",
            "interconnect_bandwidth_used",
            "host_memory_used",
            "storage_queue_depth",
            "accelerator_compute_utilization",
            "host_core_utilization",
            "qkv_page_hit_rate",
            "swap_frequency",
        ],
        "adaptive_split": {
            "enabled": True,
            "adjustment_interval_tokens": 100,
            "max_shift_per_interval_layers": 2,
        },
    },
    "thermal_power_hints": {
        "policy": "sustain_throughput_not_burst_clock",
        "runtime_probe_required": True,
        "throttle_action": "reduce_prefetch_depth_or_shift_work_to_host",
    },
    "speculative_execution_schema": {
        "enabled": "optional",
        "draft_location_preference": "host_or_small_accelerator",
        "verify_location_preference": "primary_accelerator",
        "max_draft_tokens": 5,
        "prefetch_verify_during_draft": True,
        "rollback_cost_class": "low",
    },
    "activation_checkpoint_schema": {
        "strategy": "selective",
        "checkpoint_layers": [0, 8, 16, 24, 32, 40, 48, 56, 64, 72],
        "recompute_policy": "use_when_cheaper_than_transfer_or_storage",
        "checkpoint_dtype": "bf16",
        "location_preference": "host_memory",
    },
    "mig_hints": {
        "supported_schema": True,
        "partitioning": "optional_runtime_profile",
        "preferred_partition": "runtime_decides",
        "isolation_policy": "do_not_assume_shared_memory_between_partitions",
    },
    "profiling_measured_data": {
        "present": False,
        "schema": "optional_per_layer_compute_transfer_io_bottleneck_measurements",
        "runtime_update_allowed": True,
        "missing_payload_policy": "use_cost_model_then_write_runtime_profile_sidecar",
    },
    "multi_accelerator_detail": {
        "enabled": "runtime_decides",
        "topology_source": "runtime_probe",
        "topology_schema": {
            "accelerators": "runtime_filled_list",
            "links": "runtime_filled_bandwidth_latency_matrix",
            "direct_peer_access": "runtime_filled_bool_matrix",
            "shared_host_numa_nodes": "runtime_filled_mapping",
        },
        "link_classes": ["direct_peer", "host_interconnect", "storage_direct", "network_or_none"],
        "split_strategies_supported": ["tensor_parallel", "pipeline_parallel", "expert_parallel"],
        "communication_unit": "qkv_page_or_expert_triplet",
        "p2p_required": False,
        "fallback": "single_accelerator_active_set",
    },
    "foreign_container_export": {
        "gguf": {
            "strategy": "offload.metadata_v2_json_plus_hot_scalar_keys",
            "scalar_keys": [
                "offload.format_version",
                "offload.backend_neutral",
                "offload.weight_quant_family",
                "offload.weight_bits",
                "offload.weight_kernel_family",
                "offload.qkv_k_bits",
                "offload.qkv_v_bits",
                "offload.qkv_page_size_tokens",
            ],
            "large_blocks": "json_string_values_under_offload_namespace",
        },
        "safetensors": {
            "strategy": "strings_in___metadata___under_offload_namespace",
            "large_blocks": "json_dumps_per_block",
            "sidecar_when_header_too_large": True,
        },
        "sidecar": {
            "strategy": "model.offload.json_or_metadata_v2",
            "works_with": ["gguf", "safetensors", "pytorch_bin", "custom_parts"],
        },
    },
    "backend_translation_contract": {
        "format_expresses": "intent_and_resource_needs_only",
        "runtime_translates": {
            "async_transfer.channel_count": "streams_queues_or_thread_pools",
            "pipeline_barriers": "events_barriers_or_dependency_edges",
            "static_subgraph": "graphs_command_buffers_indirect_buffers_or_function_cache",
            "matrix_accelerator": "tensor_matrix_tile_units_or_simd",
            "io_schedule.queue_model": "native_async_io_or_threaded_io",
            "multi_accelerator_detail": "device_specific_topology_runtime_plan",
        },
    },
    "compatibility": {
        "min_runtime_version": "0.6.0",
        "required_features": [
            "qkv_packed_cache_default",
            "plain_kv_persistent_storage_disabled",
            "packed_qkv_attention",
            "runtime_hardware_probe",
            "active_set_residency",
        ],
        "optional_features": [
            "direct_storage_io",
            "static_subgraph_capture",
            "speculative_decode",
            "multi_accelerator",
        ],
    },
}

print("NOTEBOOK_BUILD_ID:", NOTEBOOK_BUILD_ID)
print("SOURCE_HF_REPO_ID:", SOURCE_HF_REPO_ID)
print("SOURCE_SUBDIR:", SOURCE_SUBDIR)
print("SOURCE_FILE_PREFIX:", SOURCE_FILE_PREFIX)
print("SOURCE_FILE_COUNT:", SOURCE_FILE_COUNT)
print("TARGET_HF_REPO_ID:", TARGET_HF_REPO_ID)
print("TARGET_SUBDIR:", TARGET_SUBDIR)
print("FILES_TO_RUN:", [spec["index"] for spec in FILES_TO_RUN])
print("WORK_DIR:", WORK_DIR)


In [ ]:
import hashlib
import io
import json
import threading
import bisect
import os
import re
import struct
import time
from pathlib import Path

import requests


GGUF_TYPE_UINT8 = 0
GGUF_TYPE_INT8 = 1
GGUF_TYPE_UINT16 = 2
GGUF_TYPE_INT16 = 3
GGUF_TYPE_UINT32 = 4
GGUF_TYPE_INT32 = 5
GGUF_TYPE_FLOAT32 = 6
GGUF_TYPE_BOOL = 7
GGUF_TYPE_STRING = 8
GGUF_TYPE_ARRAY = 9
GGUF_TYPE_UINT64 = 10
GGUF_TYPE_INT64 = 11
GGUF_TYPE_FLOAT64 = 12
GGUF_TYPE_SIZE = {
    GGUF_TYPE_UINT8: 1,
    GGUF_TYPE_INT8: 1,
    GGUF_TYPE_UINT16: 2,
    GGUF_TYPE_INT16: 2,
    GGUF_TYPE_UINT32: 4,
    GGUF_TYPE_INT32: 4,
    GGUF_TYPE_FLOAT32: 4,
    GGUF_TYPE_BOOL: 1,
    GGUF_TYPE_UINT64: 8,
    GGUF_TYPE_INT64: 8,
    GGUF_TYPE_FLOAT64: 8,
}

GGUF_RUNTIME_KV_KEY_HINTS = (
    "attention",
    "attn",
    "rope",
    "rotary",
    "norm",
    "scale",
    "softcap",
    "sliding",
    "expert",
    "moe",
    "router",
    "lora",
    "head",
    "context",
    "embedding",
    "feed_forward",
    "ffn",
    "block_count",
    "vocab_size",
    "file_type",
)

GGUF_RUNTIME_KV_EXACT_KEYS = {
    "general.architecture",
    "general.name",
    "tokenizer.ggml.model",
    "tokenizer.ggml.pre",
}

GGUF_RUNTIME_KV_ALIAS_MAP = {
    "general.architecture": ("architecture", "declared_architecture", "model_type"),
    "general.name": ("model_name",),
    "block_count": ("num_hidden_layers", "n_layers"),
    "embedding_length": ("hidden_size", "hidden_dim"),
    "vocab_size": ("vocab_size",),
    "context_length": ("context_length", "max_position_embeddings"),
    "attention.head_count": ("num_attention_heads", "n_heads", "head_count"),
    "attention.head_count_kv": ("num_key_value_heads", "n_kv_heads", "head_count_kv"),
    "attention.head_count_global_kv": ("num_global_key_value_heads", "global_head_count_kv"),
    "attention.key_length": ("head_dim", "key_length"),
    "attention.value_length": ("value_head_dim", "v_head_dim", "value_length"),
    "attention.global_key_length": ("global_head_dim", "global_key_length"),
    "attention.global_value_length": ("global_value_head_dim", "global_value_length"),
    "attention.layer_norm_rms_epsilon": ("rms_norm_eps", "norm_eps"),
    "attention.q_lora_rank": ("q_lora_rank",),
    "attention.kv_lora_rank": ("kv_lora_rank",),
    "attention.qk_nope_head_dim": ("qk_nope_head_dim",),
    "attention.qk_rope_head_dim": ("qk_rope_head_dim",),
    "rope.freq_base": ("rope_theta", "theta"),
    "rope.dimension_count": ("qk_rope_head_dim", "rope_dimension_count"),
    "expert_count": ("experts_per_moe_layer", "n_experts"),
    "expert_used_count": ("routed_experts_per_token", "top_k", "num_experts_per_tok"),
    "expert_feed_forward_length": ("expert_intermediate_size", "expert_intermediate_dim"),
    "feed_forward_length": ("intermediate_size", "ffn_intermediate_size"),
    "final_logit_softcap": ("final_logit_softcap", "final_logit_softcapping", "logit_softcap"),
    "final_logit_softcapping": ("final_logit_softcapping", "final_logit_softcap", "logit_softcap"),
    "logit_softcap": ("logit_softcap", "final_logit_softcap"),
    "embedding_scale": ("embedding_scale", "embed_scale", "scale_emb"),
    "embed_scale": ("embed_scale", "embedding_scale", "scale_emb"),
    "scale_emb": ("scale_emb", "embedding_scale", "embed_scale"),
    "scale_embedding": ("scale_embedding", "embedding_scale", "embed_scale"),
    "partial_rotary_factor": ("partial_rotary_factor",),
    "full_rope_theta": ("full_rope_theta", "full_attention_rope_theta"),
    "sliding_rope_theta": ("sliding_rope_theta", "sliding_attention_rope_theta"),
    "full_attention_rope_theta": ("full_attention_rope_theta", "full_rope_theta"),
    "sliding_attention_rope_theta": ("sliding_attention_rope_theta", "sliding_rope_theta"),
    "routed_scaling_factor": ("routed_scaling_factor", "route_scale", "moe_routed_scaling_factor"),
    "route_scale": ("route_scale", "routed_scaling_factor"),
    "scoring_func": ("scoring_func", "score_func", "router_score_func"),
    "score_func": ("score_func", "scoring_func", "router_score_func"),
    "norm_topk_prob": ("norm_topk_prob", "normalize_topk_prob"),
    "normalize_topk_prob": ("normalize_topk_prob", "norm_topk_prob"),
    "sliding_window": ("sliding_window",),
    "full_attention_interval": ("full_attention_interval", "global_attention_interval"),
    "global_attention_interval": ("global_attention_interval", "full_attention_interval"),
    "full_attention_offset": ("full_attention_offset", "global_attention_offset"),
    "global_attention_offset": ("global_attention_offset", "full_attention_offset"),
}

JUJU_HEADER_BYTES = 4096
JUJU_SECTION_ENTRY_BYTES = 96
JUJU_SECTION_TABLE_RESERVED_ENTRIES = 32
JUJU_SECTION_MODEL_META = 0x0001
JUJU_SECTION_SHARED_WEIGHTS = 0x0010
JUJU_SECTION_HOT_EXPERTS = 0x0011
JUJU_SECTION_WARM_EXPERTS = 0x0012
JUJU_SECTION_COLD_EXPERTS = 0x0013
JUJU_SECTION_LAYER_ORDER_INDEX = 0x0020
JUJU_SECTION_QKV_POLICY = 0x0021
JUJU_SECTION_VISION_ENCODER = 0x0030
JUJU_SECTION_VISION_PROJ = 0x0031
JUJU_SECTION_AUDIO_ENCODER = 0x0040
JUJU_SECTION_VIDEO_ENCODER = 0x0050
JUJU_SECTION_DOCUMENT_ENCODER = 0x0060
JUJU_MODALITY_TEXT = 0x01
JUJU_MODALITY_IMAGE = 0x02
JUJU_MODALITY_AUDIO = 0x04
JUJU_MODALITY_VIDEO = 0x08
JUJU_MODALITY_DOCUMENT = 0x10
JUJU_TENSOR_BUCKET_ORDER = (
    "shared_weights",
    "hot_experts",
    "warm_experts",
    "cold_experts",
    "vision_encoder",
    "vision_projector",
    "audio_encoder",
    "video_encoder",
    "document_encoder",
)
HF_INDIVIDUAL_FILE_LIMIT_BYTES = 50 * 1024 * 1024 * 1024
DEFAULT_JUJU_UPLOAD_FILE_LIMIT_BYTES = 45 * 1024 * 1024 * 1024
JUJU_SPLIT_METADATA_RESERVE_BYTES = 512 * 1024 * 1024
JUJU_FORMAT_CONTRACT_VERSION = 2
JUJU_BINARY_WIRE_ID = "JUJU_V1_HEADER4096_SECTION96_ABS_OFFSETS"
JUJU_TOKENIZER_FILES = [
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
    "added_tokens.json",
    "tokenizer.model",
    "sentencepiece.bpe.model",
    "tiktoken.model",
    "vocab.json",
    "merges.txt",
    "tokenization_kimi.py",
    "tool_declaration_ts.py",
    "generation_config.json",
    "config.json",
    "configuration_deepseek.py",
    "configuration_kimi_k25.py",
    "modeling_deepseek.py",
    "modeling_kimi_k25.py",
    "kimi_k25_processor.py",
    "kimi_k25_vision_processing.py",
    "media_utils.py",
    "processor_config.json",
    "preprocessor_config.json",
    "image_processor_config.json",
    "feature_extractor.json",
    "video_preprocessor_config.json",
    "audio_config.json",
]
JUJU_REQUIRED_TOKENIZER_FILES = ["config.json"]
JUJU_REQUIRED_TOKENIZER_ANY_OF = ["tokenizer.json", "tokenizer.model", "sentencepiece.bpe.model", "tiktoken.model", "vocab.json"]


def juju_artifact_names(source_name):
    stem = Path(source_name).stem
    return {
        "weights": f"{stem}.juju",
        "index": f"{stem}.juju.idx",
        "verify": f"{stem}.juju.verify.json",
    }


def juju_upload_file_limit_bytes():
    raw = str(os.environ.get("JUJU_MAX_UPLOAD_FILE_BYTES", "")).strip()
    if not raw:
        return DEFAULT_JUJU_UPLOAD_FILE_LIMIT_BYTES
    value = int(raw)
    if value <= 0:
        raise ValueError("JUJU_MAX_UPLOAD_FILE_BYTES must be positive")
    return value


def juju_target_tensor_splits():
    raw = str(os.environ.get("JUJU_TARGET_TENSOR_SPLITS", "")).strip()
    if not raw:
        return 0
    value = int(raw)
    if value < 0:
        raise ValueError("JUJU_TARGET_TENSOR_SPLITS must be non-negative")
    return value


def juju_split_source_name(source_name, split_index, split_count):
    path = Path(source_name)
    suffix = path.suffix or ".gguf"
    return f"{path.stem}.split{int(split_index):02d}-of-{int(split_count):02d}{suffix}"


def align_up(value, alignment=4096):
    rem = int(value) % int(alignment)
    return int(value) if rem == 0 else int(value) + int(alignment) - rem


def fixed_bytes(value, size):
    raw = str(value or "").encode("utf-8")[:size]
    return raw + (b"\x00" * (size - len(raw)))


def read_exact(handle, size):
    data = handle.read(size)
    if len(data) != size:
        raise EOFError("unexpected EOF while reading GGUF")
    return data


def read_u32(handle):
    return struct.unpack("<I", read_exact(handle, 4))[0]


def read_u64(handle):
    return struct.unpack("<Q", read_exact(handle, 8))[0]


def read_string(handle):
    size = read_u64(handle)
    return read_exact(handle, size).decode("utf-8")


def skip_value(handle, value_type):
    if value_type == GGUF_TYPE_STRING:
        handle.seek(read_u64(handle), 1)
        return
    if value_type == GGUF_TYPE_ARRAY:
        elem_type = read_u32(handle)
        count = read_u64(handle)
        if elem_type == GGUF_TYPE_STRING:
            for _ in range(count):
                handle.seek(read_u64(handle), 1)
            return
        elem_size = GGUF_TYPE_SIZE.get(elem_type)
        if elem_size is None:
            raise ValueError(f"unsupported GGUF array element type: {elem_type}")
        handle.seek(elem_size * count, 1)
        return
    size = GGUF_TYPE_SIZE.get(value_type)
    if size is None:
        raise ValueError(f"unsupported GGUF value type: {value_type}")
    handle.seek(size, 1)


def read_gguf_scalar_value(handle, value_type):
    if value_type == GGUF_TYPE_UINT8:
        return struct.unpack("<B", read_exact(handle, 1))[0]
    if value_type == GGUF_TYPE_INT8:
        return struct.unpack("<b", read_exact(handle, 1))[0]
    if value_type == GGUF_TYPE_UINT16:
        return struct.unpack("<H", read_exact(handle, 2))[0]
    if value_type == GGUF_TYPE_INT16:
        return struct.unpack("<h", read_exact(handle, 2))[0]
    if value_type == GGUF_TYPE_UINT32:
        return read_u32(handle)
    if value_type == GGUF_TYPE_INT32:
        return struct.unpack("<i", read_exact(handle, 4))[0]
    if value_type == GGUF_TYPE_FLOAT32:
        return struct.unpack("<f", read_exact(handle, 4))[0]
    if value_type == GGUF_TYPE_BOOL:
        return bool(struct.unpack("<?", read_exact(handle, 1))[0])
    if value_type == GGUF_TYPE_STRING:
        return read_string(handle)
    if value_type == GGUF_TYPE_UINT64:
        return read_u64(handle)
    if value_type == GGUF_TYPE_INT64:
        return struct.unpack("<q", read_exact(handle, 8))[0]
    if value_type == GGUF_TYPE_FLOAT64:
        return struct.unpack("<d", read_exact(handle, 8))[0]
    return None


def should_capture_gguf_runtime_kv(key):
    lower = str(key or "").lower()
    if lower in GGUF_RUNTIME_KV_EXACT_KEYS:
        return True
    return any(hint in lower for hint in GGUF_RUNTIME_KV_KEY_HINTS)


def gguf_runtime_aliases_for_key(key):
    lower = str(key or "").lower()
    aliases = []
    for suffix, mapped in GGUF_RUNTIME_KV_ALIAS_MAP.items():
        if lower == suffix or lower.endswith("." + suffix) or lower.endswith(suffix):
            aliases.extend(mapped)
    return aliases


def gguf_tensor_row_bytes(tensor_type, cols):
    t = u32(tensor_type)
    cols = int(cols or 0)
    if cols <= 0:
        return 0
    block32 = (cols + 31) // 32
    block256 = (cols + 255) // 256
    if t == 0:
        return cols * 4
    if t in {1, 30}:
        return cols * 2
    if t == 2:
        return block32 * 18
    if t == 3:
        return block32 * 20
    if t == 6:
        return block32 * 22
    if t == 7:
        return block32 * 24
    if t == 8:
        return block32 * 34
    if t == 9:
        return block32 * 36
    if t == 10:
        return block256 * 84
    if t == 11:
        return block256 * 110
    if t == 12:
        return block256 * 144
    if t == 13:
        return block256 * 176
    if t == 14:
        return block256 * 210
    if t == 15:
        return block256 * 292
    if t == 16:
        return block256 * 66
    if t == 17:
        return block256 * 74
    if t == 18:
        return block256 * 98
    if t == 19:
        return block256 * 50
    if t == 20:
        return block32 * 18
    if t == 21:
        return block256 * 110
    if t == 22:
        return block256 * 82
    if t == 23:
        return block256 * 136
    if t == 24:
        return cols
    if t == 25:
        return cols * 2
    if t == 26:
        return cols * 4
    if t == 27:
        return cols * 8
    if t == 28:
        return cols * 8
    if t == 29:
        return block256 * 56
    if t == 39:
        return block32 * 17
    return 0


def gguf_tensor_exact_bytes(tensor_type, shape):
    shape = [int(v or 0) for v in (shape or [])]
    if not shape or shape[0] <= 0:
        return 0
    cols = shape[0] if len(shape) >= 2 else 1
    rows = 1
    if len(shape) >= 2:
        for dim in shape[1:]:
            rows *= max(0, dim)
    else:
        rows = shape[0]
    row_bytes = gguf_tensor_row_bytes(tensor_type, cols)
    return row_bytes * rows if row_bytes and rows > 0 else 0


def file_size(session, url, token=None):
    headers = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    resp = session.head(url, allow_redirects=True, headers=headers, timeout=120)
    if resp.ok and resp.headers.get("Content-Length"):
        return int(resp.headers["Content-Length"])
    headers["Range"] = "bytes=0-0"
    resp = session.get(url, headers=headers, stream=True, timeout=120)
    resp.close()
    cr = resp.headers.get("Content-Range", "")
    if "/" in cr:
        return int(cr.rsplit("/", 1)[1])
    raise RuntimeError(f"could not determine remote file size: {url}")


def fetch_range(session, url, start, end=None, token=None, stream=True):
    headers = {"Range": f"bytes={int(start)}-" + (str(int(end)) if end is not None else "")}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    resp = session.get(url, headers=headers, stream=stream, timeout=120)
    resp.raise_for_status()
    return resp


def parse_gguf_directory(prefix, total_bytes):
    handle = io.BytesIO(prefix)
    if read_exact(handle, 4) != b"GGUF":
        raise ValueError("source is not GGUF")
    version = read_u32(handle)
    tensor_count = read_u64(handle)
    kv_count = read_u64(handle)
    alignment = 32
    gguf_kv = {}
    gguf_kv_aliases = {}
    for _ in range(kv_count):
        key = read_string(handle)
        value_type = read_u32(handle)
        if key == "general.alignment" and value_type == GGUF_TYPE_UINT32:
            value = read_u32(handle)
            alignment = value
            gguf_kv[key] = value
            gguf_kv_aliases["alignment"] = value
        elif should_capture_gguf_runtime_kv(key) and value_type != GGUF_TYPE_ARRAY:
            value = read_gguf_scalar_value(handle, value_type)
            if value is None:
                skip_value(handle, value_type)
                continue
            gguf_kv[key] = value
            gguf_kv_aliases[key] = value
            for alias in gguf_runtime_aliases_for_key(key):
                gguf_kv_aliases.setdefault(alias, value)
        else:
            skip_value(handle, value_type)
    tensors = []
    for _ in range(tensor_count):
        name = read_string(handle)
        dims = read_u32(handle)
        shape = [read_u64(handle) for _ in range(dims)]
        tensor_type = read_u32(handle)
        rel_offset = read_u64(handle)
        tensors.append({
            "name": name,
            "dims": dims,
            "shape": shape,
            "type": tensor_type,
            "relative_offset": rel_offset,
        })
    data_start = align_up(handle.tell(), alignment)
    if data_start > len(prefix):
        raise EOFError(f"GGUF tensor table needs {data_start} bytes, got {len(prefix)}")
    order = sorted(range(len(tensors)), key=lambda i: tensors[i]["relative_offset"])
    for pos, idx in enumerate(order):
        cur = tensors[idx]["relative_offset"]
        nxt = tensors[order[pos + 1]]["relative_offset"] if pos + 1 < len(order) else total_bytes - data_start
        storage_bytes = max(0, nxt - cur)
        exact_bytes = gguf_tensor_exact_bytes(tensors[idx]["type"], tensors[idx]["shape"])
        tensors[idx]["source_offset"] = data_start + cur
        tensors[idx]["source_storage_bytes"] = storage_bytes
        tensors[idx]["exact_bytes"] = exact_bytes
        tensors[idx]["bytes"] = exact_bytes if exact_bytes and exact_bytes <= storage_bytes else storage_bytes
        tensors[idx]["source_padding_bytes"] = max(0, storage_bytes - tensors[idx]["bytes"])
        tensors[idx]["bucket"] = tensor_bucket(tensors[idx]["name"])
    return {
        "version": version,
        "tensor_count": tensor_count,
        "kv_count": kv_count,
        "alignment": alignment,
        "data_start": data_start,
        "gguf_kv": gguf_kv,
        "gguf_runtime": gguf_kv_aliases,
        "gguf_kv_floats": {
            k: v for k, v in gguf_kv_aliases.items()
            if isinstance(v, float)
        },
        "tensors": tensors,
    }


def read_remote_directory(session, url, token=None, initial_range_bytes=8 * 1024 * 1024):
    total = file_size(session, url, token=token)
    size = min(initial_range_bytes, total)
    while True:
        resp = fetch_range(session, url, 0, size - 1, token=token, stream=False)
        prefix = resp.content
        resp.close()
        try:
            return parse_gguf_directory(prefix, total), total
        except EOFError:
            if size >= total:
                raise
            if size >= 256 * 1024 * 1024:
                raise RuntimeError(f"Max fetch exceeded: {size}")
            size = min(size * 2, total)


def juju_estimated_tensor_payload_bytes(tensors, is_first_shard=True):
    total = JUJU_SPLIT_METADATA_RESERVE_BYTES if is_first_shard else 32 * 1024 * 1024
    for tensor in tensors:
        total = align_up(total, 4096)
        total += int(tensor.get("bytes") or 0)
    return total


def juju_aligned_tensor_bytes(tensor):
    return align_up(int(tensor.get("bytes") or 0), 4096)


def juju_groups_fit_upload_limits(groups, payload_limit_first, payload_limit_sub):
    for idx, group in enumerate(groups):
        limit = payload_limit_first if idx == 0 else payload_limit_sub
        if sum(juju_aligned_tensor_bytes(tensor) for tensor in group) > limit:
            return False
    return True


def balance_juju_tensor_groups(tensors, split_count):
    tensors = list(tensors)
    split_count = min(max(1, int(split_count)), len(tensors))
    if split_count <= 1:
        return [tensors]
    total = sum(juju_aligned_tensor_bytes(tensor) for tensor in tensors)
    target = max(1, (total + split_count - 1) // split_count)
    groups = []
    current = []
    current_bytes = 0
    remaining_groups = split_count
    for idx, tensor in enumerate(tensors):
        tensor_bytes = juju_aligned_tensor_bytes(tensor)
        remaining_tensors = len(tensors) - idx
        if (
            current
            and len(groups) < split_count - 1
            and current_bytes + tensor_bytes > target
            and remaining_tensors >= remaining_groups
        ):
            groups.append(current)
            current = []
            current_bytes = 0
            remaining_groups -= 1
        current.append(tensor)
        current_bytes += tensor_bytes
    if current:
        groups.append(current)

    while len(groups) < split_count:
        split_at = max(range(len(groups)), key=lambda i: len(groups[i]))
        group = groups[split_at]
        if len(group) <= 1:
            break
        half_bytes = sum(juju_aligned_tensor_bytes(tensor) for tensor in group) // 2
        running = 0
        cut = 1
        for idx, tensor in enumerate(group[:-1], start=1):
            running += juju_aligned_tensor_bytes(tensor)
            cut = idx
            if running >= half_bytes:
                break
        groups[split_at:split_at + 1] = [group[:cut], group[cut:]]
    return groups


def plan_juju_tensor_splits(directory, max_file_bytes=None):
    limit = int(max_file_bytes or juju_upload_file_limit_bytes())
    if limit >= HF_INDIVIDUAL_FILE_LIMIT_BYTES:
        limit = HF_INDIVIDUAL_FILE_LIMIT_BYTES - (256 * 1024 * 1024)
    payload_limit_first = limit - JUJU_SPLIT_METADATA_RESERVE_BYTES
    payload_limit_sub = limit - 32 * 1024 * 1024
    if payload_limit_first <= 0:
        raise ValueError("JUJU upload file limit is too small after metadata reserve")

    tensors = [
        tensor for tensor in sorted(directory["tensors"], key=lambda item: (int(item.get("source_offset") or 0), str(item.get("name") or "")))
        if int(tensor.get("bytes") or 0) > 0
    ]
    if not tensors:
        return [{
            "enabled": False,
            "split_index": 1,
            "split_count": 1,
            "tensor_names": [],
            "tensor_bytes": 0,
            "max_file_bytes": limit,
        }]

    groups = []
    current = []
    current_bytes = 0
    for tensor in tensors:
        tensor_bytes = int(tensor["bytes"])
        aligned_tensor_bytes = juju_aligned_tensor_bytes(tensor)
        current_limit = payload_limit_first if len(groups) == 0 else payload_limit_sub
        if aligned_tensor_bytes > current_limit:
            raise RuntimeError(
                f"single tensor exceeds upload-safe JUJU split limit: {tensor['name']} "
                f"bytes={tensor_bytes} limit={current_limit}"
            )
        if current and current_bytes + aligned_tensor_bytes > current_limit:
            groups.append(current)
            current = []
            current_bytes = 0
        current.append(tensor)
        current_bytes += aligned_tensor_bytes
    if current:
        groups.append(current)

    split_strategy = "limit_tensor_groups"
    target_split_count = juju_target_tensor_splits()
    if target_split_count > 1 and len(groups) > 1:
        balanced_groups = balance_juju_tensor_groups(tensors, max(target_split_count, len(groups)))
        if juju_groups_fit_upload_limits(balanced_groups, payload_limit_first, payload_limit_sub):
            groups = balanced_groups
            split_strategy = "balanced_tensor_groups"

    split_count = len(groups)
    planned = []
    for idx, group in enumerate(groups, start=1):
        planned.append({
            "enabled": split_count > 1,
            "split_index": idx,
            "split_count": split_count,
            "split_strategy": split_strategy,
            "target_split_count": target_split_count,
            "tensor_names": [tensor["name"] for tensor in group],
            "tensor_bytes": sum(int(tensor["bytes"]) for tensor in group),
            "estimated_file_bytes": juju_estimated_tensor_payload_bytes(group, is_first_shard=(idx == 1)),
            "max_file_bytes": limit,
        })
    return planned


def tensor_bucket(name):
    lower = str(name).lower()
    if any(k in lower for k in (
        "mm_projector",
        "multi_modal_projector",
        "vision_projector",
        "image_projector",
    )):
        return "vision_projector"
    if any(k in lower for k in (
        "vision_model.",
        "vit.",
        "visual_encoder.",
        "image_encoder.",
        "moonvit.",
        "siglip.",
    )):
        return "vision_encoder"
    if any(k in lower for k in (
        "audio_model.",
        "whisper.",
        "audio_encoder.",
    )):
        return "audio_encoder"
    if any(k in lower for k in (
        "video_model.",
        "video_encoder.",
        "temporal_encoder.",
        "timesformer.",
    )):
        return "video_encoder"
    if any(k in lower for k in (
        "document_encoder.",
        "pdf_encoder.",
        "ocr_encoder.",
    )):
        return "document_encoder"
    if "shared_expert" in lower or "shared.expert" in lower:
        return "shared_weights"
    if "attn" in lower or "attention" in lower:
        return "shared_weights"
    expert_name = (
        "_exps" in lower or
        "ffn_gate_up_exps" in lower or
        "ffn_gate_exps" in lower or
        "ffn_up_exps" in lower or
        "ffn_down_exps" in lower or
        re.search(r"(?:^|[.])(experts?|expert)[.]", lower) is not None
    )
    if expert_name:
        return "cold_experts"
    return "shared_weights"


def section_type_for_bucket(bucket):
    if bucket == "shared_weights":
        return JUJU_SECTION_SHARED_WEIGHTS
    if bucket == "hot_experts":
        return JUJU_SECTION_HOT_EXPERTS
    if bucket == "warm_experts":
        return JUJU_SECTION_WARM_EXPERTS
    if bucket == "vision_encoder":
        return JUJU_SECTION_VISION_ENCODER
    if bucket == "vision_projector":
        return JUJU_SECTION_VISION_PROJ
    if bucket == "audio_encoder":
        return JUJU_SECTION_AUDIO_ENCODER
    if bucket == "video_encoder":
        return JUJU_SECTION_VIDEO_ENCODER
    if bucket == "document_encoder":
        return JUJU_SECTION_DOCUMENT_ENCODER
    return JUJU_SECTION_COLD_EXPERTS


def _juju_json_dict(value):
    if isinstance(value, dict):
        return dict(value)
    if isinstance(value, (bytes, bytearray)):
        value = value.decode("utf-8", errors="ignore")
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return {}
        try:
            parsed = json.loads(text)
        except Exception:
            return {}
        return dict(parsed) if isinstance(parsed, dict) else {}
    return {}


def juju_metadata_files_from_contract(contract):
    files = {}
    for key in (
        "hf_metadata_files",
        "runtime_metadata_files",
        "runtime_assets",
        "runtime_asset_files",
        "tokenizer_assets",
    ):
        raw = contract.get(key)
        if isinstance(raw, dict):
            for name, value in raw.items():
                files[str(name)] = value
        elif isinstance(raw, list):
            for item in raw:
                if isinstance(item, dict):
                    name = item.get("path") or item.get("name") or item.get("file")
                    if name:
                        files[str(name)] = item.get("content", item.get("json", item.get("raw", item)))
                elif isinstance(item, str):
                    files[item] = {}
    return files


def detect_vision_config(hf_metadata_files):
    cfg = {}
    raw = (
        hf_metadata_files.get("image_processor_config.json") or
        hf_metadata_files.get("preprocessor_config.json") or
        hf_metadata_files.get("processor_config.json") or
        hf_metadata_files.get("tokenizer/image_processor_config.json")
    )
    data = _juju_json_dict(raw)
    if data:
        cfg["image_token_id"] = u32(data.get("image_token_id") or data.get("image_token_index"))
        cfg["patch_size"] = u32(data.get("patch_size") or data.get("vision_patch_size") or 14)
        cfg["encoder_hidden_dim"] = u32(data.get("hidden_size") or data.get("encoder_hidden_dim"))
    return cfg


def juju_modality_flags_from_buckets(buckets, hf_metadata_files=None):
    flags = JUJU_MODALITY_TEXT
    bucket_set = set(buckets or [])
    if bucket_set.intersection({"vision_encoder", "vision_projector"}):
        flags |= JUJU_MODALITY_IMAGE
    if "audio_encoder" in bucket_set:
        flags |= JUJU_MODALITY_AUDIO
    if "video_encoder" in bucket_set:
        flags |= JUJU_MODALITY_VIDEO
    if "document_encoder" in bucket_set:
        flags |= JUJU_MODALITY_DOCUMENT
    for name in (hf_metadata_files or {}).keys():
        lower = str(name).lower()
        if "image_processor_config.json" in lower or "preprocessor_config.json" in lower or "processor_config.json" in lower:
            flags |= JUJU_MODALITY_IMAGE
        if "audio_config.json" in lower:
            flags |= JUJU_MODALITY_AUDIO
        if "video_preprocessor_config.json" in lower or "video_config.json" in lower:
            flags |= JUJU_MODALITY_VIDEO
        if "document" in lower or "pdf" in lower or "ocr" in lower:
            flags |= JUJU_MODALITY_DOCUMENT
    return flags


def juju_modality_metadata(contract, tensors):
    metadata_files = juju_metadata_files_from_contract(contract)
    buckets = [t.get("bucket", "") for t in tensors or []]
    flags = juju_modality_flags_from_buckets(buckets, metadata_files)
    return {
        "modality_flags": flags,
        "modalities": {
            "text": bool(flags & JUJU_MODALITY_TEXT),
            "image": bool(flags & JUJU_MODALITY_IMAGE),
            "audio": bool(flags & JUJU_MODALITY_AUDIO),
            "video": bool(flags & JUJU_MODALITY_VIDEO),
            "document": bool(flags & JUJU_MODALITY_DOCUMENT),
        },
        "vision_config": detect_vision_config(metadata_files),
        "section_types": {
            "vision_encoder": JUJU_SECTION_VISION_ENCODER,
            "vision_projector": JUJU_SECTION_VISION_PROJ,
            "audio_encoder": JUJU_SECTION_AUDIO_ENCODER,
            "video_encoder": JUJU_SECTION_VIDEO_ENCODER,
            "document_encoder": JUJU_SECTION_DOCUMENT_ENCODER,
        },
        "section_policy": "write_only_nonempty_modality_sections",
    }


def pack_section(entry):
    payload = struct.pack(
        "<IIQQQII4B16s32s",
        int(entry["type"]),
        int(entry.get("flags", 0)),
        int(entry["offset"]),
        int(entry["size"]),
        int(entry.get("uncompressed_size", entry["size"])),
        int(entry.get("sequential_block_size", 4096)),
        int(entry.get("random_block_size", 4096)),
        int(entry.get("compression", 0)),
        int(entry.get("prefetch_distance", 0)),
        int(entry.get("mmap_friendly", 1)),
        0,
        bytes.fromhex(entry.get("sha256", "0" * 64))[:16].ljust(16, b"\x00"),
        fixed_bytes(entry.get("name", ""), 32),
    )
    if len(payload) > JUJU_SECTION_ENTRY_BYTES:
        raise ValueError(f"JUJU section entry is too large: {len(payload)} > {JUJU_SECTION_ENTRY_BYTES}")
    return payload + (b"\x00" * (JUJU_SECTION_ENTRY_BYTES - len(payload)))


def write_padding(out, alignment=4096):
    pad = align_up(out.tell(), alignment) - out.tell()
    if pad:
        out.write(b"\x00" * pad)


def stream_range(session, url, start, size, out, token, digest, chunk_size=16 * 1024 * 1024):
    if size <= 0:
        return
    resp = fetch_range(session, url, start, start + size - 1, token=token, stream=True)
    written = 0
    try:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            if not chunk:
                continue
            out.write(chunk)
            digest.update(chunk)
            written += len(chunk)
    finally:
        resp.close()
    if written != size:
        raise EOFError(f"short tensor range read: expected {size}, got {written}")


def u32(value):
    try:
        if value is None:
            return 0
        return max(0, min(int(value), 0xFFFFFFFF))
    except Exception:
        return 0


def contract_value(contract, *keys, default=None):
    for key in keys:
        cur = contract
        ok = True
        for part in str(key).split("."):
            if isinstance(cur, dict) and part in cur:
                cur = cur[part]
            else:
                ok = False
                break
        if ok and cur is not None:
            return cur
    return default


def mb_from_bytes(value):
    try:
        n = int(value or 0)
    except Exception:
        return 0
    if n <= 0:
        return 0
    return max(1, n // (1024 * 1024))


def juju_arch_type(contract, source_name):
    text = " ".join(
        str(x or "")
        for x in (
            contract.get("architecture"),
            contract_value(contract, "arch_meta.architecture"),
            contract.get("model_id"),
            contract.get("model_name"),
            source_name,
        )
    ).lower()
    if "glm" in text:
        return 1
    if "gemma" in text:
        return 2
    if "qwen" in text:
        return 3
    if "llama" in text:
        return 4
    if "mistral" in text:
        return 5
    return 0


def juju_weight_bits(contract):
    return u32(contract_value(
        contract,
        "source_weight_bits",
        "weight_bits",
        "weight_quant_schema.bits",
        "weight_quant_schema.weight_bits",
        default=0,
    ))


def juju_weight_encoding(contract):
    explicit = contract_value(contract, "source_weight_encoding", "weight_encoding", "weight_quant_schema.encoding", default=0)
    if explicit:
        return u32(explicit)
    family = str(contract_value(
        contract,
        "source_weight_quant_family",
        "weight_quant_family",
        "weight_quant_schema.family",
        default="",
    ) or "").lower()
    if "iq2_xxs" in family:
        return 19
    if "iq3_xxs" in family:
        return 20
    if "iq1_s" in family:
        return 32
    if "iq1_m" in family:
        return 33
    if "iq2_xs" in family:
        return 29
    if "iq2_s" in family:
        return 30
    if "iq3_s" in family:
        return 31
    if "iq4_nl" in family:
        return 27
    if "iq4_xs" in family:
        return 28
    if "bf16" in family or "bfloat16" in family:
        return 21
    if "q5_0" in family:
        return 24
    if "q5_1" in family:
        return 12
    if "q4_0" in family:
        return 22
    if "q4_1" in family:
        return 23
    if "q8_1" in family:
        return 25
    if "q8_0" in family:
        return 13
    if "iq2" in family or "ud-iq2" in family:
        return 9
    if "iq3" in family:
        return 10
    if "iq4" in family:
        return 11
    if "mxfp4" in family:
        return 4
    if "nvfp4" in family or "fp4" in family:
        return 3
    if "q8" in family:
        return 8
    if "q4" in family:
        return 7
    if "q3" in family:
        return 6
    if "q2" in family:
        return 5
    return 0


def weight_encoding_from_gguf_type(tensor_type, contract=None):
    t = u32(tensor_type)
    mapping = {
        0: 2,
        1: 1,
        2: 22,
        3: 23,
        6: 24,
        7: 12,
        8: 13,
        9: 25,
        10: 15,
        11: 16,
        12: 17,
        13: 14,
        14: 18,
        15: 34,
        16: 19,
        18: 20,
        19: 32,
        17: 29,
        22: 30,
        29: 33,
        21: 31,
        20: 27,
        23: 28,
        30: 21,
        39: 4,
    }
    enc = mapping.get(t, 0)
    if enc:
        return enc
    return juju_weight_encoding(contract or {})


def gguf_type_name(tensor_type):
    names = {
        0: "F32",
        1: "F16",
        2: "Q4_0",
        3: "Q4_1",
        6: "Q5_0",
        7: "Q5_1",
        8: "Q8_0",
        9: "Q8_1",
        10: "Q2_K",
        11: "Q3_K",
        12: "Q4_K",
        13: "Q5_K",
        14: "Q6_K",
        15: "Q8_K",
        16: "IQ2_XXS",
        17: "IQ2_XS",
        18: "IQ3_XXS",
        19: "IQ1_S",
        20: "IQ4_NL",
        21: "IQ3_S",
        22: "IQ2_S",
        23: "IQ4_XS",
        24: "I8",
        25: "I16",
        26: "I32",
        27: "I64",
        28: "F64",
        29: "IQ1_M",
        30: "BF16",
        34: "TQ1_0",
        35: "TQ2_0",
        39: "MXFP4",
    }
    return names.get(u32(tensor_type), f"GGUF_TYPE_{u32(tensor_type)}")


def quant_family_from_gguf_type(tensor_type, contract=None):
    t = u32(tensor_type)
    if t in {0, 1, 24, 25, 26, 27, 28, 30}:
        return "raw_scalar_or_integer"
    if t in {2, 3, 6, 7, 8, 9}:
        return "legacy_ggml_quant"
    if t in {10, 11, 12, 13, 14, 15}:
        return "k_quant"
    if t in {16, 17, 18, 19, 20, 21, 22, 23, 29}:
        return "importance_quant"
    if t in {34, 35}:
        return "ternary_quant"
    if t == 39:
        return "mxfp4"
    explicit = contract_value(contract or {}, "source_weight_quant_family", "weight_quant_family", "weight_quant_schema.family", default="")
    if explicit:
        return str(explicit)
    return "unknown_preserved_source_type"


def kernel_key_from_gguf_type(tensor_type, contract=None):
    return f"{quant_family_from_gguf_type(tensor_type, contract)}:{gguf_type_name(tensor_type)}"


def juju_qkv_policy(contract):
    if contract.get("qkv_cache_schema") or contract.get("qkv_policy_contract"):
        return 1
    if contract_value(contract, "qkv_packed_cache_required", default=False):
        return 1
    return 0


def juju_format_extension_contract(contract):
    return {
        "contract_version": JUJU_FORMAT_CONTRACT_VERSION,
        "binary_wire_id": JUJU_BINARY_WIRE_ID,
        "binary_wire_frozen": True,
        "header_bytes": JUJU_HEADER_BYTES,
        "section_entry_bytes": JUJU_SECTION_ENTRY_BYTES,
        "section_table_reserved_entries": JUJU_SECTION_TABLE_RESERVED_ENTRIES,
        "section_table_offset": JUJU_HEADER_BYTES,
        "offset_unit": "absolute_file_byte_offset",
        "length_unit": "exact_payload_byte_length",
        "alignment_bytes": 4096,
        "endianness": "little",
        "tensor_payload_layout": "source_bytes_preserved_without_requantization",
        "json_sections_are_extension_surface": True,
        "additive_json_fields_allowed": True,
        "unknown_json_field_policy": "engine_ignore_if_not_required",
        "unknown_required_feature_policy": "fail_closed",
        "engine_update_without_repack": [
            "new_cpu_quant_kernel",
            "new_gpu_quant_kernel",
            "new_attention_kernel",
            "new_qkv_cache_backend",
            "new_prefetch_scheduler",
            "new_residency_policy",
            "new_graph_ir_executor",
            "new_adapter_runtime",
            "new_tokenizer_loader_policy",
            "new_sampler",
            "new_validation_probe",
            "new_multimodal_executor",
        ],
        "repack_required_only_for": [
            "model_weights_changed",
            "tensor_payload_bytes_changed",
            "tensor_order_or_offsets_changed",
            "new_required_tokenizer_asset_contents",
            "new_section_compression_requiring_reencoded_payload",
            "file_checksum_or_payload_corruption",
        ],
        "reserved_extension_namespaces": [
            "MODEL_META.format_extension_contract",
            "MODEL_META.kernel_registry_contract",
            "MODEL_META.adapter_registry_contract",
            "MODEL_META.validation_contract",
            "TENSOR_INDEX.tensors[].extension",
            "TENSOR_INDEX.tensors[].kernel_contract",
            "GRAPH_IR.runtime_policy",
            "GRAPH_IR.execution_plan",
            "GRAPH_IR.priority_tables",
            "GRAPH_IR.performance_research_slots",
            "MODEL_META.multimodal_contract",
            "MODEL_META.modality_flags",
        ],
        "reserved_section_types": {
            "vision_encoder": JUJU_SECTION_VISION_ENCODER,
            "vision_projector": JUJU_SECTION_VISION_PROJ,
            "audio_encoder": JUJU_SECTION_AUDIO_ENCODER,
            "video_encoder": JUJU_SECTION_VIDEO_ENCODER,
            "document_encoder": JUJU_SECTION_DOCUMENT_ENCODER,
        },
        "modality_flags": {
            "text": JUJU_MODALITY_TEXT,
            "image": JUJU_MODALITY_IMAGE,
            "audio": JUJU_MODALITY_AUDIO,
            "video": JUJU_MODALITY_VIDEO,
            "document": JUJU_MODALITY_DOCUMENT,
        },
        "compatibility_rule": "binary_header_and_section_table_remain_stable; add new behavior through JSON sections and engine code",
    }


def juju_kernel_registry_contract(contract):
    return {
        "selection_key_order": ["weight_encoding", "gguf_type", "gguf_type_name", "quant_family", "kernel_key"],
        "required_behavior": "engine_must_execute_or_fail_closed_never_silent_zero",
        "source_type_preserved": True,
        "supported_source_families_declared": [
            "raw_fp32",
            "raw_fp16",
            "bf16",
            "legacy_q4_q5_q8",
            "k_quant_q2_q3_q4_q5_q6_q8",
            "iq1_iq2_iq3_iq4",
            "ternary_tq",
            "mxfp4",
            "vendor_dynamic_quant",
        ],
        "row_layout_rule": "preserve_source_quant_block_layout_until_kernel_decode",
        "mixed_quant_per_tensor_allowed": True,
        "per_tensor_weight_encoding_required": True,
        "per_tensor_source_type_required": True,
        "contract_weight_encoding": juju_weight_encoding(contract),
        "contract_weight_bits": juju_weight_bits(contract),
    }


def juju_tokenizer_contract():
    return {
        "tokenizer_files": list(JUJU_TOKENIZER_FILES),
        "required_files": list(JUJU_REQUIRED_TOKENIZER_FILES),
        "required_any_of": list(JUJU_REQUIRED_TOKENIZER_ANY_OF),
        "target_subdirs": ["", "tokenizer"],
        "chat_template_source": "tokenizer_config_or_model_card",
        "missing_tokenizer_behavior": "fail_text_api_if_required_tokenizer_missing",
        "input_ids_api_allowed_without_tokenizer": True,
    }


def juju_adapter_registry_contract():
    return {
        "adapter_metadata_slots_reserved": True,
        "supported_adapter_classes": [
            "lora",
            "qlora",
            "dora",
            "ia3",
            "prompt_tuning",
            "prefix_tuning",
            "runtime_delta_weight",
            "router_override",
            "expert_bias_or_scale",
        ],
        "storage_policy": "adapters_external_or_json_declared; base_tensor_payload_not_repacked",
        "merge_policy": "engine_runtime_merge_or_sidecar_cache",
        "compatibility_key_fields": ["target_tensor", "rank", "alpha", "dtype", "quant_compatibility"],
    }


def juju_validation_contract():
    return {
        "load_time_checks": [
            "magic",
            "header_size",
            "section_table_size",
            "section_offsets",
            "tensor_offsets",
            "tensor_lengths",
            "tensor_sha256_if_present",
            "tokenizer_required_any_of",
            "kernel_support_for_all_required_tensors",
        ],
        "correctness_checks": [
            "no_required_tensor_silent_zero",
            "dense_mlp_not_classified_as_expert_stream",
            "all_required_graph_ops_bound",
            "logits_finite",
            "ppl_probe_supported",
        ],
        "failure_policy": "fail_closed_with_actionable_error",
    }


def juju_research_offload_contract():
    return {
        "goal": "maximize_moe_offload_without_repacking_base_model",
        "phase_aware_execution": {
            "prefill": {
                "expected_pattern": "many_experts_active",
                "required_slots": [
                    "separate_prefill_scheduler",
                    "non_moe_compute_overlap_window",
                    "bounded_expert_residency",
                    "bulk_prefetch_stream",
                    "token_reordering_optional",
                ],
            },
            "decode": {
                "expected_pattern": "few_experts_active_per_token",
                "required_slots": [
                    "layer_level_expert_predictor",
                    "cross_layer_gate_predictor",
                    "activation_trace_predictor",
                    "semantic_prompt_hint_predictor",
                    "speculative_expert_prefetch",
                    "cache_hit_rate_feedback",
                ],
            },
        },
        "expert_cache_policy_inputs": [
            "layer_id",
            "expert_id",
            "token_position",
            "sequence_id",
            "router_topk",
            "router_score",
            "router_entropy",
            "local_routing_consistency",
            "previous_layer_experts",
            "previous_token_experts",
            "expert_hit_rate",
            "expert_load_latency_us",
            "expert_compute_latency_us",
            "pcie_bandwidth_bytes_per_s",
            "disk_bandwidth_bytes_per_s",
            "gpu_free_bytes",
            "cpu_free_bytes",
            "pinned_staging_bytes",
        ],
        "bottleneck_breaker_slots": {
            "critical_path_io": [
                "prefetch_before_router_consumer",
                "overlap_dma_with_attention_or_dense_compute",
                "two_stream_copy_compute_pipeline",
                "bounded_retry_queue",
                "io_priority_by_graph_role",
            ],
            "expert_cache_miss": [
                "proactive_cache",
                "activation_aware_cache",
                "fine_grained_expert_segments",
                "semantic_hint_cache_seed",
                "local_routing_consistency_score",
            ],
            "gpu_memory_pressure": [
                "hot_shared_residency",
                "expert_lru_or_score_eviction",
                "prefill_decode_different_budget",
                "qkv_page_eviction",
                "compressed_kv_cache",
            ],
            "token_scheduling": [
                "dynamic_token_ordering",
                "expert_batching",
                "router_entropy_adaptive_topk",
                "decode_microbatch_policy",
            ],
            "storage_path": [
                "mmap_tensor_spans",
                "direct_io_alignment",
                "pinned_cpu_stage",
                "async_read_ahead",
                "checksum_after_stream",
            ],
        },
        "research_method_slots": {
            "moe_infinity": ["sequence_level_activation_trace", "activation_aware_prefetch", "activation_aware_cache"],
            "promoe": ["proactive_expert_cache", "intermediate_result_prediction"],
            "fmoe": ["fine_grained_expert_offload", "expert_selection_patterns", "semantic_prompt_hints"],
            "duoserve_moe": ["prefill_decode_split", "dual_phase_expert_prefetch", "cache_scheduling"],
            "expertflow": ["adaptive_expert_scheduling", "memory_coordination", "dynamic_token_ordering"],
            "fate_cross_layer_gate": ["cross_layer_expert_prediction", "prediction_confidence"],
            "local_routing_consistency": ["routing_locality_metric", "offload_suitability_score"],
            "moe_speq": ["speculative_quantized_decode", "proactive_expert_prefetch"],
            "flexgen": ["gpu_cpu_disk_placement", "offload_policy_search", "weight_and_cache_compression"],
        },
        "kv_cache_research_slots": {
            "paged_attention": ["page_size_tokens", "block_table", "fragmentation_control"],
            "vattention": ["virtual_memory_backed_kv", "demand_paging_policy"],
            "infinigen": ["essential_kv_prefetch", "cpu_kv_pool", "counter_based_eviction"],
            "kivi": ["key_per_channel_quant", "value_per_token_quant", "residual_window"],
            "kvquant": ["sub_4bit_kv_quant", "outlier_aware_quant"],
            "turboquant": ["polarquant", "qjl_residual_correction", "online_vector_quantization"],
        },
        "metrics_required": [
            "expert_hit_rate",
            "expert_miss_latency_us",
            "tokens_per_second",
            "time_to_first_token_ms",
            "inter_token_latency_ms",
            "gpu_resident_expert_bytes",
            "cpu_resident_expert_bytes",
            "disk_read_bytes",
            "pcie_copy_bytes",
            "kv_cache_bytes",
            "prefetch_waste_ratio",
            "prediction_accuracy",
            "logits_finite_rate",
            "ppl_probe",
        ],
    }


def juju_contract_metadata(contract, source_name, source_repo_id):
    arch = dict(contract.get("arch_meta") or {})
    qkv = dict(contract.get("qkv_cache_schema") or {})
    model_id = contract.get("model_id") or contract.get("source_model_id") or source_repo_id
    model_name = contract.get("model_name") or model_id or Path(source_name).stem
    out = {
        "format_version": 1,
        "backend_neutral": True,
        "model_id": model_id,
        "model_name": model_name,
        "architecture": contract.get("architecture") or arch.get("architecture") or "",
        "source_weight_bits": juju_weight_bits(contract),
        "source_weight_encoding": juju_weight_encoding(contract),
        "source_weight_quant_family": contract.get("source_weight_quant_family") or contract.get("weight_quant_family") or contract_value(contract, "weight_quant_schema.family", default=""),
        "source_weight_kernel_family": contract.get("source_weight_kernel_family") or contract.get("weight_kernel_family") or contract_value(contract, "weight_quant_schema.kernel_family", default=""),
        "source_weight_block_size": u32(contract_value(contract, "source_weight_block_size", "weight_block_size", "weight_quant_schema.block_size", default=0)),
        "qkv_packed_cache_required": bool(qkv),
        "persistent_plain_kv_cache_allowed": not bool(qkv),
        "final_model_structure_contract": contract.get("final_model_structure_contract", {}),
        "pipeline_budget_contract": contract.get("pipeline_budget_contract", {}),
        "execution_path_contract": contract.get("execution_path_contract", {}),
        "expert_segmentation_contract": contract.get("expert_segmentation_contract", {}),
        "chunk_io_contract": contract.get("chunk_io_contract", {}),
        "universal_tier_contract": contract.get("universal_tier_contract", {}),
        "qkv_policy_contract": contract.get("qkv_policy_contract", qkv),
        "format_extension_contract": juju_format_extension_contract(contract),
        "kernel_registry_contract": juju_kernel_registry_contract(contract),
        "tokenizer_contract": juju_tokenizer_contract(),
        "adapter_registry_contract": juju_adapter_registry_contract(),
        "validation_contract": juju_validation_contract(),
        "research_offload_contract": juju_research_offload_contract(),
        "runtime_adapter_contract": {
            "weight_source": "juju_tensor_index",
            "offset_unit": "absolute_file_byte_offset",
            "row_layout": "source_quant_row_layout_preserved",
            "section_entry_bytes": JUJU_SECTION_ENTRY_BYTES,
            "section_table_reserved_entries": JUJU_SECTION_TABLE_RESERVED_ENTRIES,
            "header_bytes": JUJU_HEADER_BYTES,
            "alignment": 4096,
            "fail_closed": True,
        },
        "performance_contract": {
            "startup_prefetch_roles": ["shared_core", "router", "attention", "norm"],
            "streaming_roles": ["expert", "dense_ffn"],
            "direct_io_alignment": 4096,
            "mmap_friendly_sections": True,
            "split_large_uploads": True,
            "tokenizer_required_at_repo_root": True,
        },
    }
    for src, dst in (
        ("k_bits", "k_bits"),
        ("v_bits", "v_bits"),
        ("group_size", "group_size"),
        ("page_size_tokens", "page_size_tokens"),
        ("sink_tokens", "sink_tokens"),
        ("rotation_seed", "rotation_seed"),
        ("qjl_seed", "qjl_seed"),
        ("enable_qjl", "enable_qjl"),
        ("enable_rotation", "enable_rotation"),
    ):
        if src in qkv:
            out[dst] = qkv[src]
    return out


def make_header(contract, source_name, file_size_value, sections, section_sizes, index_checksum=0, modality_flags=JUJU_MODALITY_TEXT):
    header = bytearray(JUJU_HEADER_BYTES)
    header[0:8] = b"JUJU\x00\x01\x00\x00"
    struct.pack_into("<I", header, 8, 1)
    struct.pack_into("<I", header, 12, 0)
    struct.pack_into("<Q", header, 16, int(time.time()))
    struct.pack_into("<Q", header, 24, int(file_size_value))
    struct.pack_into("<Q", header, 32, JUJU_HEADER_BYTES)
    struct.pack_into("<Q", header, 40, len(sections) * JUJU_SECTION_ENTRY_BYTES)
    struct.pack_into("<Q", header, 48, section_sizes.get(JUJU_SECTION_SHARED_WEIGHTS, 0))
    struct.pack_into("<Q", header, 56, section_sizes.get(JUJU_SECTION_HOT_EXPERTS, 0))
    struct.pack_into("<Q", header, 64, section_sizes.get(JUJU_SECTION_WARM_EXPERTS, 0))
    struct.pack_into("<Q", header, 72, section_sizes.get(JUJU_SECTION_COLD_EXPERTS, 0))
    struct.pack_into("<Q", header, 80, int(index_checksum or 0) & 0xFFFFFFFFFFFFFFFF)
    model_name = contract.get("model_name") or contract.get("model_id") or Path(source_name).stem
    header[88:152] = fixed_bytes(model_name, 64)
    arch = contract.get("arch_meta") or {}
    struct.pack_into("<I", header, 152, len(sections))
    struct.pack_into("<I", header, 156, JUJU_HEADER_BYTES)
    struct.pack_into("<I", header, 160, u32(arch.get("n_layers") or arch.get("num_hidden_layers")))
    struct.pack_into("<I", header, 164, u32(arch.get("experts_per_moe_layer") or arch.get("n_experts")))
    struct.pack_into("<I", header, 168, u32(arch.get("routed_experts_per_token") or arch.get("top_k")))
    struct.pack_into("<I", header, 172, u32(arch.get("hidden_dim") or arch.get("hidden_size")))
    struct.pack_into("<I", header, 176, u32(arch.get("expert_intermediate_dim") or arch.get("expert_intermediate_size")))
    struct.pack_into("<I", header, 180, juju_weight_bits(contract))
    struct.pack_into("<I", header, 184, juju_arch_type(contract, source_name))
    struct.pack_into("<I", header, 188, u32(contract_value(contract, "segment_policy", "expert_segmentation_contract.segment_policy", default=2)))
    struct.pack_into("<I", header, 192, juju_qkv_policy(contract))
    struct.pack_into("<I", header, 196, u32(contract_value(contract, "preferred_segment_bytes", "chunk_io_contract.preferred_segment_bytes", default=4096)))
    struct.pack_into("<I", header, 200, u32(contract_value(contract, "max_segments_per_expert", "expert_segmentation_contract.max_segments_per_expert", default=8)))
    struct.pack_into("<I", header, 204, u32(contract_value(contract, "recommended_vram_mb", default=mb_from_bytes(contract_value(contract, "recommended_vram_bytes", "pipeline_budget_contract.recommended_vram_bytes", default=0)))))
    struct.pack_into("<I", header, 208, u32(contract_value(contract, "recommended_ram_mb", default=mb_from_bytes(contract_value(contract, "recommended_ram_bytes", "pipeline_budget_contract.recommended_ram_bytes", default=0)))))
    struct.pack_into("<I", header, 212, u32(modality_flags))
    return bytes(header)


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def json_section_bytes(payload):
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")


def _juju_layer_id_from_name(name):
    match = __import__("re").search(r"(?:^|[.])blk\.(\d+)\.", str(name or ""))
    return int(match.group(1)) if match else None


def _juju_first_tensor(tensors, *names):
    wanted = {str(x).lower() for x in names if x}
    for tensor in tensors:
        name = str(tensor.get("name") or "")
        if name.lower() in wanted:
            return name
    return ""


def _juju_tensors_by_prefix(tensors, prefix):
    prefix = str(prefix or "").lower()
    return [
        str(t.get("name") or "")
        for t in tensors
        if str(t.get("name") or "").lower().startswith(prefix)
    ]


def _juju_tensor_shape_map(tensors):
    return {
        str(t.get("name") or ""): list(t.get("shape") or [])
        for t in tensors
        if t.get("name")
    }


def tensor_runtime_priority(name, bucket, size):
    lower = str(name or "").lower()
    bucket = str(bucket or "")
    role = "weight"
    priority = 50
    prefetch = 50
    residency = "SLOW_MEM"
    prefetch_class = "stream"
    if bucket in {"vision_encoder", "vision_projector", "audio_encoder", "video_encoder", "document_encoder"}:
        role = bucket
        priority = 45
        prefetch = 20
        residency = "SLOW_MEM"
        prefetch_class = "stream"
    elif lower in {"token_embd.weight", "output.weight", "output_norm.weight", "rope_freqs.weight"}:
        role = "shared_core"
        priority = 100
        prefetch = 100
        residency = "FAST_MEM"
        prefetch_class = "startup_hot"
    elif ".attn_" in lower or ".attn" in lower:
        role = "attention"
        priority = 90
        prefetch = 90
        residency = "FAST_MEM"
        prefetch_class = "layer_hot"
    elif "ffn_gate_inp" in lower or "router" in lower:
        role = "router"
        priority = 95
        prefetch = 95
        residency = "FAST_MEM"
        prefetch_class = "router_hot"
    elif "_exps" in lower or "expert" in lower:
        role = "expert"
        priority = 65
        prefetch = 80
        residency = "SLOW_MEM"
        prefetch_class = "expert_stream"
    elif ".ffn_" in lower:
        role = "dense_ffn"
        priority = 75
        prefetch = 75
        residency = "FAST_MEM" if bucket == "shared_weights" else "SLOW_MEM"
        prefetch_class = "layer_warm"
    elif "norm" in lower:
        role = "norm"
        priority = 85
        prefetch = 85
        residency = "FAST_MEM"
        prefetch_class = "layer_hot"
    if int(size or 0) > 512 * 1024 * 1024 and residency == "FAST_MEM" and role != "attention":
        residency = "FAST_MEM_STREAMABLE"
    return {
        "graph_role": role,
        "runtime_priority": priority,
        "prefetch_priority": prefetch,
        "prefetch_class": prefetch_class,
        "residency_hint": residency,
    }


def infer_juju_graph_family(contract, tensors):
    text = json.dumps(contract, ensure_ascii=False).lower()
    names = {str(t.get("name") or "").lower() for t in tensors}
    if "gemma" in text:
        return "gemma_moe"
    if any("ffn_gate_up_exps.weight" in n for n in names):
        return "combined_gate_up_moe"
    if "qwen" in text:
        return "qwen"
    if "llama" in text or "mistral" in text:
        return "llama"
    if "glm" in text:
        return "glm"
    return "generic_transformer"


def first_present(*values):
    for value in values:
        if value is None:
            continue
        if isinstance(value, str) and value == "":
            continue
        return value
    return None


def juju_runtime_arch_metadata(contract, directory=None):
    arch = dict(contract.get("arch_meta") or {})
    runtime = dict((directory or {}).get("gguf_runtime") or {})
    out = dict(runtime)

    fields = {
        "declared_architecture": first_present(contract.get("architecture"), arch.get("architecture"), runtime.get("declared_architecture"), runtime.get("architecture")),
        "model_id": first_present(contract.get("model_id"), contract.get("model_name"), runtime.get("model_id")),
        "model_name": first_present(contract.get("model_name"), runtime.get("model_name")),
        "num_hidden_layers": first_present(arch.get("n_layers"), arch.get("num_hidden_layers"), runtime.get("num_hidden_layers"), runtime.get("n_layers")),
        "hidden_size": first_present(arch.get("hidden_dim"), arch.get("hidden_size"), runtime.get("hidden_size"), runtime.get("hidden_dim")),
        "vocab_size": first_present(arch.get("vocab_size"), runtime.get("vocab_size")),
        "head_dim": first_present(arch.get("head_dim"), runtime.get("head_dim"), runtime.get("key_length")),
        "value_head_dim": first_present(arch.get("value_head_dim"), arch.get("v_head_dim"), runtime.get("value_head_dim"), runtime.get("v_head_dim")),
        "global_head_dim": first_present(arch.get("global_head_dim"), runtime.get("global_head_dim")),
        "num_attention_heads": first_present(arch.get("n_heads"), arch.get("num_attention_heads"), runtime.get("num_attention_heads"), runtime.get("n_heads")),
        "num_key_value_heads": first_present(arch.get("n_kv_heads"), arch.get("num_key_value_heads"), runtime.get("num_key_value_heads"), runtime.get("n_kv_heads")),
        "num_global_key_value_heads": first_present(arch.get("num_global_key_value_heads"), runtime.get("num_global_key_value_heads")),
        "kv_lora_rank": first_present(arch.get("kv_lora_rank"), runtime.get("kv_lora_rank")),
        "q_lora_rank": first_present(arch.get("q_lora_rank"), runtime.get("q_lora_rank")),
        "qk_nope_head_dim": first_present(arch.get("qk_nope_head_dim"), runtime.get("qk_nope_head_dim")),
        "qk_rope_head_dim": first_present(arch.get("qk_rope_head_dim"), runtime.get("qk_rope_head_dim")),
        "experts_per_moe_layer": first_present(arch.get("experts_per_moe_layer"), arch.get("n_experts"), runtime.get("experts_per_moe_layer"), runtime.get("n_experts")),
        "routed_experts_per_token": first_present(arch.get("routed_experts_per_token"), arch.get("top_k"), runtime.get("routed_experts_per_token"), runtime.get("top_k")),
        "expert_intermediate_size": first_present(arch.get("expert_intermediate_size"), arch.get("expert_intermediate_dim"), runtime.get("expert_intermediate_size"), runtime.get("expert_intermediate_dim")),
        "rms_norm_eps": first_present(arch.get("rms_norm_eps"), arch.get("norm_eps"), runtime.get("rms_norm_eps"), runtime.get("norm_eps")),
        "norm_eps": first_present(arch.get("norm_eps"), arch.get("rms_norm_eps"), runtime.get("norm_eps"), runtime.get("rms_norm_eps")),
        "rope_theta": first_present(arch.get("rope_theta"), runtime.get("rope_theta"), runtime.get("theta")),
        "theta": first_present(arch.get("rope_theta"), runtime.get("theta"), runtime.get("rope_theta")),
        "sliding_window": first_present(arch.get("sliding_window"), runtime.get("sliding_window")),
        "embedding_scale": first_present(arch.get("embedding_scale"), arch.get("scale_emb"), runtime.get("embedding_scale"), runtime.get("scale_emb")),
        "scale_emb": first_present(arch.get("scale_emb"), arch.get("embedding_scale"), runtime.get("scale_emb"), runtime.get("embedding_scale")),
        "final_logit_softcap": first_present(arch.get("final_logit_softcap"), arch.get("final_logit_softcapping"), runtime.get("final_logit_softcap"), runtime.get("final_logit_softcapping"), runtime.get("logit_softcap")),
        "final_logit_softcapping": first_present(arch.get("final_logit_softcapping"), arch.get("final_logit_softcap"), runtime.get("final_logit_softcapping"), runtime.get("final_logit_softcap"), runtime.get("logit_softcap")),
        "partial_rotary_factor": first_present(arch.get("partial_rotary_factor"), runtime.get("partial_rotary_factor")),
        "full_rope_theta": first_present(arch.get("full_rope_theta"), arch.get("full_attention_rope_theta"), runtime.get("full_rope_theta"), runtime.get("full_attention_rope_theta")),
        "sliding_rope_theta": first_present(arch.get("sliding_rope_theta"), arch.get("sliding_attention_rope_theta"), runtime.get("sliding_rope_theta"), runtime.get("sliding_attention_rope_theta")),
        "routed_scaling_factor": first_present(arch.get("routed_scaling_factor"), arch.get("route_scale"), runtime.get("routed_scaling_factor"), runtime.get("route_scale")),
        "norm_topk_prob": first_present(arch.get("norm_topk_prob"), arch.get("normalize_topk_prob"), runtime.get("norm_topk_prob"), runtime.get("normalize_topk_prob")),
        "scoring_func": first_present(arch.get("scoring_func"), arch.get("score_func"), runtime.get("scoring_func"), runtime.get("score_func")),
    }
    for key, value in fields.items():
        if value is not None:
            out[key] = value
    return out


def build_layer_graph_ir(layer, tensors):
    prefix = f"blk.{layer}."
    names = set(_juju_tensors_by_prefix(tensors, prefix))

    def bind(*suffixes):
        out = []
        for suffix in suffixes:
            name = prefix + suffix
            if name in names:
                out.append(name)
        return out

    moe_weights = bind("ffn_gate_up_exps.weight", "ffn_gate_exps.weight", "ffn_up_exps.weight", "ffn_down_exps.weight")
    dense_weights = bind("ffn_gate.weight", "ffn_up.weight", "ffn_down.weight", "mlp.gate_proj.weight", "mlp.up_proj.weight", "mlp.down_proj.weight")

    ops = [
        {"op": "rms_norm", "name": "attention_input_norm", "inputs": ["hidden"], "weights": bind("attn_norm.weight", "input_layernorm.weight"), "required": False},
        {"op": "linear", "name": "q_projection", "inputs": ["attention_norm"], "weights": bind("attn_q.weight", "attention.wq.weight"), "output": "q", "required": False},
        {"op": "linear", "name": "k_projection", "inputs": ["attention_norm"], "weights": bind("attn_k.weight", "attention.wk.weight"), "output": "k", "required": False},
        {"op": "linear", "name": "v_projection", "inputs": ["attention_norm"], "weights": bind("attn_v.weight", "attention.wv.weight"), "output": "v", "required": False},
        {"op": "rms_norm", "name": "q_norm", "inputs": ["q"], "weights": bind("attn_q_norm.weight"), "required": False},
        {"op": "rms_norm", "name": "k_norm", "inputs": ["k"], "weights": bind("attn_k_norm.weight"), "required": False},
        {"op": "rope", "name": "rotary_embedding", "inputs": ["q", "k"], "weights": bind("rope_freqs.weight"), "required": False},
        {"op": "attention", "name": "attention", "inputs": ["q", "k", "v"], "kv_cache": "quantized_qkv_cache", "required": False},
        {"op": "linear", "name": "attention_output", "inputs": ["attention"], "weights": bind("attn_output.weight", "attention.wo.weight"), "output": "attention_out", "required": False},
        {"op": "residual", "name": "attention_residual", "inputs": ["hidden", "attention_out"], "required": True},
        {"op": "rms_norm", "name": "ffn_norm", "inputs": ["hidden"], "weights": bind("ffn_norm.weight", "post_attention_norm.weight", "pre_ffw_norm.weight"), "required": False},
        {"op": "hidden_snapshot", "name": "fate_gate_input_snapshot", "inputs": ["ffn_norm"], "target": "engine_state.gate_input_snapshots[layer]", "required": False},
        {"op": "linear", "name": "moe_router", "inputs": ["ffn_norm"], "weights": bind("ffn_gate_inp.weight", "router.weight"), "scale": bind("ffn_gate_inp.scale"), "output": "expert_scores", "required": False},
        {"op": "topk", "name": "expert_select", "inputs": ["expert_scores"], "config_key": "adaptive_seq_topk_entropy", "required": False},
        {"op": "moe_expert_mlp", "name": "moe_experts", "inputs": ["ffn_norm", "selected_experts"], "weights": moe_weights, "required": bool(moe_weights)},
        {"op": "dense_mlp", "name": "dense_ffn_fallback", "inputs": ["ffn_norm"], "weights": dense_weights, "required": bool(dense_weights)},
        {"op": "residual", "name": "ffn_residual", "inputs": ["hidden", "ffn_out"], "required": True},
        {"op": "scale", "name": "layer_output_scale", "inputs": ["hidden"], "weights": bind("layer_output_scale.weight"), "required": False},
    ]
    return {
        "layer": int(layer),
        "tensor_prefix": prefix,
        "available_tensors": sorted(names),
        "ops": ops,
    }


def build_juju_graph_ir(*, contract, tensor_records, sections, source_name, source_path, source_repo_id, weight_file, index_file, directory=None):
    arch = dict(contract.get("arch_meta") or {})
    runtime_arch = juju_runtime_arch_metadata(contract, directory)
    shape_map = _juju_tensor_shape_map(tensor_records)
    layers = sorted({
        layer for layer in (_juju_layer_id_from_name(t.get("name")) for t in tensor_records)
        if layer is not None
    })
    token_embd = _juju_first_tensor(tensor_records, "token_embd.weight")
    lm_head = _juju_first_tensor(tensor_records, "output.weight") or token_embd
    output_norm = _juju_first_tensor(tensor_records, "output_norm.weight", "norm.weight")
    priority_rules = [
        {"match": "token_embd.weight|output.weight|output_norm.weight|rope_freqs.weight", "priority": 100, "residency": "FAST_MEM", "prefetch": "startup_hot"},
        {"match": "attention/norm/router tensors", "priority": 85, "residency": "FAST_MEM", "prefetch": "layer_hot"},
        {"match": "expert tensors", "priority": 65, "residency": "SLOW_MEM", "prefetch": "expert_stream"},
        {"match": "large FAST_MEM tensors", "priority": "keep but streamable", "residency": "FAST_MEM_STREAMABLE", "prefetch": "bounded"},
    ]
    role_counts = {}
    bucket_counts = {}
    encoding_counts = {}
    for rec in tensor_records:
        role_counts[str(rec.get("graph_role") or "unknown")] = role_counts.get(str(rec.get("graph_role") or "unknown"), 0) + 1
        bucket_counts[str(rec.get("bucket") or "unknown")] = bucket_counts.get(str(rec.get("bucket") or "unknown"), 0) + 1
        encoding_key = str(rec.get("weight_encoding") or rec.get("gguf_type") or 0)
        encoding_counts[encoding_key] = encoding_counts.get(encoding_key, 0) + 1
    return {
        "format": "JUJU_GRAPH_IR_V1",
        "schema_version": 1,
        "required": True,
        "fail_closed_if_missing": True,
        "graph_id": f"{source_repo_id}:{source_path}:{weight_file}",
        "source": {
            "repo_id": source_repo_id,
            "source_path": source_path,
            "source_name": source_name,
            "weight_file": weight_file,
            "index_file": index_file,
        },
        "format_extension_contract": juju_format_extension_contract(contract),
        "kernel_registry_contract": juju_kernel_registry_contract(contract),
        "adapter_registry_contract": juju_adapter_registry_contract(),
        "validation_contract": juju_validation_contract(),
        "research_offload_contract": juju_research_offload_contract(),
        "architecture": {
            "family": infer_juju_graph_family(contract, tensor_records),
            "declared_architecture": contract.get("architecture") or arch.get("architecture") or "",
            "model_id": contract.get("model_id") or contract.get("model_name") or "",
            "num_hidden_layers": arch.get("n_layers") or arch.get("num_hidden_layers") or len(layers),
            "hidden_size": arch.get("hidden_dim") or arch.get("hidden_size") or (shape_map.get(token_embd, [0])[0] if token_embd else 0),
            "vocab_size": arch.get("vocab_size") or (shape_map.get(token_embd, [0, 0])[1] if token_embd and len(shape_map.get(token_embd, [])) > 1 else 0),
            "head_dim": arch.get("head_dim"),
            "num_attention_heads": arch.get("n_heads") or arch.get("num_attention_heads"),
            "num_key_value_heads": arch.get("n_kv_heads") or arch.get("num_key_value_heads"),
            "experts_per_moe_layer": arch.get("experts_per_moe_layer") or arch.get("n_experts"),
            "routed_experts_per_token": arch.get("routed_experts_per_token") or arch.get("top_k"),
            "norm_eps": arch.get("norm_eps") or arch.get("rms_norm_eps"),
            "rope": {
                "type": arch.get("rope_type") or arch.get("rope_scaling_type") or "runtime_from_source_metadata",
                "theta": arch.get("rope_theta"),
                "scaling": arch.get("rope_scaling"),
            },
            **runtime_arch,
        },
        "tokenizer_contract": juju_tokenizer_contract(),
        "quantization": {
            "weight": contract.get("weight_quant_schema", {}),
            "qkv_cache": contract.get("qkv_cache_schema", {}),
            "source_weight_bits": juju_weight_bits(contract),
            "source_weight_encoding": juju_weight_encoding(contract),
            "source_weight_family": contract.get("source_weight_quant_family"),
            "source_weight_kernel_family": contract.get("source_weight_kernel_family"),
            "tensor_weight_encoding_counts": encoding_counts,
            "kernel_requirement": "engine_must_support_every_tensor_weight_encoding_or_fail_closed",
        },
        "tensor_bindings": {
            "token_embedding": token_embd,
            "lm_head": lm_head,
            "lm_head_tied_to_token_embedding": bool(lm_head and token_embd and lm_head == token_embd),
            "final_norm": output_norm,
            "rope_freqs": _juju_first_tensor(tensor_records, "rope_freqs.weight"),
            "layer_tensor_prefix": "blk.{layer}.",
            "shape_map": shape_map,
        },
        "tensor_index_contract": {
            "binary_schema_version": 3,
            "offsets": "absolute_file_offsets",
            "lengths": "exact_payload_bytes",
            "alignment_bytes": 4096,
            "weight_encoding_field": "weight_encoding",
            "gguf_type_field": "gguf_type",
            "binary_required_fields": ["gguf_type", "weight_encoding"],
            "gguf_type_name_field": "gguf_type_name",
            "quant_family_field": "quant_family",
            "kernel_key_field": "kernel_key",
            "kernel_contract_field": "kernel_contract",
            "row_layout_field": "row_layout",
            "role_counts": role_counts,
            "bucket_counts": bucket_counts,
            "sections_embedded_in_header_table": True,
            "paired_idx_required": True,
            "external_adapter_required": False,
        },
        "ops": [
            {"op": "input_tokens", "output": "token_ids", "required": True},
            {"op": "embedding_lookup", "weights": [token_embd], "output": "hidden", "required": bool(token_embd)},
            {"op": "for_each_layer", "layers": [int(x) for x in layers], "body_ref": "layers"},
            {"op": "rms_norm", "name": "final_norm", "weights": [output_norm] if output_norm else [], "required": bool(output_norm)},
            {"op": "lm_head", "weights": [lm_head] if lm_head else [], "tied_to_embedding": bool(lm_head == token_embd), "required": bool(lm_head)},
            {"op": "sampler", "inputs": ["logits"], "required": True},
        ],
        "layers": [build_layer_graph_ir(layer, tensor_records) for layer in layers],
        "runtime_policy": {
            "execution": "graph_ir_executor_required",
            "unknown_op": "fail_closed",
            "unknown_tensor": "fail_closed_for_required_optional_skip",
            "kv_cache": "quantized_qkv_cache_required" if contract.get("qkv_cache_schema") else "runtime_default",
            "model_load": "eager_validate_header_sections_idx_tokenizer_and_kernel_support",
            "weight_decode": "juju_weight_encoding_and_gguf_type_exact_dispatch",
            "residency_policy": contract.get("residency_policy", {}),
            "prefetch_plan_hints": contract.get("prefetch_plan_hints", {}),
            "kernel_hints": contract.get("kernel_hints", {}),
            "execution_hints": contract.get("execution_hints", {}),
            "memory_management_hints": contract.get("memory_management_hints", {}),
            "adapter_contract": {
                "dense_mlp_uses_shared_path": True,
                "expert_mlp_uses_streaming_path": True,
                "required_quant_decode": "all_tensor_index_weight_encodings",
                "prefetch_must_respect_graph_role": True,
                "tokenizer_assets_must_exist": True,
            },
            "hard_defaults": {
                "vram_double_admission_guard": {"enabled": True, "counter": "pending_vram_reservation"},
                "macos_available_ram": {"count_inactive_pages": False},
                "metal_unified_memory_budget_percent": 60,
                "router_seq_topk_entropy": {
                    "enabled": True,
                    "base_k": 8,
                    "low_entropy_threshold": 0.30,
                    "low_entropy_k_multiplier": 0.50,
                    "high_entropy_threshold": 0.70,
                    "high_entropy_max_k": 12,
                },
                "duoserve_prefill_decode_split": {
                    "enabled": True,
                    "disable_lookahead_during_prefill": True,
                    "prefill_phase_source": "engine.generation_phase == PREFILL",
                },
                "expertflow_adaptive_prediction_depth": {
                    "enabled": True,
                    "entropy_over": 0.70,
                    "max_prefetch_depth": 1,
                },
                "fate_hidden_snapshot": {
                    "enabled": True,
                    "capture": "gate_input_before_router",
                    "storage": "engine_state.gate_input_snapshots[layer]",
                },
            },
        },
        "execution_plan": {
            "input": ["tokenizer_assets", "token_ids"],
            "prefill": ["embedding", "layer_loop", "kv_write", "logits"],
            "decode": ["next_token_embedding", "layer_loop", "kv_read_write", "logits"],
            "offload_units": ["shared_tensor", "expert_tensor", "dense_ffn_tensor", "qkv_page", "vision_tensor", "audio_tensor", "video_tensor", "document_tensor"],
            "io_policy": {
                "read_unit": "tensor_span",
                "alignment": 4096,
                "mmap": True,
                "stream_large_slow_mem_tensors": True,
                "protect_fastmem_roles": ["shared_core", "attention", "router", "norm"],
            },
        },
        "priority_tables": {
            "tensor_priority_fields": ["runtime_priority", "prefetch_priority", "prefetch_class", "residency_hint", "graph_role"],
            "rules": priority_rules,
            "section_priorities": [
                {
                    "name": s.get("name", ""),
                    "type": s.get("type", 0),
                    "prefetch_distance": s.get("prefetch_distance", 0),
                    "mmap_friendly": s.get("mmap_friendly", 0),
                    "priority": 100 if s.get("name") == "SHARED_WEIGHTS" else 70,
                }
                for s in sections
            ],
        },
        "moe_offload_policy": {
            "enabled": True,
            "router_first": True,
            "expert_tensor_patterns": [
                "blk.{layer}.ffn_gate_up_exps.weight",
                "blk.{layer}.ffn_gate_exps.weight",
                "blk.{layer}.ffn_up_exps.weight",
                "blk.{layer}.ffn_down_exps.weight",
            ],
            "dense_fallback_patterns": [
                "blk.{layer}.ffn_gate.weight",
                "blk.{layer}.ffn_up.weight",
                "blk.{layer}.ffn_down.weight",
            ],
            "tier_names": ["COMPUTE_MEM", "FAST_MEM", "SLOW_MEM"],
            "bucket_mapping": {
                "shared_weights": "FAST_MEM",
                "hot_experts": "FAST_MEM",
                "warm_experts": "FAST_MEM_STREAMABLE",
                "cold_experts": "SLOW_MEM",
                "vision_encoder": "SLOW_MEM",
                "vision_projector": "SLOW_MEM",
                "audio_encoder": "SLOW_MEM",
                "video_encoder": "SLOW_MEM",
                "document_encoder": "SLOW_MEM",
            },
            "admission_priority": {
                "router": 100,
                "attention": 95,
                "norm": 90,
                "dense_ffn": 80,
                "expert": 70,
                "cold_expert": 60,
            },
            "prefetch": {
                "unit": "layer_expert_tensor",
                "trigger": "router_topk_and_previous_layer_coactivation",
                "lookahead_layers": [1, 2],
                "coactivation_table": "mutable_runtime_index",
                "fallback_when_no_history": "router_scores",
                "bounded_by": ["ram_budget", "vram_budget", "staging_slots", "io_queue_depth"],
                "priority_field": "prefetch_priority",
                "score_filter": {
                    "enabled": True,
                    "vram_percentile": 0.70,
                    "ram_percentile": 0.50,
                    "drop_below_percentile": 0.50,
                },
            },
            "eviction": {
                "policy": "least_stale_predicted_next_use_max_heap",
                "protect_roles": ["router", "attention", "norm", "token_embedding", "lm_head"],
                "demote_order": ["video_encoder", "document_encoder", "audio_encoder", "vision_encoder", "cold_experts", "warm_experts", "large_shared_streamable"],
                "primary_key": "predicted_next_use_epoch",
                "tie_breakers": ["hot_score", "last_touch_epoch"],
                "rollback_required": True,
            },
            "cpu_hot_miss": {
                "enabled": True,
                "condition": "expert_in_ram_and_decode_batch_le_4",
                "decision": "cpu_ms < pcie_transfer_ms",
                "cpu_gflops_default": 100.0,
            },
            "score_aware_precision": {
                "enabled": True,
                "low_score_load_bits": 4,
                "fallback_when_nvfp4_unavailable": "int4",
                "requires_decoder": "engine_int4_or_scale4_decode",
            },
            "streaming": {
                "expert_streaming_required": True,
                "direct_io_alignment": 4096,
                "split_combined_gate_up": True,
                "allow_partial_expert_segments": True,
            },
            "telemetry": {
                "record_expert_hits": True,
                "record_layer_latency": True,
                "record_io_wait": True,
                "record_cache_promotions": True,
                "record_cache_evictions": True,
                "record_coactivation": True,
            },
            "source_contracts": {
                "residency_policy": contract.get("residency_policy", {}),
                "prefetch_plan_hints": contract.get("prefetch_plan_hints", {}),
                "dynamic_swap_triggers": contract.get("dynamic_swap_triggers", {}),
                "activation_stats_schema": contract.get("activation_stats_schema", {}),
                "sparsity_schema": contract.get("sparsity_schema", {}),
                "runtime_monitoring_hints": contract.get("runtime_monitoring_hints", {}),
            },
        },
        "validation": {
            "require_all_required_ops_bound": True,
            "require_tensor_shape_match": True,
            "require_quant_schema_match": True,
            "require_qkv_policy_match": bool(contract.get("qkv_cache_schema")),
            "allow_optional_ops_missing": True,
            "tensor_count": len(tensor_records),
            "section_count": len(sections),
        },
        "compatibility": {
            "min_engine_graph_ir_version": 1,
            "endianness": "little",
            "alignment": 4096,
            "portable_backend_terms_only": True,
        },
    }


def build_juju_shard_plan_from_hf_url(
    *,
    source_url,
    source_name,
    source_path,
    contract,
    token=None,
    source_repo_id="",
    chunk_size=16 * 1024 * 1024,
    artifact_source_name=None,
    tensor_name_allowlist=None,
    split_info=None,
):
    artifact_source_name = artifact_source_name or source_name
    artifact_names = juju_artifact_names(artifact_source_name)
    fixed_segments = []
    source_segments = []
    sections = []
    section_sizes = {}
    tensor_records = []

    def add_fixed(offset, data):
        if data:
            fixed_segments.append({"offset": int(offset), "size": len(data), "data": data})

    def add_json_section_at(pos, section_type, name, payload):
        raw = json_section_bytes(payload)
        pos = align_up(pos, 4096)
        offset = pos
        add_fixed(offset, raw)
        entry = {
            "type": section_type,
            "name": name,
            "offset": offset,
            "size": len(raw),
            "sha256": hashlib.sha256(raw).hexdigest(),
            "mmap_friendly": 0,
        }
        sections.append(entry)
        section_sizes[section_type] = section_sizes.get(section_type, 0) + len(raw)
        return pos + len(raw)

    with requests.Session() as session:
        directory, total_bytes = read_remote_directory(session, source_url, token=token)
        allowset = set(tensor_name_allowlist or [])
        if allowset:
            known = {tensor["name"] for tensor in directory["tensors"]}
            missing = sorted(allowset - known)
            if missing:
                raise RuntimeError(f"JUJU split references missing tensors: {missing[:8]}")
        active_tensors = [
            tensor for tensor in directory["tensors"]
            if int(tensor.get("bytes") or 0) > 0 and (not allowset or tensor["name"] in allowset)
        ]
        split_meta = split_info or {
            "enabled": False,
            "split_index": 1,
            "split_count": 1,
            "parent_source_name": source_name,
            "artifact_source_name": artifact_source_name,
            "tensor_count": len(active_tensors),
        }
        modality_meta = juju_modality_metadata(contract, active_tensors)
        modality_flags = int(modality_meta["modality_flags"])
        pos = JUJU_HEADER_BYTES
        table_offset = pos
        pos += JUJU_SECTION_TABLE_RESERVED_ENTRIES * JUJU_SECTION_ENTRY_BYTES
        pos = align_up(pos, 4096)

        meta = {
            "format": "JUJU_SHARDED_CONTAINER_V1",
            "source_format": "GGUF",
            "source_role": "conversion_source_only",
            "source_repo_id": source_repo_id,
            "source_path": source_path,
            "source_name": source_name,
            "artifact_source_name": artifact_source_name,
            "weight_file": artifact_names["weights"],
            "index_file": artifact_names["index"],
            "tensor_payload_layout": "4kb_aligned_tensor_sections",
            "artifact_name_policy": "preserve_original_shard_stem_change_extension_only",
            "graph_ir_format": "JUJU_GRAPH_IR_V1",
            "graph_ir_required": True,
            "split": split_meta,
            "gguf_directory": {
                "version": directory["version"],
                "tensor_count": directory["tensor_count"],
                "emitted_tensor_count": len(active_tensors),
                "kv_count": directory["kv_count"],
                "alignment": directory["alignment"],
                "data_start": directory["data_start"],
                "source_bytes": total_bytes,
                "gguf_kv": directory.get("gguf_kv", {}),
                "gguf_runtime": directory.get("gguf_runtime", {}),
                "gguf_kv_floats": directory.get("gguf_kv_floats", {}),
            },
            "contract": contract,
            "modality_flags": modality_flags,
            "multimodal_contract": modality_meta,
            **juju_contract_metadata(contract, source_name, source_repo_id),
        }
        pos = add_json_section_at(pos, JUJU_SECTION_MODEL_META, "MODEL_META", meta)
        qkv_schema = contract.get("qkv_cache_schema")
        if qkv_schema:
            pos = add_json_section_at(pos, JUJU_SECTION_QKV_POLICY, "QKV_POLICY", qkv_schema)

        for bucket in JUJU_TENSOR_BUCKET_ORDER:
            group = [t for t in active_tensors if t["bucket"] == bucket and t["bytes"] > 0]
            if not group:
                continue
            pos = align_up(pos, 4096)
            section_offset = pos
            for tensor in group:
                pos = align_up(pos, 4096)
                tensor_offset = pos
                source_segments.append({
                    "offset": tensor_offset,
                    "size": tensor["bytes"],
                    "source_offset": tensor["source_offset"],
                })
                runtime_priority = tensor_runtime_priority(tensor["name"], bucket, tensor["bytes"])
                tensor_records.append({
                    "name": tensor["name"],
                    "bucket": bucket,
                    "dims": tensor["dims"],
                    "shape": tensor["shape"],
                    "gguf_type": tensor["type"],
                    "gguf_type_name": gguf_type_name(tensor["type"]),
                    "weight_encoding": weight_encoding_from_gguf_type(tensor["type"], contract),
                    "quant_family": quant_family_from_gguf_type(tensor["type"], contract),
                    "kernel_key": kernel_key_from_gguf_type(tensor["type"], contract),
                    "row_layout": "source_gguf_quant_block_layout_preserved",
                    "source_offset": tensor["source_offset"],
                    "source_bytes": tensor["bytes"],
                    "juju_offset": tensor_offset,
                    "juju_bytes": tensor["bytes"],
                    "alignment": 4096,
                    "kernel_contract": {
                        "must_have_dot_kernel": True,
                        "must_not_return_silent_zero": True,
                        "decode_key": kernel_key_from_gguf_type(tensor["type"], contract),
                        "source_type_preserved": True,
                    },
                    **runtime_priority,
                })
                pos += tensor["bytes"]
            size = pos - section_offset
            section_type = section_type_for_bucket(bucket)
            sections.append({
                "type": section_type,
                "name": bucket.upper()[:32],
                "offset": section_offset,
                "size": size,
                "sha256": "0" * 64,
                "prefetch_distance": 2 if bucket != "shared_weights" else 0,
                "mmap_friendly": 1,
            })
            section_sizes[section_type] = section_sizes.get(section_type, 0) + size

        runtime_arch = juju_runtime_arch_metadata(contract, directory)
        graph_ir = build_juju_graph_ir(
            contract=contract,
            tensor_records=tensor_records,
            sections=list(sections),
            source_name=source_name,
            source_path=source_path,
            source_repo_id=source_repo_id,
            weight_file=artifact_names["weights"],
            index_file=artifact_names["index"],
            directory=directory,
        )
        idx = {
            "format": "JUJU_IDX_JSON_V1",
            "schema_version": 2,
            "mutable_runtime_index": True,
            "weight_file": artifact_names["weights"],
            "source_repo_id": source_repo_id,
            "source_path": source_path,
            "source_name": source_name,
            "artifact_source_name": artifact_source_name,
            "split": split_meta,
            "modality_flags": modality_flags,
            "multimodal_contract": modality_meta,
            "graph_ir_format": graph_ir["format"],
            "graph_ir_required": True,
            "graph_ir": graph_ir,
            "priority_tables": graph_ir["priority_tables"],
            "moe_offload_policy": graph_ir["moe_offload_policy"],
            "tensor_count": len(tensor_records),
            "tensors": tensor_records,
            "sections": list(sections),
            **runtime_arch,
        }
        pos = add_json_section_at(pos, JUJU_SECTION_LAYER_ORDER_INDEX, "TENSOR_INDEX", idx)
        index_checksum = int(sections[-1].get("sha256", "0" * 64)[:16], 16) if sections else 0
        file_size_value = pos
        if len(sections) > JUJU_SECTION_TABLE_RESERVED_ENTRIES:
            raise RuntimeError(f"too many JUJU sections: {len(sections)}")

        table = b"".join(pack_section(entry) for entry in sections)
        table_capacity = JUJU_SECTION_TABLE_RESERVED_ENTRIES * JUJU_SECTION_ENTRY_BYTES
        table = table + (b"\x00" * (table_capacity - len(table)))
        header = make_header(contract, artifact_source_name, file_size_value, sections, section_sizes, index_checksum=index_checksum, modality_flags=modality_flags)
        add_fixed(0, header)
        add_fixed(table_offset, table)

    idx["sections"] = sections
    return {
        "format": "juju_sharded_container_v1",
        "source_url": source_url,
        "source_name": source_name,
        "artifact_source_name": artifact_source_name,
        "source_path": source_path,
        "source_repo_id": source_repo_id,
        "weight_file": artifact_names["weights"],
        "index_file": artifact_names["index"],
        "verify_file": artifact_names["verify"],
        "split": split_meta,
        "bytes": file_size_value,
        "source_bytes": total_bytes,
        "tensor_count": len(tensor_records),
        "section_count": len(sections),
        "storage_mode": "remote_range_to_streamed_4kb_aligned_juju_sections",
        "artifact_name_policy": "original_shard_stem_with_juju_extension",
        "fixed_segments": fixed_segments,
        "source_segments": source_segments,
        "index_json": idx,
        "chunk_size": int(chunk_size),
        "token": token,
    }


class JujuVirtualFile(io.BufferedIOBase):
    def __init__(self, plan):
        super().__init__()
        self._lock = threading.RLock()
        self._plan = plan
        self._size = int(plan["bytes"])
        self._pos = 0
        self._session = requests.Session()
        self._remote_chunk = max(1, int(plan.get("chunk_size") or (16 * 1024 * 1024)))
        self._cache_start = -1
        self._cache_end = -1
        self._cache_data = b""
        segments = []
        for segment in plan["fixed_segments"]:
            segments.append({
                "kind": "fixed",
                "offset": int(segment["offset"]),
                "size": int(segment["size"]),
                "data": segment["data"],
            })
        for segment in plan["source_segments"]:
            segments.append({
                "kind": "source",
                "offset": int(segment["offset"]),
                "size": int(segment["size"]),
                "source_offset": int(segment["source_offset"]),
            })
        self._segments = sorted(segments, key=lambda item: item["offset"])
        self._offsets = [item["offset"] for item in self._segments]

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        with self._lock:
            return self._pos

    def seek(self, offset, whence=io.SEEK_SET):
        with self._lock:
            if whence == io.SEEK_SET:
                pos = int(offset)
            elif whence == io.SEEK_CUR:
                pos = self._pos + int(offset)
            elif whence == io.SEEK_END:
                pos = self._size + int(offset)
            else:
                raise ValueError(f"unsupported whence: {whence}")
            if pos < 0:
                raise ValueError("negative seek position")
            self._pos = min(pos, self._size)
            return self._pos

    def readinto(self, buffer):
        with self._lock:
            data = self.read(len(buffer))
            n = len(data)
            buffer[:n] = data
            return n

    def read(self, size=-1):
        with self._lock:
            if self.closed or self._pos >= self._size:
                return b""
            if size is None or size < 0:
                end = min(self._size, self._pos + self._remote_chunk)
            else:
                end = min(self._size, self._pos + int(size))
            chunks = []
            while self._pos < end:
                idx = bisect.bisect_right(self._offsets, self._pos) - 1
                segment = self._segments[idx] if idx >= 0 else None
                if segment and self._pos < segment["offset"] + segment["size"]:
                    rel = self._pos - segment["offset"]
                    take = min(end - self._pos, segment["size"] - rel)
                    if segment["kind"] == "fixed":
                        chunks.append(segment["data"][rel:rel + take])
                    else:
                        chunks.append(self._read_source_segment(segment, rel, take))
                    self._pos += take
                    continue
                next_offset = self._segments[idx + 1]["offset"] if idx + 1 < len(self._segments) else self._size
                take = min(end - self._pos, next_offset - self._pos)
                chunks.append(b"\x00" * take)
                self._pos += take
            return b"".join(chunks)

    def _read_source_segment(self, segment, rel, size):
        out = []
        remaining = int(size)
        source_abs = int(segment["source_offset"]) + int(rel)
        segment_end = int(segment["source_offset"]) + int(segment["size"])
        while remaining > 0:
            if self._cache_start <= source_abs < self._cache_end:
                cache_rel = source_abs - self._cache_start
                take = min(remaining, self._cache_end - source_abs)
                out.append(self._cache_data[cache_rel:cache_rel + take])
                source_abs += take
                remaining -= take
                continue
            fetch_end = min(segment_end, source_abs + max(self._remote_chunk, remaining)) - 1
            resp = fetch_range(
                self._session,
                self._plan["source_url"],
                source_abs,
                fetch_end,
                token=self._plan.get("token"),
                stream=False,
            )
            try:
                data = resp.content
            finally:
                resp.close()
            if not data:
                raise EOFError("empty source range while streaming JUJU upload")
            self._cache_start = source_abs
            self._cache_end = source_abs + len(data)
            self._cache_data = data
        return b"".join(out)

    def close(self):
        try:
            self._session.close()
        finally:
            super().close()


def prepare_juju_shard_upload_from_hf_url(
    *,
    source_url,
    source_name,
    source_path,
    index_path,
    contract,
    token=None,
    source_repo_id="",
    chunk_size=16 * 1024 * 1024,
    artifact_source_name=None,
    tensor_name_allowlist=None,
    split_info=None,
):
    index_path = Path(index_path)
    index_path.parent.mkdir(parents=True, exist_ok=True)
    plan = build_juju_shard_plan_from_hf_url(
        source_url=source_url,
        source_name=source_name,
        source_path=source_path,
        contract=contract,
        token=token,
        source_repo_id=source_repo_id,
        chunk_size=chunk_size,
        artifact_source_name=artifact_source_name,
        tensor_name_allowlist=tensor_name_allowlist,
        split_info=split_info,
    )
    index_path.write_text(json.dumps(plan["index_json"], ensure_ascii=False, indent=2), encoding="utf-8")
    info = {
        "format": plan["format"],
        "path": f"<stream:{plan['weight_file']}>",
        "index_path": str(index_path),
        "source_name": plan["source_name"],
        "artifact_source_name": plan["artifact_source_name"],
        "weight_file": plan["weight_file"],
        "index_file": plan["index_file"],
        "verify_file": plan["verify_file"],
        "split": plan["split"],
        "bytes": plan["bytes"],
        "index_bytes": index_path.stat().st_size,
        "index_sha256": sha256_file(index_path),
        "source_bytes": plan["source_bytes"],
        "source_sha256": "",
        "tensor_count": plan["tensor_count"],
        "section_count": plan["section_count"],
        "storage_mode": plan["storage_mode"],
        "artifact_name_policy": plan["artifact_name_policy"],
    }
    return info, JujuVirtualFile(plan)


def prepare_juju_shard_upload_parts_from_hf_url(
    *,
    source_url,
    source_name,
    source_path,
    index_path,
    contract,
    token=None,
    source_repo_id="",
    chunk_size=16 * 1024 * 1024,
    max_file_bytes=None,
):
    with requests.Session() as session:
        directory, total_bytes = read_remote_directory(session, source_url, token=token)
    split_plan = plan_juju_tensor_splits(directory, max_file_bytes=max_file_bytes)
    base_index_path = Path(index_path)
    parts = []
    for split in split_plan:
        if split["enabled"]:
            artifact_source_name = juju_split_source_name(source_name, split["split_index"], split["split_count"])
            child_index_path = base_index_path.parent / juju_artifact_names(artifact_source_name)["index"]
        else:
            artifact_source_name = source_name
            child_index_path = base_index_path
        split_info = {
            "enabled": bool(split["enabled"]),
            "parent_source_name": source_name,
            "artifact_source_name": artifact_source_name,
            "split_index": int(split["split_index"]),
            "split_count": int(split["split_count"]),
            "source_tensor_count": int(directory["tensor_count"]),
            "tensor_count": len(split["tensor_names"]),
            "tensor_bytes": int(split["tensor_bytes"]),
            "estimated_file_bytes": int(split.get("estimated_file_bytes") or 0),
            "max_file_bytes": int(split["max_file_bytes"]),
            "split_strategy": str(split.get("split_strategy") or "limit_tensor_groups"),
            "target_split_count": int(split.get("target_split_count") or 0),
        }
        info, stream = prepare_juju_shard_upload_from_hf_url(
            source_url=source_url,
            source_name=source_name,
            source_path=source_path,
            index_path=child_index_path,
            contract=contract,
            token=token,
            source_repo_id=source_repo_id,
            chunk_size=chunk_size,
            artifact_source_name=artifact_source_name,
            tensor_name_allowlist=split["tensor_names"],
            split_info=split_info,
        )
        info["source_bytes"] = total_bytes
        parts.append((info, stream))
    return parts


def write_juju_shard_from_hf_url(
    *,
    source_url,
    source_name,
    source_path,
    output_path,
    index_path,
    contract,
    token=None,
    source_repo_id="",
    chunk_size=16 * 1024 * 1024,
    artifact_source_name=None,
    tensor_name_allowlist=None,
    split_info=None,
):
    artifact_source_name = artifact_source_name or source_name
    output_path = Path(output_path)
    index_path = Path(index_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    index_path.parent.mkdir(parents=True, exist_ok=True)
    sections = []
    section_sizes = {}
    tensor_records = []

    def add_json_section(out, section_type, name, payload):
        raw = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
        write_padding(out, 4096)
        offset = out.tell()
        out.write(raw)
        entry = {
            "type": section_type,
            "name": name,
            "offset": offset,
            "size": len(raw),
            "sha256": hashlib.sha256(raw).hexdigest(),
            "mmap_friendly": 0,
        }
        sections.append(entry)
        section_sizes[section_type] = section_sizes.get(section_type, 0) + len(raw)

    with requests.Session() as session:
        directory, total_bytes = read_remote_directory(session, source_url, token=token)
        allowset = set(tensor_name_allowlist or [])
        if allowset:
            known = {tensor["name"] for tensor in directory["tensors"]}
            missing = sorted(allowset - known)
            if missing:
                raise RuntimeError(f"JUJU split references missing tensors: {missing[:8]}")
        active_tensors = [
            tensor for tensor in directory["tensors"]
            if int(tensor.get("bytes") or 0) > 0 and (not allowset or tensor["name"] in allowset)
        ]
        split_meta = split_info or {
            "enabled": False,
            "split_index": 1,
            "split_count": 1,
            "parent_source_name": source_name,
            "artifact_source_name": artifact_source_name,
            "tensor_count": len(active_tensors),
        }
        modality_meta = juju_modality_metadata(contract, active_tensors)
        modality_flags = int(modality_meta["modality_flags"])
        with output_path.open("wb") as out:
            out.write(b"\x00" * JUJU_HEADER_BYTES)
            table_offset = out.tell()
            out.write(b"\x00" * (JUJU_SECTION_TABLE_RESERVED_ENTRIES * JUJU_SECTION_ENTRY_BYTES))
            write_padding(out, 4096)

            meta = {
                "format": "JUJU_SHARDED_CONTAINER_V1",
                "source_format": "GGUF",
                "source_role": "conversion_source_only",
                "source_repo_id": source_repo_id,
                "source_path": source_path,
                "source_name": source_name,
                "artifact_source_name": artifact_source_name,
                "weight_file": output_path.name,
                "index_file": index_path.name,
                "tensor_payload_layout": "4kb_aligned_tensor_sections",
                "artifact_name_policy": "preserve_original_shard_stem_change_extension_only",
                "graph_ir_format": "JUJU_GRAPH_IR_V1",
                "graph_ir_required": True,
                "split": split_meta,
                "gguf_directory": {
                    "version": directory["version"],
                    "tensor_count": directory["tensor_count"],
                    "emitted_tensor_count": len(active_tensors),
                    "kv_count": directory["kv_count"],
                    "alignment": directory["alignment"],
                    "data_start": directory["data_start"],
                    "source_bytes": total_bytes,
                },
                "contract": contract,
                "modality_flags": modality_flags,
                "multimodal_contract": modality_meta,
                **juju_contract_metadata(contract, source_name, source_repo_id),
            }
            add_json_section(out, JUJU_SECTION_MODEL_META, "MODEL_META", meta)
            qkv_schema = contract.get("qkv_cache_schema")
            if qkv_schema:
                add_json_section(out, JUJU_SECTION_QKV_POLICY, "QKV_POLICY", qkv_schema)

            for bucket in JUJU_TENSOR_BUCKET_ORDER:
                group = [t for t in active_tensors if t["bucket"] == bucket and t["bytes"] > 0]
                if not group:
                    continue
                write_padding(out, 4096)
                offset = out.tell()
                digest = hashlib.sha256()
                for tensor in group:
                    write_padding(out, 4096)
                    tensor_offset = out.tell()
                    stream_range(
                        session,
                        source_url,
                        tensor["source_offset"],
                        tensor["bytes"],
                        out,
                        token,
                        digest,
                        chunk_size=chunk_size,
                    )
                    runtime_priority = tensor_runtime_priority(tensor["name"], bucket, tensor["bytes"])
                    tensor_records.append({
                        "name": tensor["name"],
                        "bucket": bucket,
                        "dims": tensor["dims"],
                        "shape": tensor["shape"],
                        "gguf_type": tensor["type"],
                        "gguf_type_name": gguf_type_name(tensor["type"]),
                        "weight_encoding": weight_encoding_from_gguf_type(tensor["type"], contract),
                        "quant_family": quant_family_from_gguf_type(tensor["type"], contract),
                        "kernel_key": kernel_key_from_gguf_type(tensor["type"], contract),
                        "row_layout": "source_gguf_quant_block_layout_preserved",
                        "source_offset": tensor["source_offset"],
                        "source_bytes": tensor["bytes"],
                        "juju_offset": tensor_offset,
                        "juju_bytes": tensor["bytes"],
                        "alignment": 4096,
                        "kernel_contract": {
                            "must_have_dot_kernel": True,
                            "must_not_return_silent_zero": True,
                            "decode_key": kernel_key_from_gguf_type(tensor["type"], contract),
                            "source_type_preserved": True,
                        },
                        **runtime_priority,
                    })
                size = out.tell() - offset
                section_type = section_type_for_bucket(bucket)
                sections.append({
                    "type": section_type,
                    "name": bucket.upper()[:32],
                    "offset": offset,
                    "size": size,
                    "sha256": digest.hexdigest(),
                    "prefetch_distance": 2 if bucket != "shared_weights" else 0,
                    "mmap_friendly": 1,
                })
                section_sizes[section_type] = section_sizes.get(section_type, 0) + size

            runtime_arch = juju_runtime_arch_metadata(contract, directory)
            graph_ir = build_juju_graph_ir(
                contract=contract,
                tensor_records=tensor_records,
                sections=list(sections),
                source_name=source_name,
                source_path=source_path,
                source_repo_id=source_repo_id,
                weight_file=output_path.name,
                index_file=index_path.name,
                directory=directory,
            )
            idx = {
                "format": "JUJU_IDX_JSON_V1",
                "schema_version": 2,
                "mutable_runtime_index": True,
                "weight_file": output_path.name,
                "source_repo_id": source_repo_id,
                "source_path": source_path,
                "source_name": source_name,
                "artifact_source_name": artifact_source_name,
                "split": split_meta,
                "modality_flags": modality_flags,
                "multimodal_contract": modality_meta,
                "graph_ir_format": graph_ir["format"],
                "graph_ir_required": True,
                "graph_ir": graph_ir,
                "priority_tables": graph_ir["priority_tables"],
                "moe_offload_policy": graph_ir["moe_offload_policy"],
                "tensor_count": len(tensor_records),
                "tensors": tensor_records,
                "sections": sections,
                **runtime_arch,
            }
            add_json_section(out, JUJU_SECTION_LAYER_ORDER_INDEX, "TENSOR_INDEX", idx)
            index_checksum = int(sections[-1].get("sha256", "0" * 64)[:16], 16) if sections else 0
            file_size_value = out.tell()
            if len(sections) > JUJU_SECTION_TABLE_RESERVED_ENTRIES:
                raise RuntimeError(f"too many JUJU sections: {len(sections)}")
            out.seek(table_offset)
            for entry in sections:
                out.write(pack_section(entry))
            out.seek(0)
            out.write(make_header(contract, artifact_source_name, file_size_value, sections, section_sizes, index_checksum=index_checksum, modality_flags=modality_flags))

    index_path.write_text(json.dumps(idx, ensure_ascii=False, indent=2), encoding="utf-8")
    return {
        "format": "juju_sharded_container_v1",
        "path": str(output_path),
        "index_path": str(index_path),
        "source_name": source_name,
        "artifact_source_name": artifact_source_name,
        "weight_file": output_path.name,
        "index_file": index_path.name,
        "verify_file": juju_artifact_names(artifact_source_name)["verify"],
        "split": split_meta,
        "bytes": output_path.stat().st_size,
        "index_bytes": index_path.stat().st_size,
        "sha256": sha256_file(output_path),
        "index_sha256": sha256_file(index_path),
        "source_bytes": total_bytes,
        "source_sha256": "",
        "tensor_count": len(tensor_records),
        "section_count": len(sections),
        "storage_mode": "remote_range_to_4kb_aligned_juju_sections",
        "artifact_name_policy": "original_shard_stem_with_juju_extension",
    }


def write_juju_shard_parts_from_hf_url(
    *,
    source_url,
    source_name,
    source_path,
    output_path,
    index_path,
    contract,
    token=None,
    source_repo_id="",
    chunk_size=16 * 1024 * 1024,
    max_file_bytes=None,
):
    with requests.Session() as session:
        directory, total_bytes = read_remote_directory(session, source_url, token=token)
    split_plan = plan_juju_tensor_splits(directory, max_file_bytes=max_file_bytes)
    base_output_path = Path(output_path)
    base_index_path = Path(index_path)
    infos = []
    for split in split_plan:
        if split["enabled"]:
            artifact_source_name = juju_split_source_name(source_name, split["split_index"], split["split_count"])
            child_output_path = base_output_path.parent / juju_artifact_names(artifact_source_name)["weights"]
            child_index_path = base_index_path.parent / juju_artifact_names(artifact_source_name)["index"]
        else:
            artifact_source_name = source_name
            child_output_path = base_output_path
            child_index_path = base_index_path
        split_info = {
            "enabled": bool(split["enabled"]),
            "parent_source_name": source_name,
            "artifact_source_name": artifact_source_name,
            "split_index": int(split["split_index"]),
            "split_count": int(split["split_count"]),
            "source_tensor_count": int(directory["tensor_count"]),
            "tensor_count": len(split["tensor_names"]),
            "tensor_bytes": int(split["tensor_bytes"]),
            "estimated_file_bytes": int(split.get("estimated_file_bytes") or 0),
            "max_file_bytes": int(split["max_file_bytes"]),
            "split_strategy": str(split.get("split_strategy") or "limit_tensor_groups"),
            "target_split_count": int(split.get("target_split_count") or 0),
        }
        info = write_juju_shard_from_hf_url(
            source_url=source_url,
            source_name=source_name,
            source_path=source_path,
            output_path=child_output_path,
            index_path=child_index_path,
            contract=contract,
            token=token,
            source_repo_id=source_repo_id,
            chunk_size=chunk_size,
            artifact_source_name=artifact_source_name,
            tensor_name_allowlist=split["tensor_names"],
            split_info=split_info,
        )
        info["source_bytes"] = total_bytes
        infos.append(info)
    return infos


In [ ]:
def ensure_hf_api():
    try:
        from huggingface_hub import HfApi, create_repo
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"])
        from huggingface_hub import HfApi, create_repo
    return HfApi, create_repo


def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    if token:
        return token
    return getpass("HF write token. Add it to Colab Secrets as HF_TOKEN to avoid pasting: ").strip()


def gguf_verify_json_name(spec):
    return f"{Path(spec['name']).stem}.offload.verify.json"


def target_gguf_repo_path(spec):
    return repo_path_join(TARGET_SUBDIR, spec["name"])


def target_juju_repo_path(spec):
    return repo_path_join(TARGET_SUBDIR, juju_artifact_names(spec["name"])["weights"])


def target_juju_index_repo_path(spec):
    return repo_path_join(TARGET_SUBDIR, juju_artifact_names(spec["name"])["index"])


def juju_verify_json_name(spec):
    return juju_artifact_names(spec["name"])["verify"]


def source_gguf_hub_url(spec):
    from huggingface_hub import hf_hub_url
    return hf_hub_url(
        repo_id=SOURCE_HF_REPO_ID,
        filename=spec["source_path"],
        repo_type="model",
    )


def portable_stem_for_gguf(spec):
    return Path(spec["name"]).stem


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def json_compact(obj):
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), sort_keys=True)


def beta_pdf(x, dim):
    if x <= -1.0 or x >= 1.0:
        return 0.0
    variance = 1.0 / float(dim)
    sigma = math.sqrt(variance)
    return math.exp(-(x * x) / (2.0 * variance)) / (sigma * math.sqrt(2.0 * math.pi))


def lloyd_max_codebook(bits, dim, max_iters=100):
    levels = 1 << int(bits)
    sigma = 1.0 / math.sqrt(float(dim))
    support_min = -1.0
    support_max = 1.0
    if bits == 2:
        centroids = [-1.51 * sigma, -0.453 * sigma, 0.453 * sigma, 1.51 * sigma]
        thresholds = [support_min, (-1.51 - 0.453) * 0.5 * sigma, 0.0, (0.453 + 1.51) * 0.5 * sigma, support_max]
        return {"bits": bits, "dim": dim, "centroids": centroids, "thresholds": thresholds, "threshold_encoding": "finite_normalized_support"}
    centroids = [(-3.5 + 7.0 * i / max(1, levels - 1)) * sigma for i in range(levels)]
    thresholds = [0.0] * (levels + 1)
    for _ in range(max_iters):
        thresholds[0] = support_min
        for i in range(1, levels):
            thresholds[i] = (centroids[i - 1] + centroids[i]) * 0.5
        thresholds[levels] = support_max
        converged = True
        for i in range(levels):
            steps = 1000
            lo = thresholds[i]
            hi = thresholds[i + 1]
            step = (hi - lo) / float(steps)
            sum_x = 0.0
            sum_w = 0.0
            for j in range(steps + 1):
                x = lo + j * step
                w = beta_pdf(x, dim)
                sum_x += x * w
                sum_w += w
            if sum_w > 1e-10:
                nxt = sum_x / sum_w
                if abs(nxt - centroids[i]) > 1e-6:
                    converged = False
                centroids[i] = nxt
        if converged:
            break
    return {"bits": bits, "dim": dim, "centroids": centroids, "thresholds": thresholds, "threshold_encoding": "finite_normalized_support"}


def xorshift64(value):
    value &= (1 << 64) - 1
    value ^= (value << 13) & ((1 << 64) - 1)
    value ^= value >> 7
    value ^= (value << 17) & ((1 << 64) - 1)
    return value & ((1 << 64) - 1)


def deterministic_signs(count, seed):
    value = int(seed) or 1
    signs = []
    for _ in range(int(count)):
        value = xorshift64(value)
        signs.append(1 if (value & 1) else -1)
    return signs


def build_qkv_aux(profile, head_dim_override=None):
    qkv = profile["qkv_cache_schema"]
    head_dim = int(head_dim_override or profile["arch_meta"].get("head_dim") or 128)
    bits = sorted({int(qkv["k_bits"]), int(qkv["v_bits"]), int(qkv["outlier"]["bits"])})
    return {
        "head_dim": head_dim,
        "codebooks": {str(bit): lloyd_max_codebook(bit, head_dim) for bit in bits},
        "rotation_aux": {
            "type": "hadamard_signs",
            "seed": int(qkv["rotation"]["seed"]),
            "signs": deterministic_signs(head_dim, int(qkv["rotation"]["seed"])),
        },
        "qjl_aux": {
            "type": "seed_only",
            "seed": int(qkv["qjl"]["seed"]),
            "materialized": False,
        },
        "outlier_channel_map": list(range(int(qkv["outlier"]["channels"]))),
    }


_QKV_AUX_CACHE = {}


def get_qkv_aux(head_dim=None):
    global _QKV_AUX_CACHE
    key = int(head_dim or MODEL_PROFILE["arch_meta"].get("head_dim") or 128)
    if key not in _QKV_AUX_CACHE:
        _QKV_AUX_CACHE[key] = build_qkv_aux(MODEL_PROFILE, head_dim_override=key)
    return _QKV_AUX_CACHE[key]


def build_gguf_kv_entries(contract):
    qkv = contract["qkv_cache_schema"]
    residency = contract["residency_policy"]
    execution = contract["execution_hints"]
    scalar_entries = {
        "offload.format_version": ("uint32", 2),
        "offload.backend_neutral": ("bool", True),
        "offload.async_channel_count": ("uint32", int(execution["async_transfer"]["channel_count"])),
        "offload.simd_width_hint": ("uint32", int(execution.get("simd_width_hint", 0))),
        "offload.weight_quant_family": ("string", str(contract["source_weight_quant_family"])),
        "offload.weight_kernel_family": ("string", str(contract["source_weight_kernel_family"])),
        "offload.weight_bits": ("uint32", int(contract["source_weight_bits"])),
        "offload.weight_block_size": ("uint32", int(contract["source_weight_block_size"])),
        "offload.optimization_catalog_count": ("uint32", int(contract["optimization_catalog_v1"]["actual_count"])),
        "offload.qkv_k_bits": ("uint32", int(qkv["k_bits"])),
        "offload.qkv_v_bits": ("uint32", int(qkv["v_bits"])),
        "offload.qkv_group_size": ("uint32", int(qkv["group_size"])),
        "offload.qkv_page_size_tokens": ("uint32", int(qkv["page_size_tokens"])),
        "offload.qkv_sink_tokens": ("uint32", int(qkv["residency"]["sink_tokens"])),
        "offload.anchor_layers": ("string", json_compact(residency["anchors"]["layers"])),
        "offload.valid_split_points": ("string", json_compact(residency["valid_split_points"])),
        "offload.reentry_points": ("string", json_compact(residency["reentry_points"])),
        "offload.pipeline_barrier_points": ("string", json_compact(execution["pipeline_barriers"]["points"])),
    }
    if "file_index" in contract:
        scalar_entries["offload.file_index"] = ("uint32", int(contract["file_index"]))
    if "file_count" in contract:
        scalar_entries["offload.file_count"] = ("uint32", int(contract["file_count"]))
    if "target_file" in contract:
        scalar_entries["offload.target_file"] = ("string", str(contract["target_file"]))
    complex_blocks = {
        "offload.metadata_v2": contract,
        "offload.optimization_catalog_v1": contract["optimization_catalog_v1"],
        "offload.juju_container_contract": contract["optimization_catalog_v1"]["juju_container_contract"],
        "offload.final_model_structure_contract": contract["optimization_catalog_v1"]["final_model_structure_contract"],
        "offload.routing_optimization_contract": contract["optimization_catalog_v1"]["routing_optimization_contract"],
        "offload.code_generation_moe_contract": contract["optimization_catalog_v1"]["code_generation_moe_contract"],
        "offload.structure_finalization_contract": contract["optimization_catalog_v1"]["structure_finalization_contract"],
        "offload.expert_segmentation_contract": contract["optimization_catalog_v1"]["expert_segmentation_contract"],
        "offload.moepic_segment_contract": contract["optimization_catalog_v1"]["moepic_segment_contract"],
        "offload.moeshard_matrix_split_contract": contract["optimization_catalog_v1"]["moeshard_matrix_split_contract"],
        "offload.chunk_io_contract": contract["optimization_catalog_v1"]["chunk_io_contract"],
        "offload.universal_tier_contract": contract["optimization_catalog_v1"]["universal_tier_contract"],
        "offload.pipeline_budget_contract": contract["optimization_catalog_v1"]["pipeline_budget_contract"],
        "offload.execution_path_contract": contract["optimization_catalog_v1"]["execution_path_contract"],
        "offload.path_specific_io_hint_schema": contract["optimization_catalog_v1"]["path_specific_io_hint_schema"],
        "offload.layer_order_index_schema": contract["optimization_catalog_v1"]["layer_order_index_schema"],
        "offload.juju_physical_container_contract": contract["optimization_catalog_v1"]["juju_physical_container_contract"],
        "offload.juju_header_schema": contract["optimization_catalog_v1"]["juju_header_schema"],
        "offload.juju_section_table_schema": contract["optimization_catalog_v1"]["juju_section_table_schema"],
        "offload.juju_expert_chunk_schema": contract["optimization_catalog_v1"]["juju_expert_chunk_schema"],
        "offload.juju_index_contract": contract["optimization_catalog_v1"]["juju_index_contract"],
        "offload.prediction_contract": contract["optimization_catalog_v1"]["prediction_contract"],
        "offload.hardware_probe_contract": contract["optimization_catalog_v1"]["hardware_probe_contract"],
        "offload.runtime_design_principles": contract["optimization_catalog_v1"]["runtime_design_principles"],
        "offload.offload_strategy_contract": contract["optimization_catalog_v1"]["offload_strategy_contract"],
        "offload.adaptive_controller_contract": contract["optimization_catalog_v1"]["adaptive_controller_contract"],
        "offload.gguf_to_juju_converter_contract": contract["optimization_catalog_v1"]["gguf_to_juju_converter_contract"],
        "offload.qkv_cache_contract": contract["optimization_catalog_v1"]["qkv_cache_contract"],
        "offload.format_layout_contract": contract["optimization_catalog_v1"]["format_layout_contract"],
        "offload.engine_pipeline_contract": contract["optimization_catalog_v1"]["engine_pipeline_contract"],
        "offload.expert_scheduler_contract": contract["optimization_catalog_v1"]["expert_scheduler_contract"],
        "offload.runtime_update_files": contract["optimization_catalog_v1"]["runtime_update_files"],
        "offload.arch_meta": contract["arch_meta"],
        "offload.weight_quant_schema": contract["weight_quant_schema"],
        "offload.qkv_cache_schema": contract["qkv_cache_schema"],
        "offload.qkv_aux_data": contract["qkv_aux_data"],
        "offload.residency_policy": contract["residency_policy"],
        "offload.execution_hints": contract["execution_hints"],
        "offload.dynamic_swap_triggers": contract["dynamic_swap_triggers"],
        "offload.memory_management_hints": contract["memory_management_hints"],
        "offload.prefetch_plan_hints": contract["prefetch_plan_hints"],
        "offload.kernel_hints": contract["kernel_hints"],
        "offload.kernel_config_hints": contract["kernel_config_hints"],
        "offload.cpu_execution_hints": contract["cpu_execution_hints"],
        "offload.context_length_table": contract["context_length_table"],
        "offload.io_schedule": contract["io_schedule"],
        "offload.streaming_attention": contract["streaming_attention"],
        "offload.numerical_stability": contract["numerical_stability"],
        "offload.continuous_batching": contract["continuous_batching"],
        "offload.batch_schedule": contract["batch_schedule"],
        "offload.tensor_layout_schema": contract["tensor_layout_schema"],
        "offload.integrity_schema": contract["integrity_schema"],
        "offload.per_layer_swap_cost": contract["per_layer_swap_cost"],
        "offload.runtime_monitoring_hints": contract["runtime_monitoring_hints"],
        "offload.thermal_power_hints": contract["thermal_power_hints"],
        "offload.speculative_execution_schema": contract["speculative_execution_schema"],
        "offload.activation_checkpoint_schema": contract["activation_checkpoint_schema"],
        "offload.mig_hints": contract["mig_hints"],
        "offload.profiling_measured_data": contract["profiling_measured_data"],
        "offload.multi_accelerator_detail": contract["multi_accelerator_detail"],
        "offload.compatibility": contract["compatibility"],
        "offload.runtime_note": contract["runtime_note"],
        "offload.sglang_runtime_note": contract["sglang_runtime_note"],
        "offload.engine_contract": contract["engine_contract"],
        "offload.backend_translation_contract": contract["backend_translation_contract"],
    }
    entries = dict(scalar_entries)
    for key, value in complex_blocks.items():
        entries[key] = ("string", json_compact(value))
    for key in ("container_contract", "part_contract_coverage", "current_file_metadata", "shard_set_manifest", "files_present"):
        if key in contract:
            entries[f"offload.{key}"] = ("string", json_compact(contract[key]))
    if contract.get("source_gguf_meta"):
        entries["offload.source_gguf_meta"] = ("string", json_compact(contract["source_gguf_meta"]))
    return entries


def gguf_strategy_json(entries, include_metadata_v2):
    out = {}
    for key, (dtype, value) in entries.items():
        if not include_metadata_v2 and key == "offload.metadata_v2":
            continue
        out[key] = {"type": dtype, "value": value}
    return out


def build_safetensors_metadata(contract):
    entries = build_gguf_kv_entries(contract)
    meta = {
        "format": "pt",
        "offload.format_version": "2",
        "offload.backend_neutral": "true",
    }
    for key, (_dtype, value) in entries.items():
        if key in {"offload.format_version", "offload.backend_neutral"}:
            continue
        meta[key] = value if isinstance(value, str) else json_compact(value)
    return {"__metadata__": meta}


def write_foreign_metadata_exports_for_stem(contract, stem):
    entries = build_gguf_kv_entries(contract)
    paths = {}
    paths["gguf_strategy_a"] = GGUF_EXPORT_DIR / f"{stem}.gguf-kv.strategy_a.json"
    paths["gguf_strategy_b"] = GGUF_EXPORT_DIR / f"{stem}.gguf-kv.strategy_b.json"
    paths["sidecar"] = SIDECAR_EXPORT_DIR / f"{stem}.offload.json"
    paths["safetensors_metadata"] = SAFETENSORS_EXPORT_DIR / f"{stem}.safetensors-metadata.json"

    paths["gguf_strategy_a"].write_text(json.dumps({
        "strategy": "A",
        "description": "GGUF native scalar KVs plus JSON string blocks",
        "kv": gguf_strategy_json(entries, include_metadata_v2=False),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["gguf_strategy_b"].write_text(json.dumps({
        "strategy": "B",
        "description": "single offload.metadata_v2 JSON plus format scalar",
        "kv": {
            "offload.metadata_v2": {"type": "string", "value": entries["offload.metadata_v2"][1]},
            "offload.format_version": {"type": "uint32", "value": 2},
            "offload.backend_neutral": {"type": "bool", "value": True},
        },
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["sidecar"].write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8")
    paths["safetensors_metadata"].write_text(json.dumps(build_safetensors_metadata(contract), ensure_ascii=False, indent=2), encoding="utf-8")
    return paths


GGUF_TYPE_UINT8 = 0
GGUF_TYPE_INT8 = 1
GGUF_TYPE_UINT16 = 2
GGUF_TYPE_INT16 = 3
GGUF_TYPE_UINT32 = 4
GGUF_TYPE_INT32 = 5
GGUF_TYPE_FLOAT32 = 6
GGUF_TYPE_BOOL = 7
GGUF_TYPE_STRING = 8
GGUF_TYPE_ARRAY = 9
GGUF_TYPE_UINT64 = 10
GGUF_TYPE_INT64 = 11
GGUF_TYPE_FLOAT64 = 12
GGUF_TYPE_SIZE = {
    GGUF_TYPE_UINT8: 1,
    GGUF_TYPE_INT8: 1,
    GGUF_TYPE_UINT16: 2,
    GGUF_TYPE_INT16: 2,
    GGUF_TYPE_UINT32: 4,
    GGUF_TYPE_INT32: 4,
    GGUF_TYPE_FLOAT32: 4,
    GGUF_TYPE_BOOL: 1,
    GGUF_TYPE_UINT64: 8,
    GGUF_TYPE_INT64: 8,
    GGUF_TYPE_FLOAT64: 8,
}


def gguf_read_exact(handle, size):
    data = handle.read(size)
    if len(data) != size:
        raise EOFError("unexpected EOF while reading GGUF")
    return data


def gguf_read_u32(handle):
    return struct.unpack("<I", gguf_read_exact(handle, 4))[0]


def gguf_read_u64(handle):
    return struct.unpack("<Q", gguf_read_exact(handle, 8))[0]


def gguf_read_string_value(handle):
    size = gguf_read_u64(handle)
    return gguf_read_exact(handle, size).decode("utf-8")


def gguf_skip_value(handle, value_type):
    if value_type == GGUF_TYPE_STRING:
        size = gguf_read_u64(handle)
        handle.seek(size, 1)
        return
    if value_type == GGUF_TYPE_ARRAY:
        elem_type = gguf_read_u32(handle)
        count = gguf_read_u64(handle)
        if elem_type == GGUF_TYPE_STRING:
            for _ in range(count):
                size = gguf_read_u64(handle)
                handle.seek(size, 1)
            return
        elem_size = GGUF_TYPE_SIZE.get(elem_type)
        if elem_size is None:
            raise ValueError(f"unsupported GGUF array element type: {elem_type}")
        handle.seek(elem_size * count, 1)
        return
    size = GGUF_TYPE_SIZE.get(value_type)
    if size is None:
        raise ValueError(f"unsupported GGUF value type: {value_type}")
    handle.seek(size, 1)


def gguf_skip_tensor_infos(handle, tensor_count):
    for _ in range(tensor_count):
        _name = gguf_read_string_value(handle)
        dims = gguf_read_u32(handle)
        handle.seek(8 * dims, 1)
        handle.seek(4, 1)
        handle.seek(8, 1)


def gguf_encode_string(text):
    raw = str(text).encode("utf-8")
    return struct.pack("<Q", len(raw)) + raw


def gguf_encode_kv(key, dtype, value):
    key_raw = gguf_encode_string(key)
    if dtype == "uint32":
        return key_raw + struct.pack("<I", GGUF_TYPE_UINT32) + struct.pack("<I", int(value))
    if dtype == "bool":
        return key_raw + struct.pack("<I", GGUF_TYPE_BOOL) + struct.pack("<?", bool(value))
    if dtype == "string":
        return key_raw + struct.pack("<I", GGUF_TYPE_STRING) + gguf_encode_string(value)
    raise ValueError(f"unsupported GGUF export dtype: {dtype}")



def parse_content_range_total(header_value):
    if not header_value:
        return None
    try:
        return int(str(header_value).rsplit("/", 1)[1])
    except Exception:
        return None


def fetch_http_range(session, url, start, end=None, token=None, stream=False):
    headers = {"Range": f"bytes={start}-{'' if end is None else end}"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    resp = session.get(url, headers=headers, stream=stream, timeout=120)
    if resp.status_code != 206:
        resp.close()
        raise RuntimeError(f"server did not honor HTTP range request: status={resp.status_code} url={url}")
    return resp


def remote_file_size(session, url, token=None):
    resp = fetch_http_range(session, url, 0, 0, token=token, stream=False)
    try:
        total = parse_content_range_total(resp.headers.get("Content-Range"))
        if total is None:
            raise RuntimeError(f"missing total size in Content-Range for {url}")
        return total
    finally:
        resp.close()


def resolve_hf_download(spec, token=None):
    from huggingface_hub import get_hf_file_metadata
    hub_url = source_gguf_hub_url(spec)
    meta = get_hf_file_metadata(hub_url, token=token, timeout=60)
    return {
        "hub_url": hub_url,
        "download_url": getattr(meta, "location", None) or hub_url,
        "size": int(getattr(meta, "size", 0) or 0),
        "etag": getattr(meta, "etag", None),
        "commit_hash": getattr(meta, "commit_hash", None),
    }


def parse_gguf_prefix(prefix_raw, contract=None, strategy="A"):
    src = io.BytesIO(prefix_raw)
    source_gguf_meta = {}
    alignment = 32
    magic = gguf_read_exact(src, 4)
    if magic != b"GGUF":
        raise ValueError("not a GGUF prefix")
    _version = gguf_read_u32(src)
    tensor_count = gguf_read_u64(src)
    kv_count = gguf_read_u64(src)
    for _ in range(kv_count):
        key = gguf_read_string_value(src)
        value_type = gguf_read_u32(src)
        if key == "general.alignment" and value_type == GGUF_TYPE_UINT32:
            alignment = struct.unpack("<I", gguf_read_exact(src, 4))[0]
        elif any(key.startswith(prefix) for prefix in GGUF_ARCH_META_PREFIXES):
            source_gguf_meta[key] = gguf_decode_value(src, value_type)
        else:
            gguf_skip_value(src, value_type)
    tensor_info_start = src.tell()
    gguf_skip_tensor_infos(src, tensor_count)
    tensor_info_end = src.tell()
    data_start = ((tensor_info_end + alignment - 1) // alignment) * alignment
    if data_start > len(prefix_raw):
        raise EOFError(f"GGUF header/tensor table needs {data_start} bytes, got {len(prefix_raw)}")
    return {
        "old_data_start": data_start,
        "source_gguf_meta": source_gguf_meta,
    }



def read_remote_gguf_source_meta(url, contract, token=None, total_bytes_hint=0):
    import requests
    header_bytes = GGUF_HEADER_RANGE_BYTES
    with requests.Session() as session:
        total_bytes = int(total_bytes_hint or 0) or remote_file_size(session, url, token=token)
        for _ in range(6):
            end = min(header_bytes, total_bytes) - 1
            resp = fetch_http_range(session, url, 0, end, token=token, stream=False)
            prefix_raw = resp.content
            resp.close()
            try:
                return parse_gguf_prefix(prefix_raw, contract, strategy="A")["source_gguf_meta"]
            except EOFError:
                if header_bytes >= total_bytes:
                    raise
                header_bytes *= 2
    raise RuntimeError("could not read GGUF source metadata from ranged prefix")


def read_hf_gguf_source_meta(spec, contract, token=None):
    info = resolve_hf_download(spec, token=token)
    try:
        return read_remote_gguf_source_meta(info["download_url"], contract, token=None, total_bytes_hint=info.get("size") or 0)
    except RuntimeError as exc:
        size = int(info.get("size") or 0)
        if "status=416" not in str(exc) or size <= 0 or size > SMALL_FILE_DIRECT_GET_LIMIT_BYTES:
            raise
        import requests
        print(f"Range unavailable for small GGUF metadata read ({size} bytes); using direct GET fallback")
        resp = requests.get(info["download_url"], timeout=120)
        resp.raise_for_status()
        return parse_gguf_prefix(resp.content, contract, strategy="A")["source_gguf_meta"]


def gguf_decode_scalar(handle, value_type):
    if value_type == GGUF_TYPE_STRING:
        return gguf_read_string_value(handle)
    scalar_formats = {
        GGUF_TYPE_UINT8: "<B",
        GGUF_TYPE_INT8: "<b",
        GGUF_TYPE_UINT16: "<H",
        GGUF_TYPE_INT16: "<h",
        GGUF_TYPE_UINT32: "<I",
        GGUF_TYPE_INT32: "<i",
        GGUF_TYPE_FLOAT32: "<f",
        GGUF_TYPE_BOOL: "<?",
        GGUF_TYPE_UINT64: "<Q",
        GGUF_TYPE_INT64: "<q",
        GGUF_TYPE_FLOAT64: "<d",
    }
    fmt = scalar_formats.get(value_type)
    if fmt is None:
        raise ValueError(f"unsupported GGUF scalar type: {value_type}")
    return struct.unpack(fmt, gguf_read_exact(handle, GGUF_TYPE_SIZE[value_type]))[0]


def gguf_decode_value(handle, value_type):
    if value_type == GGUF_TYPE_ARRAY:
        elem_type = gguf_read_u32(handle)
        count = gguf_read_u64(handle)
        return [gguf_decode_scalar(handle, elem_type) for _ in range(count)]
    return gguf_decode_scalar(handle, value_type)


def read_gguf_kv(model_path, prefix="offload."):
    model_path = Path(model_path)
    out = {}

    def wanted(key):
        if prefix is None:
            return True
        if isinstance(prefix, (tuple, list, set)):
            return any(key.startswith(item) for item in prefix)
        return key.startswith(prefix)

    with model_path.open("rb") as handle:
        magic = gguf_read_exact(handle, 4)
        if magic != b"GGUF":
            raise ValueError(f"not a GGUF file: {model_path}")
        _version = gguf_read_u32(handle)
        _tensor_count = gguf_read_u64(handle)
        kv_count = gguf_read_u64(handle)
        for _ in range(kv_count):
            key = gguf_read_string_value(handle)
            value_type = gguf_read_u32(handle)
            if wanted(key):
                out[key] = gguf_decode_value(handle, value_type)
            else:
                gguf_skip_value(handle, value_type)
    return out


def parse_json_if_possible(value):
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except json.JSONDecodeError:
        return value


def normalize_offload_metadata(meta, prefer_metadata_v2=True):
    offload = {}
    for key, value in dict(meta).items():
        if not key.startswith("offload."):
            continue
        field = key[len("offload."):]
        offload[field] = parse_json_if_possible(value)
    if prefer_metadata_v2 and isinstance(offload.get("metadata_v2"), dict):
        merged = dict(offload["metadata_v2"])
        for key, value in offload.items():
            if key != "metadata_v2" and key not in merged:
                merged[key] = value
        return merged
    return offload


def read_safetensors_metadata(model_path):
    model_path = Path(model_path)
    with model_path.open("rb") as handle:
        header_size = struct.unpack("<Q", gguf_read_exact(handle, 8))[0]
        header = json.loads(gguf_read_exact(handle, header_size).decode("utf-8"))
    return header.get("__metadata__", {})


def find_offload_sidecar(model_path):
    model_path = Path(model_path)
    stem = model_path.stem
    parent = model_path.parent
    candidates = [
        parent / f"{stem}.offload.json",
        parent / "offload_meta.json",
        parent / "metadata" / "sidecar" / f"{stem}.offload.json",
        parent.parent / "metadata" / "sidecar" / f"{stem}.offload.json",
    ]
    for candidate in candidates:
        if not candidate.exists():
            continue
        try:
            return json.loads(candidate.read_text(encoding="utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError):
            continue
    return None


def load_offload_meta(model_path, prefer_sidecar=False, prefer_metadata_v2=True):
    model_path = Path(model_path)
    sidecar = find_offload_sidecar(model_path)
    if prefer_sidecar and sidecar is not None:
        return sidecar

    ext = model_path.suffix.lower()
    meta = {}
    if ext == ".gguf":
        meta = read_gguf_kv(model_path, prefix="offload.")
    elif ext == ".safetensors":
        meta = read_safetensors_metadata(model_path)
    elif ext == ".json":
        return json.loads(model_path.read_text(encoding="utf-8"))

    offload = normalize_offload_metadata(meta, prefer_metadata_v2=prefer_metadata_v2)
    if offload:
        return offload
    if sidecar is not None:
        return sidecar
    return {}


def build_arch_meta(source_gguf_meta=None):
    arch = dict(MODEL_PROFILE["arch_meta"])
    source_gguf_meta = dict(source_gguf_meta or {})
    if source_gguf_meta.get("general.architecture"):
        arch["model_family"] = source_gguf_meta["general.architecture"]
    key_map = {
        "block_count": "n_layers",
        "attention.head_count": "n_heads",
        "attention.head_count_kv": "n_kv_heads",
        "attention.key_length": "head_dim",
        "attention.value_length": "head_dim",
        "embedding_length": "hidden_dim",
        "feed_forward_length": "ffn_dim",
        "context_length": "max_seq_len",
    }
    for key, value in source_gguf_meta.items():
        suffix = key.split(".", 1)[-1]
        if suffix in key_map:
            arch[key_map[suffix]] = value
    arch["source_gguf_metadata_keys"] = sorted(source_gguf_meta.keys())
    return arch


def build_juju_file_contract(file_spec, source_gguf_meta=None):
    idx = int(file_spec["index"])
    count = int(file_spec["count"])
    source_gguf_meta = dict(source_gguf_meta or {})
    current_file = {
        "index": idx,
        "count": count,
        "name": file_spec["name"],
        "source_path": file_spec["source_path"],
        "target_path": target_juju_repo_path(file_spec),
    }
    all_files = [
        {
            "index": int(spec["index"]),
            "count": int(spec["count"]),
            "name": spec["name"],
            "source_path": spec["source_path"],
            "target_path": target_juju_repo_path(spec),
        }
        for spec in GGUF_FILE_SPECS
    ]
    return {
        "format_version": 2,
        "contract_name": "StorageLLM JUJU physical shard contract v1",
        "patched_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "notebook_build_id": NOTEBOOK_BUILD_ID,
        "artifact_format": "juju",
        "source_repo": SOURCE_HF_REPO_ID,
        "source_subdir": SOURCE_SUBDIR,
        "source_file": file_spec["source_path"],
        "source_gguf_meta": source_gguf_meta,
        "source_weight_quant_family": MODEL_PROFILE["weight_quant_schema"]["source_family"],
        "source_weight_kernel_family": MODEL_PROFILE["weight_quant_schema"]["source_kernel_family"],
        "source_weight_bits": MODEL_PROFILE["weight_quant_schema"]["source_bits"],
        "source_weight_block_size": MODEL_PROFILE["weight_quant_schema"]["source_block_size"],
        "optimization_catalog_v1": MODEL_PROFILE["format_maximization_5000"],
        "target_repo": TARGET_HF_REPO_ID,
        "target_file": target_juju_repo_path(file_spec),
        "file_index": idx,
        "file_count": count,
        "container_contract": {
            "primary_metadata_location": "juju_model_meta_section_and_paired_idx",
            "sidecar_required_for_runtime": False,
            "folder_name_runtime_dependency": False,
            "file_name_runtime_dependency": "only_for_shard_order_discovery",
            "runtime_should_read": ["JUJU header", "MODEL_META section", "QKV_POLICY section", ".juju.idx TENSOR_INDEX"],
        },
        "files_present": list(range(1, count + 1)),
        "current_file_metadata": current_file,
        "shard_set_manifest": {
            "runtime_lookup_key": "file_index",
            "file_count": count,
            "files": all_files,
        },
        "part_contract_coverage": {
            "scope": "full_juju_shard_set",
            "current_file_index": idx,
            "current_file_name": file_spec["name"],
            "required_file_count": count,
            "all_file_names_embedded": True,
            "per_file_contract_repeated": True,
            "metadata_repetition_reason": "each JUJU shard carries MODEL_META and a paired mutable .juju.idx",
            "runtime_selection_rule": "use offload.file_index/current_file_metadata for this shard and shard_set_manifest.files for sibling lookup",
        },
        "hardware_topology": {
            "embedded": False,
            "runtime_probe_required": True,
            "reason": "JUJU remains portable; engine probes VRAM, interconnect, CPU ISA, RAM, NVMe, and direct-storage support at startup",
        },
        "arch_meta": build_arch_meta(source_gguf_meta),
        "weight_quant_schema": MODEL_PROFILE["weight_quant_schema"],
        "qkv_cache_schema": MODEL_PROFILE["qkv_cache_schema"],
        "qkv_aux_data": get_qkv_aux(),
        "residency_policy": MODEL_PROFILE["residency_policy"],
        "prefetch_plan_hints": MODEL_PROFILE["prefetch_plan_hints"],
        "kernel_hints": MODEL_PROFILE["kernel_hints"],
        "execution_hints": MODEL_PROFILE["execution_hints"],
        "dynamic_swap_triggers": MODEL_PROFILE["dynamic_swap_triggers"],
        "memory_management_hints": MODEL_PROFILE["memory_management_hints"],
        "cpu_execution_hints": MODEL_PROFILE["cpu_execution_hints"],
        "kernel_config_hints": MODEL_PROFILE["kernel_config_hints"],
        "context_length_table": MODEL_PROFILE["context_length_table"],
        "io_schedule": MODEL_PROFILE["io_schedule"],
        "streaming_attention": MODEL_PROFILE["streaming_attention"],
        "numerical_stability": MODEL_PROFILE["numerical_stability"],
        "continuous_batching": MODEL_PROFILE["continuous_batching"],
        "batch_schedule": MODEL_PROFILE["batch_schedule"],
        "tensor_layout_schema": MODEL_PROFILE["tensor_layout_schema"],
        "tensor_layout_per_tensor": {
            "mode": "runtime_parse_juju_idx_tensor_index",
            "source": ".juju.idx tensors[] plus JUJU section table",
            "file_index": idx,
            "layout_overrides": [],
        },
        "integrity_schema": MODEL_PROFILE["integrity_schema"],
        "integrity_per_layer": {
            "mode": "runtime_parse_juju_section_table_idx_and_file_sha256",
            "file_level_sha256_recorded_in_verify_json": True,
            "layer_level_checksums_runtime_derivable": True,
        },
        "per_layer_swap_cost": {
            "mode": "runtime_estimate_from_juju_section_and_tensor_sizes",
            "transfer_time_model": "bytes_div_runtime_measured_bandwidth",
            "recompute_time_model": "runtime_profile_or_conservative_unknown",
            "recommendation": "runtime_compare_transfer_vs_recompute",
        },
        "activation_stats_schema": MODEL_PROFILE["activation_stats_schema"],
        "sparsity_schema": MODEL_PROFILE["sparsity_schema"],
        "runtime_monitoring_hints": MODEL_PROFILE["runtime_monitoring_hints"],
        "thermal_power_hints": MODEL_PROFILE["thermal_power_hints"],
        "speculative_execution_schema": MODEL_PROFILE["speculative_execution_schema"],
        "activation_checkpoint_schema": MODEL_PROFILE["activation_checkpoint_schema"],
        "mig_hints": MODEL_PROFILE["mig_hints"],
        "profiling_measured_data": {
            "present": False,
            "schema": "runtime_updates_measured_per_layer_data_after_first_profile_run",
            "runtime_update_allowed": True,
        },
        "multi_accelerator_detail": MODEL_PROFILE["multi_accelerator_detail"],
        "foreign_container_export": MODEL_PROFILE["foreign_container_export"],
        "backend_translation_contract": MODEL_PROFILE["backend_translation_contract"],
        "compatibility": MODEL_PROFILE["compatibility"],
        "runtime_note": {
            "required_flags": REQUIRED_RUNTIME_FLAGS,
            "reason": "model_specific_flags_from_env_or_source_notes",
        },
        "sglang_runtime_note": {
            "required_flag": REQUIRED_RUNTIME_FLAGS[0] if REQUIRED_RUNTIME_FLAGS else "",
            "required_flags": REQUIRED_RUNTIME_FLAGS,
            "reason": "model_specific_sglang_flags_from_env_or_source_notes",
        },
        "engine_contract": {
            "persistent_plain_kv_cache_allowed": False,
            "projection_kv_scratch_allowed": True,
            "qkv_packed_cache_required": True,
            "attention_must_read_packed_qkv": True,
            "offload_units": ["juju_tensor_section_span", "expert_triplet", "qkv_page"],
            "shared_experts_fusion_allowed": SHARED_EXPERTS_FUSION_ALLOWED,
            "required_runtime_flags": REQUIRED_RUNTIME_FLAGS,
        },
    }


def validate_v2_contract(contract):
    errors = []
    if contract["hardware_topology"]["embedded"] is not False:
        errors.append("hardware topology must not be embedded")
    if contract["engine_contract"]["persistent_plain_kv_cache_allowed"] is not False:
        errors.append("persistent plain KV is still allowed")
    if contract["engine_contract"]["qkv_packed_cache_required"] is not True:
        errors.append("QKV packed cache is not required")
    required_flags = contract["engine_contract"].get("required_runtime_flags", [])
    if contract["engine_contract"]["shared_experts_fusion_allowed"] is False and not required_flags:
        errors.append("shared expert fusion is disabled but required_runtime_flags is empty")
    if contract["sglang_runtime_note"].get("required_flags", []) != required_flags:
        errors.append("sglang runtime note must mirror engine required_runtime_flags")
    required_sections = [
        "io_schedule",
        "streaming_attention",
        "numerical_stability",
        "per_layer_swap_cost",
        "continuous_batching",
        "batch_schedule",
        "tensor_layout_per_tensor",
        "integrity_per_layer",
        "mig_hints",
        "profiling_measured_data",
        "multi_accelerator_detail",
        "optimization_catalog_v1",
    ]
    for section in required_sections:
        if section not in contract:
            errors.append(f"missing required v2 section: {section}")
    catalog = contract.get("optimization_catalog_v1", {})
    catalog_items = catalog.get("items", []) if isinstance(catalog, dict) else []
    expected_catalog_count = int(catalog.get("target_count", 5000)) if isinstance(catalog, dict) else 5000
    if expected_catalog_count != 5000:
        errors.append(f"optimization catalog target must be 5000, got {expected_catalog_count}")
    if len(catalog_items) != 5000:
        errors.append(f"optimization catalog must contain exactly 5000 items, got {len(catalog_items)}")
    if isinstance(catalog, dict) and int(catalog.get("actual_count", -1)) != len(catalog_items):
        errors.append("optimization catalog actual_count does not match item count")
    if len({item.get("id") for item in catalog_items if isinstance(item, dict)}) != len(catalog_items):
        errors.append("optimization catalog ids must be unique")
    required_catalog_sections = (
        "juju_physical_container_contract", "juju_header_schema", "juju_section_table_schema", "juju_expert_chunk_schema",
        "juju_index_contract", "prediction_contract", "hardware_probe_contract", "runtime_design_principles",
        "offload_strategy_contract", "adaptive_controller_contract", "gguf_to_juju_converter_contract", "qkv_cache_contract",
        "format_layout_contract", "engine_pipeline_contract", "expert_scheduler_contract", "runtime_update_files",
    )
    for catalog_section in required_catalog_sections:
        if catalog_section not in catalog:
            errors.append(f"optimization catalog missing {catalog_section}")
    if isinstance(catalog, dict):
        principles = catalog.get("runtime_design_principles", {})
        if principles.get("no_strategy_hardcoding") is not True:
            errors.append("runtime design principles must forbid hardcoded strategy")
        if principles.get("pinned_overallocation_forbidden") is not True:
            errors.append("runtime design principles must forbid pinned memory over-allocation")
        qkv_contract = catalog.get("qkv_cache_contract", {})
        if qkv_contract.get("required") is not True or qkv_contract.get("default_path") != "quantized_qkv_cache":
            errors.append("QKV cache contract must be required and default to quantized_qkv_cache")
        juju_contract = catalog.get("juju_container_contract", {})
        if juju_contract.get("format_name") != "JUJU" or juju_contract.get("qkv_required_default") is not True:
            errors.append("JUJU container contract must be primary and require QKV by default")
        tier_contract = catalog.get("universal_tier_contract", {})
        if set(tier_contract.get("tier_names", {}).values()) != {"COMPUTE_MEM", "FAST_MEM", "SLOW_MEM"}:
            errors.append("universal tier contract must define COMPUTE_MEM FAST_MEM SLOW_MEM")
        segment_contract = catalog.get("expert_segmentation_contract", {})
        if segment_contract.get("enabled") is not True or segment_contract.get("segment_cap") != 4:
            errors.append("expert segmentation contract must be enabled with segment_cap 4")
        chunk_io = catalog.get("chunk_io_contract", {})
        if chunk_io.get("compute_io_separate_streams_required") is not True or chunk_io.get("pinned_memory_reuse_required") is not True:
            errors.append("chunk IO contract must require separate streams and pinned memory reuse")
        final_structure = catalog.get("final_model_structure_contract", {})
        if final_structure.get("locked") is not True or final_structure.get("exact_mode_default") is not True:
            errors.append("final model structure must be locked with exact mode default")
        component_names = {c.get("name") for c in final_structure.get("components", []) if isinstance(c, dict)}
        required_components = {"SHARED_CORE", "ROUTING_BRAIN", "PREDICTOR_BANK", "EXPERT_GROUPS", "EXPERT_SEGMENTS", "BUDDY_FALLBACK", "QKV_SUBSYSTEM", "CODE_LOCALITY_CACHE", "TIER_SCHEDULER"}
        if not required_components <= component_names:
            errors.append("final model structure missing required JUJU components")
        code_contract = catalog.get("code_generation_moe_contract", {})
        evidence = code_contract.get("evidence", {}) if isinstance(code_contract, dict) else {}
        if evidence.get("same_token_jaccard") != 0.649 or evidence.get("different_token_jaccard") != 0.175:
            errors.append("code MoE locality evidence must preserve Jaccard values 0.649 and 0.175")
        finalization = catalog.get("structure_finalization_contract", {})
        if finalization.get("model_structure_change_required_after_this") is not False:
            errors.append("structure finalization must state no model structure change is required after this")
    residency = contract.get("residency_policy", {})
    if not residency.get("valid_split_points") or not residency.get("reentry_points"):
        errors.append("residency policy must include valid_split_points and reentry_points")
    cpu_hints = contract.get("cpu_execution_hints", {})
    if "numa_hints" not in cpu_hints:
        errors.append("cpu_execution_hints must include detailed numa_hints")
    forbidden_terms = ["cuda_graph", "cudaStream", "hipStream", "MTLCommandQueue", "VkQueue", "warp_count"]
    encoded = json.dumps(contract, ensure_ascii=False)
    for term in forbidden_terms:
        if term in encoded:
            errors.append(f"backend-specific term leaked into portable footer: {term}")
    optional = set(contract.get("compatibility", {}).get("optional_features", []))
    if "cuda_graph" in optional or "multi_gpu" in optional or "gds" in optional:
        errors.append("optional_features must use portable names, not CUDA/GPU-specific names")
    return errors


def release_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def cleanup_paths(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            if path.is_dir():
                shutil.rmtree(path)
            else:
                path.unlink()
            print("Deleted local:", path)
    release_memory()


In [ ]:
# ========================= CHANGE HERE / RUN HERE =========================
# Paste one HF repo/folder/file URL here, or leave it blank and the cell will
# show an input box before starting. Setup cells do not scan or download.
USER_SOURCE_MODEL_URL = ""

# Optional. If both are blank, the notebook first tries HF_TOKEN -> whoami(),
# then shows one input box for owner or owner/name.
USER_TARGET_HF_REPO_ID = os.environ.get("TARGET_HF_REPO_ID", "")
USER_TARGET_HF_OWNER = os.environ.get("TARGET_HF_OWNER", os.environ.get("HF_USERNAME", ""))
USER_TARGET_SUBDIR = None

USER_START_FILE_INDEX = int(os.environ.get("START_FILE_INDEX", os.environ.get("START_PART", "1")))
USER_END_FILE_INDEX = os.environ.get("END_FILE_INDEX", os.environ.get("END_PART", ""))
USER_END_FILE_INDEX = int(USER_END_FILE_INDEX) if str(USER_END_FILE_INDEX).strip() else None

USER_UPLOAD_TO_HF = True
USER_UPLOAD_FOREIGN_METADATA = True
USER_DELETE_LOCAL_AFTER_UPLOAD = True
USER_RUNTIME_ASSETS_ONLY = os.environ.get("RUNTIME_ASSETS_ONLY", "0") == "1"


def _juju_prompt_text(label, default=""):
    default = str(default or "").strip()
    suffix = f" [{default}]" if default else ""
    value = input(f"{label}{suffix}: ").strip()
    return value or default


def _juju_hf_owner_from_token(token):
    if not token:
        return ""
    try:
        info = HfApi(token=token).whoami()
        return str(info.get("name") or info.get("fullname") or "").strip()
    except Exception:
        return ""


if not str(USER_SOURCE_MODEL_URL or "").strip():
    USER_SOURCE_MODEL_URL = _juju_prompt_text("SOURCE HF URL / repo / folder / GGUF file")

HfApi, create_repo = ensure_hf_api()
HF_TOKEN = get_hf_token() if USER_UPLOAD_TO_HF else (env_or_colab_secret("HF_TOKEN") or "")

if USER_UPLOAD_TO_HF and not str(USER_TARGET_HF_REPO_ID or "").strip() and not str(USER_TARGET_HF_OWNER or "").strip():
    USER_TARGET_HF_OWNER = _juju_hf_owner_from_token(HF_TOKEN)

if USER_UPLOAD_TO_HF and not str(USER_TARGET_HF_REPO_ID or "").strip() and not str(USER_TARGET_HF_OWNER or "").strip():
    target_value = _juju_prompt_text("TARGET HF repo id owner/name, or owner only")
    if "/" in target_value:
        USER_TARGET_HF_REPO_ID = target_value
    else:
        USER_TARGET_HF_OWNER = target_value

configure_juju_notebook_job(
    source_url=USER_SOURCE_MODEL_URL,
    target_repo_id=USER_TARGET_HF_REPO_ID,
    target_owner=USER_TARGET_HF_OWNER,
    target_subdir=USER_TARGET_SUBDIR,
    start_file_index=USER_START_FILE_INDEX,
    end_file_index=USER_END_FILE_INDEX,
    upload_to_hf=USER_UPLOAD_TO_HF,
    upload_foreign_metadata=USER_UPLOAD_FOREIGN_METADATA,
    delete_local_after_upload=USER_DELETE_LOCAL_AFTER_UPLOAD,
)
# ========================================================================

if UPLOAD_TO_HF and not HF_TOKEN:
    raise RuntimeError("Missing Hugging Face write token")

api = HfApi(token=HF_TOKEN)
if UPLOAD_TO_HF:
    create_repo(repo_id=TARGET_HF_REPO_ID, repo_type="model", private=False, exist_ok=True, token=HF_TOKEN)

model_card_path = MODEL_ROOT / "README.md"
model_card_path.write_text(f'''# {SOURCE_FILE_PREFIX} StorageLLM Offload

Source GGUF repo: https://huggingface.co/{SOURCE_HF_REPO_ID}
Source GGUF path: {SOURCE_FILE_NAME or SOURCE_SUBDIR or "repository root"}

This repo contains StorageLLM JUJU shard artifacts generated from source GGUF.
GGUF is the conversion source only. The uploaded runtime files preserve the
original shard stem and replace the extension with `.juju` plus a paired
`.juju.idx` runtime index. Tensor payloads are rewritten into 4KB-aligned
JUJU sections so O_DIRECT/GDS eligibility and hot/warm/cold placement can be
real runtime behavior instead of metadata-only hints.

Runtime flags embedded in the JUJU contract:
`{", ".join(REQUIRED_RUNTIME_FLAGS) if REQUIRED_RUNTIME_FLAGS else "none"}`.

Download note: the notebook sends `SOURCE_HF_TOKEN` when set, otherwise it uses
the same `HF_TOKEN` used for upload. This keeps the flow working if the source
repo becomes gated and the token has accepted access.
''', encoding="utf-8")
if UPLOAD_TO_HF:
    api.upload_file(
        path_or_fileobj=str(model_card_path),
        path_in_repo="README.md",
        repo_id=TARGET_HF_REPO_ID,
        repo_type="model",
        commit_message="Add StorageLLM offload model card",
    )

# Upload tokenizer/config/runtime assets once so the HF model repo is usable, not just a weight bucket.
UPLOAD_RUNTIME_ASSETS = os.environ.get("UPLOAD_RUNTIME_ASSETS", os.environ.get("UPLOAD_TOKENIZER_ASSETS", "1")) != "0"
UPLOAD_TOKENIZER_ASSETS = UPLOAD_RUNTIME_ASSETS
_DEFAULT_RUNTIME_ASSET_FILES = [
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
    "added_tokens.json",
    "tokenizer.model",
    "sentencepiece.bpe.model",
    "tiktoken.model",
    "vocab.json",
    "merges.txt",
    "tokenization_kimi.py",
    "tool_declaration_ts.py",
    "configuration_deepseek.py",
    "configuration_kimi_k25.py",
    "modeling_deepseek.py",
    "modeling_kimi_k25.py",
    "kimi_k25_processor.py",
    "kimi_k25_vision_processing.py",
    "media_utils.py",
    "processor_config.json",
    "preprocessor_config.json",
    "image_processor_config.json",
    "feature_extractor.json",
    "video_preprocessor_config.json",
    "audio_config.json",
]
_HARD_REQUIRED_RUNTIME_FILES = {
    "config.json",
    "generation_config.json",
    "tokenizer_config.json",
    "chat_template.jinja",
}
_HARD_REQUIRED_TOKENIZER_ANY_OF = {"tokenizer.json", "tokenizer.model", "sentencepiece.bpe.model", "tiktoken.model", "vocab.json"}
_RUNTIME_TOKENIZER_LIKE_FILES = {
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "chat_template.jinja",
    "added_tokens.json",
    "tokenizer.model",
    "sentencepiece.bpe.model",
    "tiktoken.model",
    "vocab.json",
    "merges.txt",
    "tokenization_kimi.py",
    "tool_declaration_ts.py",
}

import hashlib
import io

try:
    from huggingface_hub import hf_hub_download
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[hf_transfer]"])
    from huggingface_hub import hf_hub_download


def _runtime_env_list(name, default):
    raw = os.environ.get(name)
    values = list(default) if raw is None else [item.strip().strip("/") for item in raw.split(",")]
    deduped = []
    for value in values:
        if value not in deduped:
            deduped.append(value)
    return deduped


def _runtime_env_set(name, default, hard_required=None, empty_uses_default=True):
    raw = os.environ.get(name)
    if raw is None:
        values = set(default)
    else:
        values = {item.strip() for item in raw.split(",") if item.strip()}
        if empty_uses_default and not values:
            values = set(default)
    if hard_required:
        values.update(hard_required)
    return values


RUNTIME_ASSET_FILES = _runtime_env_list("RUNTIME_ASSET_FILES", _DEFAULT_RUNTIME_ASSET_FILES)
RUNTIME_REQUIRED_FILES = _runtime_env_set("RUNTIME_REQUIRED_FILES", _HARD_REQUIRED_RUNTIME_FILES, _HARD_REQUIRED_RUNTIME_FILES)
RUNTIME_REQUIRED_ANY_OF = _runtime_env_set("RUNTIME_REQUIRED_ANY_OF", _HARD_REQUIRED_TOKENIZER_ANY_OF, None)
RUNTIME_MAX_FILE_BYTES = int(os.environ.get("RUNTIME_MAX_FILE_BYTES", os.environ.get("TOKENIZER_MAX_FILE_BYTES", str(512 * 1024 * 1024))))
TOKENIZER_ASSET_FILES = list(RUNTIME_ASSET_FILES)
TOKENIZER_REQUIRED_FILES = set(RUNTIME_REQUIRED_FILES)
TOKENIZER_REQUIRED_ANY_OF = set(RUNTIME_REQUIRED_ANY_OF)
TOKENIZER_MAX_FILE_BYTES = RUNTIME_MAX_FILE_BYTES


def _default_runtime_asset_source_repos():
    repos = []
    for env_name in ("SOURCE_RUNTIME_ASSET_REPO_ID", "SOURCE_TOKENIZER_REPO_ID", "SOURCE_CONFIG_REPO_ID"):
        value = os.environ.get(env_name, "").strip()
        if value:
            repos.append(value)
    if SOURCE_HF_REPO_ID:
        repos.append(SOURCE_HF_REPO_ID)
    raw_fallback = os.environ.get("RUNTIME_ASSET_FALLBACK_REPO_IDS", os.environ.get("TOKENIZER_FALLBACK_REPO_IDS", "")).strip()
    if raw_fallback:
        repos.extend([item.strip() for item in raw_fallback.split(",") if item.strip()])
    else:
        haystack = f"{SOURCE_HF_REPO_ID} {TARGET_HF_REPO_ID} {SOURCE_FILE_PREFIX}".lower()
        if "gemma" in haystack:
            repos.extend([
                "google/gemma-4-26B-A4B-it",
                "OsaurusAI/gemma-4-26B-A4B-it-mxfp4",
                "google/gemma-4-26B-A4B",
            ])
        elif "kimi" in haystack:
            repos.extend([
                "moonshotai/Kimi-K2.6",
                "moonshotai/Kimi-K2-Instruct",
                "unsloth/Kimi-K2-Thinking",
            ])
        elif "glm-5.1" in haystack or "glm5.1" in haystack or "glm_5_1" in haystack:
            repos.append("zai-org/GLM-5.1")
    deduped = []
    for repo in repos:
        if repo and repo not in deduped:
            deduped.append(repo)
    return deduped


def _runtime_asset_source_paths(filename):
    if os.environ.get("SOURCE_TOKENIZER_SUBDIR"):
        source_subdirs = _runtime_env_list("SOURCE_TOKENIZER_SUBDIR", [""])
    else:
        default_subdirs = ["", "tokenizer"]
        if SOURCE_SUBDIR:
            default_subdirs.append(SOURCE_SUBDIR)
        source_subdirs = _runtime_env_list("RUNTIME_ASSET_SOURCE_SUBDIRS", default_subdirs)
    out = []
    for subdir in source_subdirs:
        out.append(repo_path_join(subdir, filename) if subdir else filename)
    return out


def _runtime_asset_target_paths(filename):
    default_subdirs = [""]
    if TARGET_SUBDIR:
        default_subdirs.append(TARGET_SUBDIR)
    target_subdirs = _runtime_env_list("RUNTIME_ASSET_TARGET_SUBDIRS", default_subdirs)
    paths = []
    for subdir in target_subdirs:
        paths.append(repo_path_join(subdir, filename) if subdir else filename)
    if filename in _RUNTIME_TOKENIZER_LIKE_FILES:
        for subdir in _runtime_env_list("TOKENIZER_MIRROR_SUBDIRS", ["tokenizer"]):
            paths.append(repo_path_join(subdir, filename) if subdir else filename)
    deduped = []
    for path in paths:
        if path not in deduped:
            deduped.append(path)
    return deduped


def _sha256_path(path):
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(chunk)
    return sha.hexdigest()


def _download_runtime_asset(repo_id, path, token):
    try:
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=path,
            revision=os.environ.get("SOURCE_RUNTIME_ASSET_REVISION", os.environ.get("SOURCE_TOKENIZER_REVISION", "main")) or "main",
            token=token or None,
            cache_dir=str(MODEL_ROOT / "hf_runtime_asset_cache"),
            repo_type="model",
        )
        size = Path(local_path).stat().st_size
        if size > RUNTIME_MAX_FILE_BYTES:
            raise RuntimeError(f"runtime asset too large: {path} bytes={size}")
        return {
            "repo": repo_id,
            "path": path,
            "bytes": size,
            "sha256": _sha256_path(local_path),
            "local_path": local_path,
        }, None
    except Exception as exc:
        return None, f"{repo_id}:{path}:{type(exc).__name__}:{str(exc)[:300]}"


def upload_runtime_assets_for_current_job(api, hf_token, source_token):
    summary = {"enabled": bool(UPLOAD_RUNTIME_ASSETS and UPLOAD_TO_HF), "uploaded": [], "skipped": [], "errors": {}}
    if not summary["enabled"]:
        return summary
    source_repos = _default_runtime_asset_source_repos()
    summary.update({
        "source_repos": source_repos,
        "required_files": sorted(RUNTIME_REQUIRED_FILES),
        "required_any_of": sorted(RUNTIME_REQUIRED_ANY_OF),
        "target_policy": "root_plus_target_subdir_and_tokenizer_mirror",
        "download_backend": "huggingface_hub.hf_hub_download",
    })
    uploaded_required_any = set()
    uploaded_required_files = set()
    for filename in RUNTIME_ASSET_FILES:
        found = None
        attempts = []
        for repo_id in source_repos:
            for source_path in _runtime_asset_source_paths(filename):
                print("Checking runtime asset:", repo_id, source_path)
                found, err = _download_runtime_asset(repo_id, source_path, source_token)
                if found:
                    break
                if err:
                    attempts.append(err)
            if found:
                break
        if not found:
            summary["errors"][filename] = attempts[:12]
            if filename in RUNTIME_REQUIRED_FILES:
                raise FileNotFoundError(f"required runtime asset not found: {filename}; attempts={attempts[:8]}")
            summary["skipped"].append(filename)
            print("Skipping missing optional runtime asset:", filename)
            continue
        if filename in RUNTIME_REQUIRED_ANY_OF:
            uploaded_required_any.add(filename)
        if filename in RUNTIME_REQUIRED_FILES:
            uploaded_required_files.add(filename)
        for target_path in _runtime_asset_target_paths(filename):
            print("Uploading runtime asset:", found["repo"], found["path"], "->", TARGET_HF_REPO_ID, target_path, found["bytes"], "bytes")
            api.upload_file(
                path_or_fileobj=found["local_path"],
                path_in_repo=target_path,
                repo_id=TARGET_HF_REPO_ID,
                repo_type="model",
                commit_message=f"Add runtime asset {target_path}",
            )
            summary["uploaded"].append({
                "source_repo": found["repo"],
                "source_path": found["path"],
                "target_path": target_path,
                "bytes": found["bytes"],
                "sha256": found["sha256"],
            })
    missing_required = sorted(RUNTIME_REQUIRED_FILES - uploaded_required_files)
    if missing_required:
        raise FileNotFoundError(f"required runtime assets missing: {missing_required}")
    if RUNTIME_REQUIRED_ANY_OF and not uploaded_required_any:
        raise FileNotFoundError(f"no tokenizer asset found; need one of {sorted(RUNTIME_REQUIRED_ANY_OF)}")
    summary["required_files_found"] = sorted(uploaded_required_files)
    summary["required_any_found"] = sorted(uploaded_required_any)
    manifest = json.dumps(summary, ensure_ascii=False, indent=2).encode("utf-8")
    for manifest_path in [repo_path_join(TARGET_SUBDIR, "runtime_assets_manifest.json") if TARGET_SUBDIR else "runtime_assets_manifest.json"]:
        api.upload_file(
            path_or_fileobj=io.BytesIO(manifest),
            path_in_repo=manifest_path,
            repo_id=TARGET_HF_REPO_ID,
            repo_type="model",
            commit_message="Add runtime asset manifest",
        )
    return summary


def upload_tokenizer_assets_for_current_job(api, hf_token, source_token):
    return upload_runtime_assets_for_current_job(api, hf_token, source_token)


source_download_token = os.environ.get("SOURCE_HF_TOKEN") or HF_TOKEN
runtime_asset_upload_summary = upload_runtime_assets_for_current_job(api, HF_TOKEN, source_download_token)
tokenizer_upload_summary = runtime_asset_upload_summary
if USER_RUNTIME_ASSETS_ONLY:
    run_summary = {
        "ok": True,
        "asset_only": True,
        "notebook_build_id": NOTEBOOK_BUILD_ID,
        "source_repo_id": SOURCE_HF_REPO_ID,
        "target_repo_id": TARGET_HF_REPO_ID,
        "target_subdir": TARGET_SUBDIR,
        "started_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "finished_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
        "runtime_asset_upload": runtime_asset_upload_summary,
        "tokenizer_upload": tokenizer_upload_summary,
    }
    summary_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", SOURCE_FILE_PREFIX).strip("_") or "juju"
    summary_path = VERIFY_DIR / f"{summary_stem}_runtime_assets_only_summary.json"
    summary_path.write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(run_summary, ensure_ascii=False, indent=2))
    print("RUNTIME_ASSETS_ONLY_SUMMARY:", summary_path)
    raise SystemExit("RUNTIME_ASSETS_ONLY=1: runtime assets uploaded; skipped JUJU weight materialization")

run_summary = {
    "ok": True,
    "errors": [],
    "notebook_build_id": NOTEBOOK_BUILD_ID,
    "source_repo_id": SOURCE_HF_REPO_ID,
    "source_subdir": SOURCE_SUBDIR,
    "source_file_prefix": SOURCE_FILE_PREFIX,
    "source_file_count": SOURCE_FILE_COUNT,
    "target_repo_id": TARGET_HF_REPO_ID,
    "target_subdir": TARGET_SUBDIR,
    "files": [],
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "continue_on_part_error": CONTINUE_ON_PART_ERROR,
    "max_upload_file_bytes": juju_upload_file_limit_bytes(),
    "hf_individual_file_limit_bytes": HF_INDIVIDUAL_FILE_LIMIT_BYTES,
    "runtime_asset_upload": runtime_asset_upload_summary,
    "tokenizer_upload": tokenizer_upload_summary,
}

source_download_token = os.environ.get("SOURCE_HF_TOKEN") or HF_TOKEN

for spec in FILES_TO_RUN:
    idx = int(spec["index"])
    name = spec["name"]
    juju_names = juju_artifact_names(name)
    local_out = OUT_DIR / juju_names["weights"]
    local_idx = OUT_DIR / juju_names["index"]
    local_verify = VERIFY_DIR / juju_names["verify"]
    export_paths = {}
    uploaded = False
    local_cleanup_paths = [local_out, local_idx, local_verify]
    part_infos = []
    part_results = []
    print("\\n========== GGUF", f"{idx:05d}/{SOURCE_FILE_COUNT:05d}", name, "==========")
    try:
        base_contract = build_juju_file_contract(spec)
        source_gguf_meta = read_hf_gguf_source_meta(spec, base_contract, token=source_download_token)
        contract = build_juju_file_contract(spec, source_gguf_meta=source_gguf_meta)
        errors = validate_v2_contract(contract)
        if errors:
            raise RuntimeError("contract validation failed: " + "; ".join(errors))

        base_stem = Path(name).stem
        stale_paths = [local_out, local_idx, local_verify]
        stale_paths += list(OUT_DIR.glob(f"{base_stem}.split*-of-*.juju"))
        stale_paths += list(OUT_DIR.glob(f"{base_stem}.split*-of-*.juju.idx"))
        stale_paths += list(VERIFY_DIR.glob(f"{base_stem}.split*-of-*.juju.verify.json"))
        for artifact_path in stale_paths:
            if artifact_path.exists():
                artifact_path.unlink()

        write_parts = write_juju_shard_parts_from_hf_url(
            source_url=source_gguf_hub_url(spec),
            source_name=spec["name"],
            source_path=spec["source_path"],
            output_path=local_out,
            index_path=local_idx,
            contract=contract,
            token=source_download_token,
            source_repo_id=SOURCE_HF_REPO_ID,
            chunk_size=HTTP_CHUNK_BYTES,
            max_file_bytes=juju_upload_file_limit_bytes(),
        )
        if EXPORT_FOREIGN_METADATA:
            export_paths = write_foreign_metadata_exports_for_stem(contract, portable_stem_for_gguf(spec))

        for part_no, write_info in enumerate(write_parts, start=1):
            part_out_path = Path(write_info["path"])
            part_idx_path = Path(write_info["index_path"])
            part_verify_path = VERIFY_DIR / write_info["verify_file"]
            target_path = repo_path_join(TARGET_SUBDIR, write_info["weight_file"])
            target_index_path = repo_path_join(TARGET_SUBDIR, write_info["index_file"])
            local_cleanup_paths.extend([part_out_path, part_idx_path, part_verify_path])
            verify = {
                "ok": True,
                "errors": [],
                "source_repo_id": SOURCE_HF_REPO_ID,
                "target_repo_id": TARGET_HF_REPO_ID,
                "file_index": idx,
                "file_count": int(spec["count"]),
                "source_path": spec["source_path"],
                "target_path": target_path,
                "target_index_path": target_index_path,
                "source_gguf_meta_keys": sorted(source_gguf_meta.keys()),
                "input": {"path": source_gguf_hub_url(spec), "bytes": write_info["source_bytes"], "sha256": write_info.get("source_sha256", "")},
                "output": write_info,
                "split": write_info.get("split", {}),
                "patch_mode": "remote_gguf_to_juju_shard_materialization",
                "contract": {
                    "qkv_packed_cache_required": True,
                    "persistent_plain_kv_cache_allowed": False,
                    "shared_experts_fusion_allowed": SHARED_EXPERTS_FUSION_ALLOWED,
                    "required_runtime_flags": REQUIRED_RUNTIME_FLAGS,
                    "hardware_topology_embedded": False,
                    "full_shard_set_contract_embedded": True,
                },
                "patched_at": contract["patched_at"],
            }
            part_verify_path.write_text(json.dumps(verify, ensure_ascii=False, indent=2), encoding="utf-8")
            if UPLOAD_TO_HF:
                print("Uploading JUJU shard:", part_out_path, "->", target_path)
                api.upload_file(
                    path_or_fileobj=str(part_out_path),
                    path_in_repo=target_path,
                    repo_id=TARGET_HF_REPO_ID,
                    repo_type="model",
                    commit_message=f"Add JUJU shard {idx:05d}/{SOURCE_FILE_COUNT:05d} part {part_no:02d}/{len(write_parts):02d}",
                )
                print("Uploading JUJU index:", part_idx_path, "->", target_index_path)
                api.upload_file(
                    path_or_fileobj=str(part_idx_path),
                    path_in_repo=target_index_path,
                    repo_id=TARGET_HF_REPO_ID,
                    repo_type="model",
                    commit_message=f"Add JUJU index {idx:05d}/{SOURCE_FILE_COUNT:05d} part {part_no:02d}/{len(write_parts):02d}",
                )
                print("Uploading verify JSON:", part_verify_path, "->", f"verify/{part_verify_path.name}")
                api.upload_file(
                    path_or_fileobj=str(part_verify_path),
                    path_in_repo=f"verify/{part_verify_path.name}",
                    repo_id=TARGET_HF_REPO_ID,
                    repo_type="model",
                    commit_message=f"Add JUJU verify {idx:05d}/{SOURCE_FILE_COUNT:05d} part {part_no:02d}/{len(write_parts):02d}",
                )
            part_infos.append(write_info)
            part_results.append({
                "part": part_no,
                "target_path": target_path,
                "target_index_path": target_index_path,
                "verify_path": f"verify/{part_verify_path.name}",
                "bytes": write_info["bytes"],
                "index_bytes": write_info["index_bytes"],
                "tensor_count": write_info["tensor_count"],
                "split": write_info.get("split", {}),
            })

        if UPLOAD_TO_HF and UPLOAD_FOREIGN_METADATA and export_paths:
            for export_kind, export_path in export_paths.items():
                if export_kind.startswith("gguf_"):
                    metadata_path = f"metadata/gguf/{export_path.name}"
                elif export_kind == "safetensors_metadata":
                    metadata_path = f"metadata/safetensors/{export_path.name}"
                else:
                    metadata_path = f"metadata/sidecar/{export_path.name}"
                print("Uploading foreign metadata:", export_path, "->", metadata_path)
                api.upload_file(
                    path_or_fileobj=str(export_path),
                    path_in_repo=metadata_path,
                    repo_id=TARGET_HF_REPO_ID,
                    repo_type="model",
                    commit_message=f"Add offload metadata export {export_kind} {idx:05d}/{SOURCE_FILE_COUNT:05d}",
                )
        uploaded = bool(UPLOAD_TO_HF)

        aggregate_write_info = {
            "source_bytes": part_infos[0]["source_bytes"] if part_infos else 0,
            "bytes": sum(info["bytes"] for info in part_infos),
            "index_bytes": sum(info["index_bytes"] for info in part_infos),
            "section_count": sum(info["section_count"] for info in part_infos),
            "tensor_count": sum(info["tensor_count"] for info in part_infos),
            "storage_mode": "split_remote_range_to_4kb_aligned_juju_sections" if len(part_infos) > 1 else (part_infos[0]["storage_mode"] if part_infos else ""),
            "split_count": len(part_infos),
        }
        run_summary["files"].append({
            "index": idx,
            "name": name,
            "uploaded": uploaded,
            "target_path": part_results[0]["target_path"] if len(part_results) == 1 else [part["target_path"] for part in part_results],
            "target_index_path": part_results[0]["target_index_path"] if len(part_results) == 1 else [part["target_index_path"] for part in part_results],
            "input_bytes": aggregate_write_info["source_bytes"],
            "output_bytes": aggregate_write_info["bytes"],
            "index_bytes": aggregate_write_info["index_bytes"],
            "storage_mode": aggregate_write_info["storage_mode"],
            "section_count": aggregate_write_info["section_count"],
            "tensor_count": aggregate_write_info["tensor_count"],
            "split_count": aggregate_write_info["split_count"],
            "parts": part_results,
            "foreign_metadata_exports": {k: str(v) for k, v in export_paths.items()},
        })

        if DELETE_LOCAL_AFTER_UPLOAD and (uploaded or not UPLOAD_TO_HF):
            cleanup_paths([*local_cleanup_paths, *export_paths.values()])
    except Exception as exc:
        run_summary["ok"] = False
        err = {"index": idx, "name": name, "type": type(exc).__name__, "message": str(exc)[:1000]}
        run_summary["errors"].append(err)
        print("ERROR:", json.dumps(err, ensure_ascii=False))
        if not KEEP_FAILED_LOCAL_FILES:
            cleanup_paths([p for p in [*local_cleanup_paths, *export_paths.values()] if p])
        if not CONTINUE_ON_PART_ERROR:
            raise
    finally:
        release_memory()

run_summary["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
summary_stem = re.sub(r"[^A-Za-z0-9._-]+", "_", SOURCE_FILE_PREFIX).strip("_") or "juju"
summary_path = VERIFY_DIR / f"{summary_stem}_juju_materialize_run_summary.json"
summary_path.write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(run_summary, ensure_ascii=False, indent=2))
print("RUN_SUMMARY:", summary_path)
